
# HL7 Healthcare Interoperability and Laboratory Data Integration

## Step 1: Project Overview

This project explores how healthcare information systems exchange clinical laboratory data using Health Level Seven (HL7) messaging standards. The project simulates the transmission of laboratory results from a Laboratory Information System (LIS) to an Electronic Health Record (EHR).

Using Python, HL7 version 2.5.1 messages will be generated, parsed, validated, and transformed into structured datasets. Later phases will incorporate PostgreSQL for relational data storage and Power BI for visualizing message processing, data quality, and laboratory workflow performance.

All patient and laboratory information used in this project will be synthetic. No actual patient records or protected health information (PHI) will be used.

## Project Objectives

- Understand HL7 v2.5.1 message structure and common laboratory segments.
- Generate synthetic patient, laboratory order, and test result data.
- Parse and validate HL7 ORU^R01 messages using Python.
- Transform HL7 messages into structured relational data.
- Analyze data quality and interoperability performance using SQL and Power BI.
  


## Step 2: Python Environment Verification

The Python environment was checked to identify the installed Python version, executable location, and operating system. This verification ensures that the Jupyter Notebook is using the correct environment before installing additional libraries.

The environment was successfully verified using Python 3.12.8 through Anaconda on Windows.


In [1]:

import sys
import platform

print("Python Version:", platform.python_version())
print("Python Executable:", sys.executable)
print("Operating System:", platform.system())
print("Jupyter Environment: Ready")


Python Version: 3.12.8
Python Executable: C:\Users\angel\anaconda3\python.exe
Operating System: Windows
Jupyter Environment: Ready



## Step 3: HL7 Library Installation and Verification

The `hl7apy` library was installed to support the creation, parsing, and validation of HL7 version 2.x messages. This library will be used to simulate laboratory data exchange between a Laboratory Information System (LIS) and an Electronic Health Record (EHR).

The installation was completed using `%pip install hl7apy` within Jupyter Notebook. The installed library was then imported and its version verified.

The successful installation of `hl7apy` version 1.3.5 confirmed that the environment was ready for HL7 message development.


In [2]:

%pip install hl7apy


   ---------------------------------------- 0.0/974.7 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/974.7 kB ? eta -:--:--
   ---------------------------------------- 974.7/974.7 kB 3.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:

import hl7apy
from importlib.metadata import version

print("HL7apy Version:", version("hl7apy"))
print("HL7 Library Installation: Successful")


HL7apy Version: 1.3.5
HL7 Library Installation: Successful



## Step 4: Synthetic Patient and Laboratory Data Generation

This step generates a synthetic clinical laboratory dataset to simulate patient records, laboratory orders, and test results originating from a Laboratory Information System (LIS).

The dataset includes 1,000 fictitious patients and multiple laboratory orders representing Complete Blood Count (CBC), Basic Metabolic Panel (BMP), and Comprehensive Metabolic Panel (CMP) testing. Python is used to generate patient identifiers, laboratory order numbers, collection timestamps, test results, and standardized Logical Observation Identifiers Names and Codes (LOINC) identifiers.

Each laboratory panel contains multiple individual test results, reflecting the relationship between laboratory orders and their associated observations. Synthetic numerical values are generated using predefined ranges for demonstration purposes rather than clinical interpretation.

A fixed random seed ensures that the dataset can be reproduced. The generated records will serve as the foundation for creating HL7 v2.5.1 ORU^R01 messages, validating message structure, and evaluating healthcare data interoperability.

All patient and laboratory information is synthetic and contains no actual patient records or protected health information (PHI).


In [4]:

import pandas as pd
import random
from datetime import datetime, timedelta

# Set random seed for reproducibility
random.seed(42)

# Define laboratory tests:
# (LOINC code, unit, synthetic minimum, synthetic maximum)

tests = {
    "Hemoglobin": ("718-7", "g/dL", 10.0, 17.0),
    "Hematocrit": ("4544-3", "%", 30.0, 52.0),
    "WBC": ("6690-2", "10*3/uL", 3.0, 15.0),
    "RBC": ("789-8", "10*6/uL", 3.5, 6.0),
    "Platelets": ("777-3", "10*3/uL", 100, 500),
    "MCV": ("787-2", "fL", 70.0, 105.0),
    "MCH": ("785-6", "pg", 23.0, 35.0),
    "MCHC": ("786-4", "g/dL", 30.0, 37.0),
    "RDW": ("788-0", "%", 10.0, 18.0),

    "Sodium": ("2951-2", "mmol/L", 130, 150),
    "Potassium": ("2823-3", "mmol/L", 3.0, 6.0),
    "Chloride": ("2075-0", "mmol/L", 95, 115),
    "CO2": ("2028-9", "mmol/L", 18, 35),
    "Glucose": ("2345-7", "mg/dL", 60, 200),
    "BUN": ("3094-0", "mg/dL", 5, 40),
    "Creatinine": ("2160-0", "mg/dL", 0.5, 2.5),
    "Calcium": ("17861-6", "mg/dL", 7.5, 11.0),

    "Albumin": ("1751-7", "g/dL", 2.5, 5.5),
    "Total Protein": ("2885-2", "g/dL", 5.0, 9.0),
    "Total Bilirubin": ("1975-2", "mg/dL", 0.1, 2.5),
    "ALP": ("6768-6", "U/L", 30, 180),
    "AST": ("1920-8", "U/L", 10, 100),
    "ALT": ("1742-6", "U/L", 7, 100)
}

# Define laboratory panels
panels = {
    "CBC": [
        "Hemoglobin", "Hematocrit", "WBC", "RBC",
        "Platelets", "MCV", "MCH", "MCHC", "RDW"
    ],

    "BMP": [
        "Sodium", "Potassium", "Chloride", "CO2",
        "Glucose", "BUN", "Creatinine", "Calcium"
    ],

    "CMP": [
        "Sodium", "Potassium", "Chloride", "CO2",
        "Glucose", "BUN", "Creatinine", "Calcium",
        "Albumin", "Total Protein", "Total Bilirubin",
        "ALP", "AST", "ALT"
    ]
}

# Generate synthetic laboratory records
records = []
order_counter = 1

for patient_num in range(1, 1001):

    patient_id = f"P{patient_num:05d}"

    # Each patient receives two distinct panel orders
    selected_panels = random.sample(
        list(panels.keys()), 2
    )

    for panel_name in selected_panels:

        order_id = f"ORD{order_counter:06d}"

        collection_time = (
            datetime(2026, 10, 16, 8, 0)
            + timedelta(minutes=order_counter * 2)
        )

        result_time = collection_time + timedelta(
            minutes=random.randint(30, 180)
        )

        for test_name in panels[panel_name]:

            loinc, unit, minimum, maximum = tests[test_name]

            result = round(
                random.uniform(minimum, maximum), 1
            )

            records.append({
                "Patient_ID": patient_id,
                "Order_ID": order_id,
                "Panel": panel_name,
                "Collection_Time": collection_time,
                "Result_Time": result_time,
                "Test_Name": test_name,
                "LOINC_Code": loinc,
                "Result": result,
                "Unit": unit,
                "Result_Status": "Final"
            })

        order_counter += 1

# Create pandas DataFrame
df_labs = pd.DataFrame(records)

# Display dataset summary
print("Synthetic Laboratory Dataset Created")
print("------------------------------------")
print("Total Patients:", df_labs["Patient_ID"].nunique())
print("Total Orders:", df_labs["Order_ID"].nunique())
print("Total Test Types:", df_labs["Test_Name"].nunique())
print("Total Results:", len(df_labs))

print("\nOrders by Panel:")
print(
    df_labs.groupby("Panel")["Order_ID"].nunique()
)

print("\nFirst 10 Laboratory Records:")
display(df_labs.head(10))


C:\Users\angel\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\angel\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Synthetic Laboratory Dataset Created
------------------------------------
Total Patients: 1000
Total Orders: 2000
Total Test Types: 23
Total Results: 20612

Orders by Panel:
Panel
BMP    673
CBC    670
CMP    657
Name: Order_ID, dtype: int64

First 10 Laboratory Records:


,Patient_ID,Order_ID,Panel,Collection_Time,Result_Time,Test_Name,LOINC_Code,Result,Unit,Result_Status
0,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,Sodium,2951-2,144.8,mmol/L,Final
1,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,Potassium,2823-3,3.7,mmol/L,Final
2,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,Chloride,2075-0,97.8,mmol/L,Final
3,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,CO2,2028-9,19.7,mmol/L,Final
4,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,Glucose,2345-7,163.7,mg/dL,Final
5,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,BUN,3094-0,24.1,mg/dL,Final
6,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,Creatinine,2160-0,1.7,mg/dL,Final
7,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,Calcium,17861-6,7.6,mg/dL,Final
8,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,Albumin,1751-7,2.8,g/dL,Final
9,P00001,ORD000001,CMP,2026-10-16 08:02:00,2026-10-16 08:38:00,Total Protein,2885-2,5.9,g/dL,Final


In [5]:

# Verify unique patients and laboratory orders

print("Patient and Order Verification")
print("------------------------------")

print("Unique Patient IDs:",
      df_labs["Patient_ID"].nunique())

print("Unique Order IDs:",
      df_labs["Order_ID"].nunique())

print("\nFirst 10 Unique Patients:")
print(df_labs["Patient_ID"].drop_duplicates().head(10).to_list())

print("\nOrders Per Patient:")
print(
    df_labs.groupby("Patient_ID")["Order_ID"]
    .nunique().value_counts().sort_index()
)

print("\nDuplicate Patient-Order-Test Records:")
print(
    df_labs.duplicated(
        subset=["Patient_ID", "Order_ID", "Test_Name"]
    ).sum()
)


Patient and Order Verification
------------------------------
Unique Patient IDs: 1000
Unique Order IDs: 2000

First 10 Unique Patients:
['P00001', 'P00002', 'P00003', 'P00004', 'P00005', 'P00006', 'P00007', 'P00008', 'P00009', 'P00010']

Orders Per Patient:
Order_ID
2    1000
Name: count, dtype: int64

Duplicate Patient-Order-Test Records:
0



### Step 4 Results and Verification

The synthetic laboratory dataset was successfully generated with 1,000 unique patients, 2,000 laboratory orders, and 20,612 individual laboratory results across 23 distinct test types. The dataset includes CBC, BMP, and CMP panels, with each patient assigned two laboratory orders.

Verification confirmed that all 1,000 patients had two distinct laboratory orders and that no duplicate patient-order-test combinations were identified. The use of unique patient and order identifiers establishes the relationships needed for subsequent HL7 message generation and relational database development.

These results confirm that the initial dataset was generated as intended. Additional checks for missing values, data consistency, and laboratory result integrity will be performed in the next step.



## Step 5: Data Quality and Validation

Data quality validation is performed to evaluate the completeness, consistency, and integrity of the synthetic laboratory dataset before HL7 message generation. Accurate patient identifiers, laboratory orders, test codes, results, and timestamps are essential for reliable data exchange between a Laboratory Information System (LIS) and an Electronic Health Record (EHR).

The validation process examines missing values, duplicate records, laboratory test and LOINC code consistency, chronological accuracy, and expected panel composition. These checks help identify potential data quality issues that could interfere with HL7 message processing or result interpretation.

### Step 5A: Missing Value Assessment

The dataset is examined for missing values across all columns, including patient identifiers, laboratory orders, collection and result timestamps, test names, LOINC codes, numerical results, units, and result statuses.

Identifying missing information before message generation helps establish whether the source dataset is sufficiently complete for subsequent processing.


In [6]:

# Step 5A: Missing Value Assessment

print("Step 5A: Missing Value Assessment")
print("--------------------------------")

# Count missing values in each column
missing_values = df_labs.isnull().sum()

# Calculate percentage of missing values
missing_percent = (
    df_labs.isnull().mean() * 100
).round(2)

# Create summary table
missing_summary = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percent": missing_percent
})

# Display results
display(missing_summary)

# Overall missing value count
total_missing = df_labs.isnull().sum().sum()

print("\nTotal Missing Values:", total_missing)

if total_missing == 0:
    print("Validation Status: PASSED")
    print("No missing values detected.")
else:
    print("Validation Status: REVIEW REQUIRED")
    print("Missing values detected in the dataset.")


Step 5A: Missing Value Assessment
--------------------------------


,Missing_Count,Missing_Percent
Patient_ID,0,0.0
Order_ID,0,0.0
Panel,0,0.0
Collection_Time,0,0.0
Result_Time,0,0.0
Test_Name,0,0.0
LOINC_Code,0,0.0
Result,0,0.0
Unit,0,0.0
Result_Status,0,0.0



Total Missing Values: 0
Validation Status: PASSED
No missing values detected.



### Step 5B: Duplicate Record and Identifier Validation

This step evaluates the integrity of patient identifiers, laboratory order numbers, and individual test records. The validation checks for duplicate laboratory results, verifies that each order belongs to a single patient, and confirms that each patient has the expected number of distinct orders.

These checks help identify duplicate records and inconsistent relationships that could affect laboratory data exchange between an LIS and an EHR.


In [7]:

# Step 5B: Duplicate Record and Identifier Validation

print("Step 5B: Duplicate and Identifier Validation")
print("-------------------------------------------")

# Check duplicate patient-order-test combinations
duplicate_results = df_labs.duplicated(
    subset=["Patient_ID", "Order_ID", "Test_Name"]
).sum()

# Check whether orders belong to multiple patients
order_patient_counts = df_labs.groupby(
    "Order_ID"
)["Patient_ID"].nunique()

invalid_order_links = (order_patient_counts != 1).sum()

# Check number of orders assigned to each patient
patient_order_counts = df_labs.groupby(
    "Patient_ID"
)["Order_ID"].nunique()

invalid_patient_orders = (patient_order_counts != 2).sum()

# Check for blank identifiers
blank_patient_ids = (
    df_labs["Patient_ID"].astype(str).str.strip() == ""
).sum()

blank_order_ids = (
    df_labs["Order_ID"].astype(str).str.strip() == ""
).sum()

# Display validation results
validation_results = pd.DataFrame({
    "Validation_Check": [
        "Duplicate Patient-Order-Test Records",
        "Orders Linked to Multiple Patients",
        "Patients Without Exactly Two Orders",
        "Blank Patient Identifiers",
        "Blank Order Identifiers"
    ],
    "Issues_Found": [
        duplicate_results,
        invalid_order_links,
        invalid_patient_orders,
        blank_patient_ids,
        blank_order_ids
    ]
})

display(validation_results)

# Overall validation status
total_issues = validation_results["Issues_Found"].sum()

print("\nTotal Issues Identified:", total_issues)

if total_issues == 0:
    print("Validation Status: PASSED")
else:
    print("Validation Status: REVIEW REQUIRED")


Step 5B: Duplicate and Identifier Validation
-------------------------------------------


,Validation_Check,Issues_Found
0,Duplicate Patient-Order-Test Records,0
1,Orders Linked to Multiple Patients,0
2,Patients Without Exactly Two Orders,0
3,Blank Patient Identifiers,0
4,Blank Order Identifiers,0



Total Issues Identified: 0
Validation Status: PASSED



### Step 5C: Laboratory Test, LOINC Code, and Unit Validation

This step evaluates the consistency of laboratory test names, Logical Observation Identifiers Names and Codes (LOINC) identifiers, and measurement units across the synthetic dataset.

Each laboratory test is compared against the predefined test dictionary used during data generation. This process identifies missing or mismatched test codes, inconsistent measurement units, and unexpected laboratory test identifiers.

Validating these relationships helps ensure that laboratory observations are consistently represented before conversion into HL7 v2.5.1 messages. The validation is based on the project's predefined mappings and does not replace verification against an authoritative LOINC terminology database.


In [8]:

# Step 5C: Laboratory Test, LOINC Code, and Unit Validation

print("Step 5C: LOINC and Unit Validation")
print("---------------------------------")

# Build reference mapping from the original test dictionary
reference_tests = pd.DataFrame([
    {
        "Test_Name": name,
        "Expected_LOINC": values[0],
        "Expected_Unit": values[1]
    }
    for name, values in tests.items()
])

# Compare laboratory records against reference mappings
df_validation = df_labs.merge(
    reference_tests,
    on="Test_Name",
    how="left",
    validate="many_to_one"
)

# Identify unexpected test names
unknown_tests = df_validation[
    df_validation["Expected_LOINC"].isna()
]

# Identify incorrect LOINC codes
loinc_mismatches = df_validation[
    df_validation["Expected_LOINC"].notna()
    & (
        df_validation["LOINC_Code"]
        != df_validation["Expected_LOINC"]
    )
]

# Identify incorrect measurement units
unit_mismatches = df_validation[
    df_validation["Expected_Unit"].notna()
    & (
        df_validation["Unit"]
        != df_validation["Expected_Unit"]
    )
]

# Create validation summary
loinc_validation = pd.DataFrame({
    "Validation_Check": [
        "Unknown Laboratory Test Names",
        "LOINC Code Mismatches",
        "Measurement Unit Mismatches"
    ],
    "Issues_Found": [
        len(unknown_tests),
        len(loinc_mismatches),
        len(unit_mismatches)
    ]
})

display(loinc_validation)

# Display unique laboratory test mappings
print("\nUnique Laboratory Test Mappings:")

display(
    df_labs[
        ["Test_Name", "LOINC_Code", "Unit"]
    ]
    .drop_duplicates()
    .sort_values("Test_Name")
    .reset_index(drop=True)
)

# Overall validation status
total_issues = loinc_validation["Issues_Found"].sum()

print("\nTotal Issues Identified:", total_issues)

if total_issues == 0:
    print("Validation Status: PASSED")
else:
    print("Validation Status: REVIEW REQUIRED")


Step 5C: LOINC and Unit Validation
---------------------------------


,Validation_Check,Issues_Found
0,Unknown Laboratory Test Names,0
1,LOINC Code Mismatches,0
2,Measurement Unit Mismatches,0



Unique Laboratory Test Mappings:


,Test_Name,LOINC_Code,Unit
0,ALP,6768-6,U/L
1,ALT,1742-6,U/L
2,AST,1920-8,U/L
3,Albumin,1751-7,g/dL
4,BUN,3094-0,mg/dL
5,CO2,2028-9,mmol/L
6,Calcium,17861-6,mg/dL
7,Chloride,2075-0,mmol/L
8,Creatinine,2160-0,mg/dL
9,Glucose,2345-7,mg/dL



Total Issues Identified: 0
Validation Status: PASSED



### Step 5D: Timestamp and Chronological Validation

This step evaluates the chronological consistency of laboratory collection and result timestamps. Each laboratory result is checked to confirm that its result timestamp occurs after the specimen collection timestamp.

The validation also examines whether individual results associated with the same laboratory order have consistent collection times and identifies unexpected negative or zero turnaround times.

Turnaround time is calculated in minutes using the difference between the result and collection timestamps. These checks establish the chronological integrity of the synthetic dataset before HL7 message generation and provide a foundation for subsequent laboratory workflow analysis.


In [9]:

# Step 5D: Timestamp and Chronological Validation

print("Step 5D: Timestamp Validation")
print("----------------------------")

# Calculate laboratory turnaround time in minutes
df_labs["Turnaround_Minutes"] = (
    df_labs["Result_Time"] - df_labs["Collection_Time"]
).dt.total_seconds() / 60

# Identify results occurring before or at collection time
invalid_chronology = (
    df_labs["Result_Time"] <= df_labs["Collection_Time"]
).sum()

# Identify negative or zero turnaround times
invalid_turnaround = (
    df_labs["Turnaround_Minutes"] <= 0
).sum()

# Check for inconsistent collection times within orders
collection_consistency = (
    df_labs.groupby("Order_ID")["Collection_Time"]
    .nunique()
)

inconsistent_collections = (
    collection_consistency > 1
).sum()

# Check for inconsistent result timestamps within orders
result_consistency = (
    df_labs.groupby("Order_ID")["Result_Time"]
    .nunique()
)

inconsistent_result_times = (
    result_consistency > 1
).sum()

# Create validation summary
timestamp_validation = pd.DataFrame({
    "Validation_Check": [
        "Results Before or At Collection Time",
        "Negative or Zero Turnaround Times",
        "Orders With Inconsistent Collection Times",
        "Orders With Inconsistent Result Times"
    ],
    "Issues_Found": [
        invalid_chronology,
        invalid_turnaround,
        inconsistent_collections,
        inconsistent_result_times
    ]
})

display(timestamp_validation)

# Display turnaround time statistics
print("\nTurnaround Time Summary (Minutes):")
display(df_labs["Turnaround_Minutes"].describe().round(2))

# Overall validation status
total_issues = timestamp_validation["Issues_Found"].sum()

print("\nTotal Issues Identified:", total_issues)

if total_issues == 0:
    print("Validation Status: PASSED")
else:
    print("Validation Status: REVIEW REQUIRED")


Step 5D: Timestamp Validation
----------------------------


,Validation_Check,Issues_Found
0,Results Before or At Collection Time,0
1,Negative or Zero Turnaround Times,0
2,Orders With Inconsistent Collection Times,0
3,Orders With Inconsistent Result Times,0



Turnaround Time Summary (Minutes):


count    20612.00
mean       106.25
std         44.71
min         30.00
25%         67.00
50%        107.00
75%        146.00
max        180.00
Name: Turnaround_Minutes, dtype: float64


Total Issues Identified: 0
Validation Status: PASSED



### Step 5E: Laboratory Panel Composition and Result Count Validation

This step evaluates whether each laboratory order contains the expected number and combination of tests for its assigned panel. CBC, BMP, and CMP orders are compared against the predefined panel definitions used during synthetic data generation.

The validation identifies incomplete panels, unexpected tests, duplicate observations, and orders associated with inconsistent panel assignments. These checks help ensure that laboratory orders and their individual results are correctly structured before conversion into HL7 v2.5.1 messages.

The panel definitions are specific to this simulation and are not intended to represent every laboratory's clinical testing configuration.


In [10]:

# Step 5E: Laboratory Panel Composition Validation

print("Step 5E: Laboratory Panel Validation")
print("-----------------------------------")

# Check each order against its expected panel definition
panel_issues = []

for order_id, group in df_labs.groupby("Order_ID"):

    assigned_panels = group["Panel"].unique()

    # Each order must have exactly one recognized panel
    if len(assigned_panels) != 1:
        panel_issues.append((order_id, "Inconsistent Panel"))
        continue

    panel_name = assigned_panels[0]

    if panel_name not in panels:
        panel_issues.append((order_id, "Unknown Panel"))
        continue

    expected_tests = set(panels[panel_name])
    actual_tests = set(group["Test_Name"])

    # Check for missing or unexpected tests
    if actual_tests != expected_tests:
        panel_issues.append((order_id, "Incorrect Test Composition"))

    # Check expected number of results
    if len(group) != len(panels[panel_name]):
        panel_issues.append((order_id, "Incorrect Result Count"))

# Create summary of panel composition issues
panel_issue_df = pd.DataFrame(
    panel_issues,
    columns=["Order_ID", "Issue"]
)

print("Total Orders Evaluated:",
      df_labs["Order_ID"].nunique())

print("Total Panel Issues:",
      len(panel_issue_df))

if panel_issue_df.empty:
    print("Validation Status: PASSED")
else:
    print("Validation Status: REVIEW REQUIRED")
    display(panel_issue_df.head(20))

# Display panel order and result counts
panel_summary = df_labs.groupby("Panel").agg(
    Total_Orders=("Order_ID", "nunique"),
    Total_Results=("Test_Name", "count"),
    Unique_Tests=("Test_Name", "nunique")
).reset_index()

print("\nLaboratory Panel Summary:")
display(panel_summary)

# Confirm total results
print("\nTotal Laboratory Results:",
      panel_summary["Total_Results"].sum())


Step 5E: Laboratory Panel Validation
-----------------------------------
Total Orders Evaluated: 2000
Total Panel Issues: 0
Validation Status: PASSED

Laboratory Panel Summary:


,Panel,Total_Orders,Total_Results,Unique_Tests
0,BMP,673,5384,8
1,CBC,670,6030,9
2,CMP,657,9198,14



Total Laboratory Results: 20612



### Step 5F: Exporting the Validated Laboratory Dataset

Following successful data quality validation, the synthetic laboratory dataset is exported to a CSV file for persistent storage and future use.

The exported dataset contains patient identifiers, laboratory order numbers, panel assignments, collection and result timestamps, laboratory test names, LOINC codes, numerical results, measurement units, result statuses, and calculated turnaround times.

Saving the validated dataset provides a consistent source for subsequent HL7 message generation, PostgreSQL database development, and Power BI analysis. The CSV file also allows the original dataset to be reloaded without repeating the data generation process.

All exported information is synthetic and contains no protected health information (PHI).


In [11]:

# Step 5F: Export Validated Laboratory Dataset

from pathlib import Path

# Define project directory
project_dir = Path(
    r"C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project"
)

# Create data directory if it does not exist
data_dir = project_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# Define CSV output file
csv_path = data_dir / "synthetic_laboratory_data.csv"

# Export validated dataset
df_labs.to_csv(csv_path, index=False)

# Verify saved file by reloading it
df_saved = pd.read_csv(csv_path)

print("Step 5F: Dataset Export Verification")
print("-----------------------------------")
print("File Saved:", csv_path)
print("Total Records:", len(df_saved))
print("Total Columns:", len(df_saved.columns))
print("Unique Patients:", df_saved["Patient_ID"].nunique())
print("Unique Orders:", df_saved["Order_ID"].nunique())

# Confirm exported data matches original dataset
expected_rows = len(df_labs)
expected_columns = len(df_labs.columns)

if (
    len(df_saved) == expected_rows
    and len(df_saved.columns) == expected_columns
):
    print("\nExport Status: PASSED")
else:
    print("\nExport Status: REVIEW REQUIRED")


Step 5F: Dataset Export Verification
-----------------------------------
File Saved: C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project\data\synthetic_laboratory_data.csv
Total Records: 20612
Total Columns: 11
Unique Patients: 1000
Unique Orders: 2000

Export Status: PASSED



### Step 5 Results: Data Quality and Validation Summary

Data quality validation was completed on the synthetic laboratory dataset containing 1,000 patients, 2,000 laboratory orders, and 20,612 individual test results across CBC, BMP, and CMP panels.

The dataset passed all five validation sections. No missing values, duplicate patient-order-test records, inconsistent patient or order relationships, LOINC mapping mismatches, or chronological errors were identified. All 2,000 laboratory orders contained the expected tests and result counts based on the predefined panel configurations.

Laboratory turnaround times ranged from 30 to 180 minutes, with a mean of 106.25 minutes and a median of 107 minutes. These values reflect simulated processing times rather than actual laboratory performance.

The validation results confirm that the dataset meets the structural and internal consistency requirements established for this project. However, the checks do not independently establish clinical validity or compliance with all HL7 requirements.

The validated dataset will serve as the foundation for generating HL7 v2.5.1 ORU^R01 messages and evaluating the exchange of laboratory information between a simulated LIS and EHR.



## Step 6: HL7 v2.5.1 Laboratory Message Generation

This step begins the transformation of validated laboratory data into HL7 version 2.5.1 ORU^R01 messages. These messages are commonly used to transmit laboratory observations and results between healthcare information systems.

The process will simulate laboratory result transmission from a Laboratory Information System (LIS) to an Electronic Health Record (EHR). Each message will contain standardized segments identifying the sending and receiving systems, patient, laboratory order, and individual test results.

### Step 6A: Laboratory Order Selection and Preparation

A single laboratory order is selected from the validated dataset to demonstrate HL7 message construction. The selected order contains a Comprehensive Metabolic Panel (CMP) with 14 individual test results.

The order is examined to verify its patient identifier, order number, panel assignment, collection and result timestamps, LOINC codes, numerical results, and measurement units.

This initial inspection establishes the source information needed to construct and validate the first HL7 message before expanding the process to all 2,000 laboratory orders.


In [12]:

# Step 6A: Select and Prepare a Laboratory Order

import pandas as pd
from pathlib import Path

# Define saved dataset location
csv_path = Path(
    r"C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project\data\synthetic_laboratory_data.csv"
)

# Reload validated laboratory dataset
df_hl7 = pd.read_csv(
    csv_path,
    parse_dates=["Collection_Time", "Result_Time"]
)

# Select the first laboratory order
selected_order_id = "ORD000001"

df_order = df_hl7[
    df_hl7["Order_ID"] == selected_order_id
].copy()

# Sort observations by original dataset order
df_order = df_order.sort_index()

# Display selected order information
print("Step 6A: Laboratory Order Preparation")
print("------------------------------------")

print("Order ID:", selected_order_id)
print("Patient ID:", df_order["Patient_ID"].iloc[0])
print("Panel:", df_order["Panel"].iloc[0])
print("Number of Results:", len(df_order))

print(
    "Collection Time:",
    df_order["Collection_Time"].iloc[0]
)

print(
    "Result Time:",
    df_order["Result_Time"].iloc[0]
)

print("\nLaboratory Test Results:")

display(
    df_order[
        [
            "Test_Name",
            "LOINC_Code",
            "Result",
            "Unit",
            "Result_Status"
        ]
    ].reset_index(drop=True)
)


Step 6A: Laboratory Order Preparation
------------------------------------
Order ID: ORD000001
Patient ID: P00001
Panel: CMP
Number of Results: 14
Collection Time: 2026-10-16 08:02:00
Result Time: 2026-10-16 08:38:00

Laboratory Test Results:


,Test_Name,LOINC_Code,Result,Unit,Result_Status
0,Sodium,2951-2,144.8,mmol/L,Final
1,Potassium,2823-3,3.7,mmol/L,Final
2,Chloride,2075-0,97.8,mmol/L,Final
3,CO2,2028-9,19.7,mmol/L,Final
4,Glucose,2345-7,163.7,mg/dL,Final
5,BUN,3094-0,24.1,mg/dL,Final
6,Creatinine,2160-0,1.7,mg/dL,Final
7,Calcium,17861-6,7.6,mg/dL,Final
8,Albumin,1751-7,2.8,g/dL,Final
9,Total Protein,2885-2,5.9,g/dL,Final



### Step 6B: HL7 Message Header and Patient Identification

This step begins construction of an HL7 v2.5.1 ORU^R01 laboratory results message using the `hl7apy` Python library.

The Message Header (MSH) segment identifies the sending Laboratory Information System (LIS), receiving Electronic Health Record (EHR), message timestamp, message type, unique message control identifier, processing mode, and HL7 version.

The Patient Identification (PID) segment associates the laboratory message with the synthetic patient identifier from the validated dataset.

The MSH and PID segments are created first to establish the message context before adding laboratory order information and individual test results in subsequent steps.


In [13]:

# Step 6B: Create HL7 Message Header and Patient Identification

from hl7apy.core import Message

# Create HL7 v2.5.1 ORU_R01 message structure
hl7_message = Message("ORU_R01", version="2.5.1")

# Define message header
msh = hl7_message.msh

msh.msh_3 = "SYNTHETIC_LIS"
msh.msh_4 = "SIMULATED_HOSPITAL"
msh.msh_5 = "SIMULATED_EHR"
msh.msh_6 = "SIMULATED_HOSPITAL"

msh.msh_7 = "20261016083800"
msh.msh_9 = "ORU^R01^ORU_R01"
msh.msh_10 = "MSG000001"
msh.msh_11 = "P"
msh.msh_12 = "2.5.1"

# Create patient result group
patient_group = hl7_message.add_group("ORU_R01_PATIENT_RESULT")

# Create patient group and PID segment
patient = patient_group.add_group("ORU_R01_PATIENT")
pid = patient.add_segment("PID")

# Assign synthetic patient identifier
pid.pid_3 = "P00001^^^SIMULATED_HOSPITAL^MR"

# Display generated segments
print("Step 6B: HL7 Header and Patient Identification")
print("---------------------------------------------")

print("\nMSH Segment:")
print(msh.to_er7())

print("\nPID Segment:")
print(pid.to_er7())


Step 6B: HL7 Header and Patient Identification
---------------------------------------------

MSH Segment:
MSH|^~\&|SYNTHETIC_LIS|SIMULATED_HOSPITAL|SIMULATED_EHR|SIMULATED_HOSPITAL|20261016083800||ORU^R01^ORU_R01|MSG000001|P|2.5.1

PID Segment:
PID|||P00001^^^SIMULATED_HOSPITAL^MR



### Step 6C: Laboratory Order and Observation Results

This step expands the HL7 v2.5.1 ORU^R01 message by adding the Observation Request (OBR) segment and individual Observation Result (OBX) segments.

The OBR segment identifies the laboratory order, panel, and associated timestamps. Each OBX segment represents an individual laboratory result, including the test identifier, LOINC code, numerical value, measurement unit, and result status.

The selected Comprehensive Metabolic Panel (CMP) contains 14 laboratory observations. These observations are organized under a single laboratory order to demonstrate how multiple test results can be transmitted within one HL7 message.

The resulting message structure will be examined and validated in the next step.


In [14]:

# Step 6C: Add Laboratory Order and Observation Results

# Create order observation group
order_group = patient_group.add_group(
    "ORU_R01_ORDER_OBSERVATION"
)

# Add OBR segment
obr = order_group.add_segment("OBR")

obr.obr_1 = "1"
obr.obr_2 = selected_order_id
obr.obr_4 = "24323-8^Comprehensive metabolic panel^LN"

# Add laboratory timestamps
collection_time = df_order["Collection_Time"].iloc[0]
result_time = df_order["Result_Time"].iloc[0]

obr.obr_7 = collection_time.strftime("%Y%m%d%H%M%S")
obr.obr_22 = result_time.strftime("%Y%m%d%H%M%S")
obr.obr_25 = "F"

# Create individual observation groups
for index, row in enumerate(df_order.itertuples(), start=1):

    observation_group = order_group.add_group(
        "ORU_R01_OBSERVATION"
    )

    obx = observation_group.add_segment("OBX")

    obx.obx_1 = str(index)
    obx.obx_2 = "NM"

    obx.obx_3 = (
        f"{row.LOINC_Code}^{row.Test_Name}^LN"
    )

    obx.obx_5 = str(row.Result)
    obx.obx_6 = row.Unit
    obx.obx_11 = "F"

# Display generated OBR and OBX segments
print("Step 6C: Laboratory Order and Results")
print("------------------------------------")

print("\nOBR Segment:")
print(obr.to_er7())

print("\nOBX Segments:")

for observation in order_group.children:
    if observation.name == "ORU_R01_OBSERVATION":
        print(observation.obx.to_er7())


Step 6C: Laboratory Order and Results
------------------------------------

OBR Segment:
OBR|1|ORD000001||24323-8^Comprehensive metabolic panel^LN|||20261016080200|||||||||||||||20261016083800|||F

OBX Segments:
OBX|1|NM|2951-2^Sodium^LN||144.8|mmol/L|||||F
OBX|2|NM|2823-3^Potassium^LN||3.7|mmol/L|||||F
OBX|3|NM|2075-0^Chloride^LN||97.8|mmol/L|||||F
OBX|4|NM|2028-9^CO2^LN||19.7|mmol/L|||||F
OBX|5|NM|2345-7^Glucose^LN||163.7|mg/dL|||||F
OBX|6|NM|3094-0^BUN^LN||24.1|mg/dL|||||F
OBX|7|NM|2160-0^Creatinine^LN||1.7|mg/dL|||||F
OBX|8|NM|17861-6^Calcium^LN||7.6|mg/dL|||||F
OBX|9|NM|1751-7^Albumin^LN||2.8|g/dL|||||F
OBX|10|NM|2885-2^Total Protein^LN||5.9|g/dL|||||F
OBX|11|NM|1975-2^Total Bilirubin^LN||1.5|mg/dL|||||F
OBX|12|NM|6768-6^ALP^LN||114.2|U/L|||||F
OBX|13|NM|1920-8^AST^LN||74.4|U/L|||||F
OBX|14|NM|1742-6^ALT^LN||72.2|U/L|||||F



### Step 6D: HL7 Message Structure and Validation

This step evaluates the completed HL7 v2.5.1 ORU^R01 laboratory results message by converting its segments into the standard HL7 ER7 pipe-delimited format.

The message is examined to confirm that the Message Header (MSH), Patient Identification (PID), Observation Request (OBR), and Observation Result (OBX) segments are present. The number of OBX segments is compared against the original laboratory order to verify that all 14 CMP observations were included.

The `hl7apy` library is then used to perform structural validation against its HL7 v2.5.1 definitions. Any validation errors will be reviewed and corrected before generating messages for the complete laboratory dataset.

Successful structural validation will establish that the message meets the library's applicable requirements, although additional interface-specific testing would be necessary in a production healthcare environment.


In [15]:

# Step 6D: HL7 Message Structure and Validation

from hl7apy.exceptions import ValidationError

print("Step 6D: HL7 Message Validation")
print("------------------------------")

# Convert the complete message to ER7 format
hl7_text = hl7_message.to_er7()

# Split the message into individual segments
segments = hl7_text.split("\r")

# Count each segment type
segment_counts = {}

for segment in segments:
    if segment:
        segment_name = segment[:3]
        segment_counts[segment_name] = (
            segment_counts.get(segment_name, 0) + 1
        )

print("\nHL7 Segment Counts:")

for name, count in segment_counts.items():
    print(f"{name}: {count}")

# Verify expected segment counts
expected_counts = {
    "MSH": 1,
    "PID": 1,
    "OBR": 1,
    "OBX": len(df_order)
}

segment_check = (
    all(
        segment_counts.get(name, 0) == count
        for name, count in expected_counts.items()
    )
    and len(segments) == sum(expected_counts.values())
)

print("\nSegment Count Verification:",
      "PASSED" if segment_check else "REVIEW REQUIRED")

# Validate HL7 message structure
print("\nHL7 Structural Validation:")

try:
    hl7_message.validate()
    print("Validation Status: PASSED")

except ValidationError as error:
    print("Validation Status: REVIEW REQUIRED")
    print("Validation Details:", error)

# Display complete HL7 message
print("\nComplete HL7 Message:")
print(hl7_text.replace("\r", "\n"))


Step 6D: HL7 Message Validation
------------------------------

HL7 Segment Counts:
MSH: 1
PID: 1
OBR: 1
OBX: 14

Segment Count Verification: PASSED

HL7 Structural Validation:
Validation Status: REVIEW REQUIRED
Validation Details: Missing required child PID.PID_5

Complete HL7 Message:
MSH|^~\&|SYNTHETIC_LIS|SIMULATED_HOSPITAL|SIMULATED_EHR|SIMULATED_HOSPITAL|20261016083800||ORU^R01^ORU_R01|MSG000001|P|2.5.1
PID|||P00001^^^SIMULATED_HOSPITAL^MR
OBR|1|ORD000001||24323-8^Comprehensive metabolic panel^LN|||20261016080200|||||||||||||||20261016083800|||F
OBX|1|NM|2951-2^Sodium^LN||144.8|mmol/L|||||F
OBX|2|NM|2823-3^Potassium^LN||3.7|mmol/L|||||F
OBX|3|NM|2075-0^Chloride^LN||97.8|mmol/L|||||F
OBX|4|NM|2028-9^CO2^LN||19.7|mmol/L|||||F
OBX|5|NM|2345-7^Glucose^LN||163.7|mg/dL|||||F
OBX|6|NM|3094-0^BUN^LN||24.1|mg/dL|||||F
OBX|7|NM|2160-0^Creatinine^LN||1.7|mg/dL|||||F
OBX|8|NM|17861-6^Calcium^LN||7.6|mg/dL|||||F
OBX|9|NM|1751-7^Albumin^LN||2.8|g/dL|||||F
OBX|10|NM|2885-2^Total Protein^LN||5.9

In [16]:

# Step 6D: Correct Missing Required PID Field

from hl7apy.exceptions import ValidationError

# Assign a synthetic patient name
pid.pid_5 = "TESTPATIENT^ONE"

print("Step 6D: PID Field Correction")
print("----------------------------")

print("Updated PID Segment:")
print(pid.to_er7())

# Revalidate the complete HL7 message
print("\nHL7 Structural Validation:")

try:
    hl7_message.validate()
    print("Validation Status: PASSED")

except ValidationError as error:
    print("Validation Status: REVIEW REQUIRED")
    print("Validation Details:", error)


Step 6D: PID Field Correction
----------------------------
Updated PID Segment:
PID|||P00001^^^SIMULATED_HOSPITAL^MR||TESTPATIENT^ONE

HL7 Structural Validation:
Validation Status: PASSED



### Step 6D Results: HL7 Structural Validation and Error Resolution

The initial HL7 v2.5.1 ORU^R01 message was successfully generated with one MSH segment, one PID segment, one OBR segment, and 14 OBX segments representing the Comprehensive Metabolic Panel results.

The initial structural validation identified a missing required patient name field (PID-5). Although the synthetic patient identifier was present in PID-3, the HL7 message structure required the patient name field to be populated.

A fictional patient name was added to PID-5, and the message was revalidated using the `hl7apy` library. The corrected message passed structural validation without additional errors.

This process demonstrates how HL7 validation can identify missing required fields and support troubleshooting before messages are transmitted between healthcare information systems. The successful validation confirms compliance with the library's applicable structural requirements, although additional testing would be necessary for production interoperability.



### Step 6E: Synthetic Patient Demographics Generation and Validation

This step creates a separate synthetic patient demographics dataset to support HL7 patient identification and future PostgreSQL database development.

Each of the 1,000 existing patient identifiers is assigned a unique fictional patient name, date of birth, and administrative sex value. Patient identifiers are preserved from the original laboratory dataset to maintain consistent relationships between patients, laboratory orders, and test results.

The demographics dataset is validated for unique patient identifiers, missing values, and consistency with the laboratory dataset. Once verified, it is exported to CSV for future use in HL7 message generation and relational database development.

All demographic information is artificially generated and does not represent actual patients or protected health information (PHI).


In [17]:

# Step 6E: Synthetic Patient Demographics Generation

import pandas as pd
import random
from datetime import datetime, timedelta
from pathlib import Path

# Set seed for reproducibility
random.seed(42)

# Use existing patient identifiers
patient_ids = sorted(df_hl7["Patient_ID"].unique())

# Create synthetic demographic records
patient_records = []

for index, patient_id in enumerate(patient_ids, start=1):

    # Generate a fictional date of birth
    dob = datetime(1940, 1, 1) + timedelta(
        days=random.randint(0, 23000)
    )

    # Assign synthetic administrative sex
    sex = random.choice(["F", "M"])

    patient_records.append({
        "Patient_ID": patient_id,
        "Last_Name": f"TESTPATIENT{index:04d}",
        "First_Name": "SYNTHETIC",
        "Date_of_Birth": dob.strftime("%Y-%m-%d"),
        "Sex": sex
    })

# Create patient DataFrame
df_patients = pd.DataFrame(patient_records)

# Verify demographics dataset
print("Step 6E: Patient Demographics Verification")
print("-----------------------------------------")

print("Total Patient Records:", len(df_patients))
print("Unique Patient IDs:",
      df_patients["Patient_ID"].nunique())

print("Duplicate Patient IDs:",
      df_patients["Patient_ID"].duplicated().sum())

print("Missing Values:",
      df_patients.isnull().sum().sum())

# Verify patient IDs match laboratory dataset
lab_patient_ids = set(df_hl7["Patient_ID"])
demographic_ids = set(df_patients["Patient_ID"])

print("Patient ID Relationships Match:",
      lab_patient_ids == demographic_ids)

# Display first 10 synthetic patients
print("\nFirst 10 Synthetic Patients:")
display(df_patients.head(10))

# Save demographics dataset
data_dir = Path(
    r"C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project\data"
)

data_dir.mkdir(parents=True, exist_ok=True)

patient_csv_path = data_dir / "synthetic_patients.csv"

df_patients.to_csv(patient_csv_path, index=False)

print("\nPatient Dataset Saved:")
print(patient_csv_path)


Step 6E: Patient Demographics Verification
-----------------------------------------
Total Patient Records: 1000
Unique Patient IDs: 1000
Duplicate Patient IDs: 0
Missing Values: 0
Patient ID Relationships Match: True

First 10 Synthetic Patients:


,Patient_ID,Last_Name,First_Name,Date_of_Birth,Sex
0,P00001,TESTPATIENT0001,SYNTHETIC,1997-05-13,F
1,P00002,TESTPATIENT0002,SYNTHETIC,1942-03-30,M
2,P00003,TESTPATIENT0003,SYNTHETIC,1961-12-20,F
3,P00004,TESTPATIENT0004,SYNTHETIC,1952-07-08,F
4,P00005,TESTPATIENT0005,SYNTHETIC,2000-09-16,F
5,P00006,TESTPATIENT0006,SYNTHETIC,1992-12-22,M
6,P00007,TESTPATIENT0007,SYNTHETIC,1942-11-07,F
7,P00008,TESTPATIENT0008,SYNTHETIC,1948-05-28,F
8,P00009,TESTPATIENT0009,SYNTHETIC,1960-11-14,F
9,P00010,TESTPATIENT0010,SYNTHETIC,1990-05-08,F



Patient Dataset Saved:
C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project\data\synthetic_patients.csv



### Step 6E Results: Synthetic Patient Demographics Verification

The synthetic patient demographics dataset was successfully generated with 1,000 unique patient records. Each patient was assigned a fictional name, date of birth, and administrative sex value while preserving the patient identifiers established in the original laboratory dataset.

Validation confirmed that all patient identifiers were unique, no missing values or duplicate patient records were detected, and all patient identifiers matched those in the laboratory dataset.

The demographics dataset was exported to `synthetic_patients.csv` for future HL7 message generation and PostgreSQL integration. Maintaining separate patient and laboratory datasets supports relational database development and helps preserve consistent patient identification across the simulated healthcare information systems.



### Step 6F: HL7 Message Generation with Patient Demographics

This step integrates the validated laboratory dataset with the synthetic patient demographics dataset to generate complete HL7 v2.5.1 ORU^R01 laboratory results messages.

Patient records are linked to laboratory orders using the unique Patient_ID field. The HL7 Patient Identification (PID) segment is populated with the patient's synthetic identifier, name, date of birth, and administrative sex. The Observation Request (OBR) segment contains laboratory order and panel information, while individual Observation Result (OBX) segments contain laboratory test results, LOINC identifiers, measurement units, and result statuses.

The first laboratory order is used to test the message-generation process before expanding it to all 2,000 orders. Structural validation is performed using the hl7apy library to identify potential message formatting or required-field errors.

This process establishes a reusable approach for transforming structured clinical laboratory data into HL7 messages for subsequent parsing and PostgreSQL integration.


In [18]:

# Step 6F: Generate HL7 Message with Patient Demographics

from hl7apy.core import Message
from hl7apy.exceptions import ValidationError
import pandas as pd

print("Step 6F: HL7 Message with Patient Demographics")
print("---------------------------------------------")

# Select first laboratory order
selected_order_id = "ORD000001"

order_data = df_hl7[
    df_hl7["Order_ID"] == selected_order_id
].copy()

# Retrieve patient associated with the order
patient_id = order_data["Patient_ID"].iloc[0]

patient_data = df_patients[
    df_patients["Patient_ID"] == patient_id
].iloc[0]

# Create new HL7 v2.5.1 message
hl7_full = Message("ORU_R01", version="2.5.1")

# MSH: Message Header
msh = hl7_full.msh

msh.msh_3 = "SYNTHETIC_LIS"
msh.msh_4 = "SIMULATED_HOSPITAL"
msh.msh_5 = "SIMULATED_EHR"
msh.msh_6 = "SIMULATED_HOSPITAL"

result_time = order_data["Result_Time"].iloc[0]

msh.msh_7 = result_time.strftime("%Y%m%d%H%M%S")
msh.msh_9 = "ORU^R01^ORU_R01"
msh.msh_10 = "MSG000001"
msh.msh_11 = "P"
msh.msh_12 = "2.5.1"

# PID: Patient Identification
patient_result = hl7_full.add_group(
    "ORU_R01_PATIENT_RESULT"
)

patient_group = patient_result.add_group(
    "ORU_R01_PATIENT"
)

pid = patient_group.add_segment("PID")

pid.pid_3 = f"{patient_id}^^^SIMULATED_HOSPITAL^MR"

pid.pid_5 = (
    f"{patient_data['Last_Name']}^"
    f"{patient_data['First_Name']}"
)

pid.pid_7 = pd.to_datetime(
    patient_data["Date_of_Birth"]
).strftime("%Y%m%d")

pid.pid_8 = patient_data["Sex"]

# OBR: Laboratory Order
order_group = patient_result.add_group(
    "ORU_R01_ORDER_OBSERVATION"
)

obr = order_group.add_segment("OBR")

obr.obr_1 = "1"
obr.obr_2 = selected_order_id
obr.obr_4 = "24323-8^Comprehensive metabolic panel^LN"

collection_time = order_data["Collection_Time"].iloc[0]

obr.obr_7 = collection_time.strftime("%Y%m%d%H%M%S")
obr.obr_22 = result_time.strftime("%Y%m%d%H%M%S")
obr.obr_25 = "F"

# OBX: Individual Laboratory Results
for index, row in enumerate(
    order_data.itertuples(), start=1
):

    observation = order_group.add_group(
        "ORU_R01_OBSERVATION"
    )

    obx = observation.add_segment("OBX")

    obx.obx_1 = str(index)
    obx.obx_2 = "NM"

    obx.obx_3 = (
        f"{row.LOINC_Code}^{row.Test_Name}^LN"
    )

    obx.obx_5 = str(row.Result)
    obx.obx_6 = row.Unit

    # Convert final status to HL7 result status code
    obx.obx_11 = "F"

# Display message
hl7_full_text = hl7_full.to_er7()

print("\nGenerated HL7 Message:")
print(hl7_full_text.replace("\r", "\n"))

# Verify segment counts
segments = [
    s for s in hl7_full_text.split("\r") if s
]

print("\nSegment Verification:")
print("MSH:", sum(s.startswith("MSH") for s in segments))
print("PID:", sum(s.startswith("PID") for s in segments))
print("OBR:", sum(s.startswith("OBR") for s in segments))
print("OBX:", sum(s.startswith("OBX") for s in segments))

# Validate HL7 structure
print("\nHL7 Structural Validation:")

try:
    hl7_full.validate()
    print("Validation Status: PASSED")

except ValidationError as error:
    print("Validation Status: REVIEW REQUIRED")
    print("Validation Details:", error)


Step 6F: HL7 Message with Patient Demographics
---------------------------------------------

Generated HL7 Message:
MSH|^~\&|SYNTHETIC_LIS|SIMULATED_HOSPITAL|SIMULATED_EHR|SIMULATED_HOSPITAL|20261016083800||ORU^R01^ORU_R01|MSG000001|P|2.5.1
PID|||P00001^^^SIMULATED_HOSPITAL^MR||TESTPATIENT0001^SYNTHETIC||19970513|F
OBR|1|ORD000001||24323-8^Comprehensive metabolic panel^LN|||20261016080200|||||||||||||||20261016083800|||F
OBX|1|NM|2951-2^Sodium^LN||144.8|mmol/L|||||F
OBX|2|NM|2823-3^Potassium^LN||3.7|mmol/L|||||F
OBX|3|NM|2075-0^Chloride^LN||97.8|mmol/L|||||F
OBX|4|NM|2028-9^CO2^LN||19.7|mmol/L|||||F
OBX|5|NM|2345-7^Glucose^LN||163.7|mg/dL|||||F
OBX|6|NM|3094-0^BUN^LN||24.1|mg/dL|||||F
OBX|7|NM|2160-0^Creatinine^LN||1.7|mg/dL|||||F
OBX|8|NM|17861-6^Calcium^LN||7.6|mg/dL|||||F
OBX|9|NM|1751-7^Albumin^LN||2.8|g/dL|||||F
OBX|10|NM|2885-2^Total Protein^LN||5.9|g/dL|||||F
OBX|11|NM|1975-2^Total Bilirubin^LN||1.5|mg/dL|||||F
OBX|12|NM|6768-6^ALP^LN||114.2|U/L|||||F
OBX|13|NM|1920-8^AST^LN||7


### Step 6F Results: HL7 Message Generation and Validation

The first complete HL7 v2.5.1 ORU^R01 message was successfully generated by integrating the synthetic patient demographics and laboratory datasets.

The message contained one MSH segment, one PID segment, one OBR segment, and 14 OBX segments representing the individual Comprehensive Metabolic Panel results. Patient identifiers, demographic information, laboratory order numbers, timestamps, LOINC identifiers, numerical results, and measurement units were incorporated into the message.

Structural validation using the hl7apy library returned a successful result, confirming that the message met the library's applicable HL7 v2.5.1 structural requirements.

This initial test demonstrates the transformation of structured laboratory data into an HL7 message while preserving the relationships between patients, laboratory orders, and test results. The validated message provides a foundation for expanding message generation to all 2,000 laboratory orders and subsequently parsing the transmitted information into PostgreSQL.



### Step 6G: Batch HL7 Message Generation and Validation

This step expands the validated HL7 v2.5.1 ORU^R01 message-generation process to all 2,000 laboratory orders in the synthetic dataset.

A reusable Python function will generate messages using patient demographics, laboratory order information, standardized panel identifiers, and individual laboratory results. Each message will receive a unique message control identifier to support message tracking and future database integration.

The process will generate messages for CBC, BMP, and CMP laboratory panels and validate their structure using the hl7apy library. Validation results will be recorded to identify successfully generated messages and any messages requiring additional review.

The generated HL7 messages will be preserved separately from the original source datasets for subsequent parsing, PostgreSQL integration, and data reconciliation.


In [19]:

# Step 6G-2: Define HL7 Laboratory Panel Identifiers

panel_loinc = {
    "CBC": {
        "code": "58410-2",
        "description": "CBC panel"
    },
    "BMP": {
        "code": "24321-2",
        "description": "Basic metabolic panel"
    },
    "CMP": {
        "code": "24323-8",
        "description": "Comprehensive metabolic panel"
    }
}

# Verify that every panel has a LOINC mapping
dataset_panels = set(df_hl7["Panel"].unique())
mapped_panels = set(panel_loinc.keys())

print("Step 6G-2: Panel Identifier Verification")
print("--------------------------------------")

print("Panels in Dataset:", sorted(dataset_panels))
print("Panels in Mapping:", sorted(mapped_panels))

missing_mappings = dataset_panels - mapped_panels

print("Missing Panel Mappings:", len(missing_mappings))

# Display mapping table
panel_mapping_df = pd.DataFrame([
    {
        "Panel": panel,
        "LOINC_Code": details["code"],
        "Description": details["description"]
    }
    for panel, details in panel_loinc.items()
])

display(panel_mapping_df)

if len(missing_mappings) == 0:
    print("\nPanel Mapping Status: PASSED")
else:
    print("\nPanel Mapping Status: REVIEW REQUIRED")
    print("Unmapped Panels:", missing_mappings)


Step 6G-2: Panel Identifier Verification
--------------------------------------
Panels in Dataset: ['BMP', 'CBC', 'CMP']
Panels in Mapping: ['BMP', 'CBC', 'CMP']
Missing Panel Mappings: 0


,Panel,LOINC_Code,Description
0,CBC,58410-2,CBC panel
1,BMP,24321-2,Basic metabolic panel
2,CMP,24323-8,Comprehensive metabolic panel



Panel Mapping Status: PASSED



#### Step 6G-3: Reusable HL7 Message-Generation Function

A reusable Python function is developed to generate HL7 v2.5.1 ORU^R01 messages from the validated laboratory and synthetic patient datasets.

The function retrieves the patient demographics associated with each laboratory order, identifies the appropriate laboratory panel, and constructs the MSH, PID, OBR, and OBX segments. Unique message control identifiers are assigned to support message tracking and future PostgreSQL integration.

The function also performs structural validation using the hl7apy library and returns the generated message along with its validation status. This approach supports consistent message generation across different laboratory panels while reducing repetitive code.


In [20]:

# Step 6G-3: Reusable HL7 Message Generation Function

from hl7apy.core import Message
from hl7apy.exceptions import ValidationError
import pandas as pd

# Create a patient lookup for efficient processing
patient_lookup = df_patients.set_index("Patient_ID")

def generate_hl7_message(order_data, message_id):

    # Verify one order and one patient per message
    if order_data["Order_ID"].nunique() != 1:
        raise ValueError("Message must contain exactly one order.")

    if order_data["Patient_ID"].nunique() != 1:
        raise ValueError("Order contains multiple patient IDs.")

    patient_id = order_data["Patient_ID"].iloc[0]
    order_id = order_data["Order_ID"].iloc[0]
    panel_name = order_data["Panel"].iloc[0]

    if order_data["Panel"].nunique() != 1:
        raise ValueError("Order contains multiple panel types.")

    if panel_name not in panel_loinc:
        raise ValueError(f"Unknown panel: {panel_name}")

    # Retrieve synthetic patient demographics
    patient = patient_lookup.loc[patient_id]

    collection_time = pd.to_datetime(
        order_data["Collection_Time"].iloc[0]
    )

    result_time = pd.to_datetime(
        order_data["Result_Time"].iloc[0]
    )

    # Create HL7 v2.5.1 message
    message = Message("ORU_R01", version="2.5.1")

    # MSH: Message Header
    msh = message.msh

    msh.msh_3 = "SYNTHETIC_LIS"
    msh.msh_4 = "SIMULATED_HOSPITAL"
    msh.msh_5 = "SIMULATED_EHR"
    msh.msh_6 = "SIMULATED_HOSPITAL"
    msh.msh_7 = result_time.strftime("%Y%m%d%H%M%S")
    msh.msh_9 = "ORU^R01^ORU_R01"
    msh.msh_10 = message_id
    msh.msh_11 = "P"
    msh.msh_12 = "2.5.1"

    # PID: Patient Identification
    patient_result = message.add_group(
        "ORU_R01_PATIENT_RESULT"
    )

    patient_group = patient_result.add_group(
        "ORU_R01_PATIENT"
    )

    pid = patient_group.add_segment("PID")

    pid.pid_3 = (
        f"{patient_id}^^^SIMULATED_HOSPITAL^MR"
    )

    pid.pid_5 = (
        f"{patient['Last_Name']}^"
        f"{patient['First_Name']}"
    )

    pid.pid_7 = pd.to_datetime(
        patient["Date_of_Birth"]
    ).strftime("%Y%m%d")

    pid.pid_8 = patient["Sex"]

    # OBR: Laboratory Order
    order_group = patient_result.add_group(
        "ORU_R01_ORDER_OBSERVATION"
    )

    obr = order_group.add_segment("OBR")

    obr.obr_1 = "1"
    obr.obr_2 = order_id

    panel_info = panel_loinc[panel_name]

    obr.obr_4 = (
        f"{panel_info['code']}^"
        f"{panel_info['description']}^LN"
    )

    obr.obr_7 = collection_time.strftime("%Y%m%d%H%M%S")
    obr.obr_22 = result_time.strftime("%Y%m%d%H%M%S")
    obr.obr_25 = "F"

    # OBX: Laboratory Results
    for index, row in enumerate(
        order_data.itertuples(index=False), start=1
    ):

        observation = order_group.add_group(
            "ORU_R01_OBSERVATION"
        )

        obx = observation.add_segment("OBX")

        obx.obx_1 = str(index)
        obx.obx_2 = "NM"

        obx.obx_3 = (
            f"{row.LOINC_Code}^{row.Test_Name}^LN"
        )

        obx.obx_5 = str(row.Result)
        obx.obx_6 = row.Unit

        obx.obx_11 = "F"

    # Validate message
    try:
        message.validate()
        validation_status = "PASSED"
        validation_error = None

    except ValidationError as error:
        validation_status = "FAILED"
        validation_error = str(error)

    return {
        "Message_ID": message_id,
        "Order_ID": order_id,
        "Patient_ID": patient_id,
        "Panel": panel_name,
        "Result_Count": len(order_data),
        "Validation_Status": validation_status,
        "Validation_Error": validation_error,
        "HL7_Message": message.to_er7()
    }

print("Step 6G-3: HL7 Generation Function Created")
print("-----------------------------------------")
print("Function Status: READY")


Step 6G-3: HL7 Generation Function Created
-----------------------------------------
Function Status: READY



#### Step 6G-4: HL7 Message Generation Testing Across Laboratory Panels

The reusable HL7 message-generation function is tested using one laboratory order from each panel: Complete Blood Count (CBC), Basic Metabolic Panel (BMP), and Comprehensive Metabolic Panel (CMP).

Each test message is generated using its corresponding patient demographics, laboratory order information, panel-level LOINC identifier, and individual laboratory observations.

The testing process evaluates message structure, observation counts, unique message control identifiers, and HL7 v2.5.1 structural validation results. Testing all three panel types helps identify potential issues before generating messages for the complete dataset of 2,000 laboratory orders.


In [21]:

# Step 6G-4: Test HL7 Generation Across All Panels

print("Step 6G-4: HL7 Panel Testing")
print("---------------------------")

test_messages = []

# Select one laboratory order from each panel
for index, panel_name in enumerate(
    ["CBC", "BMP", "CMP"], start=1
):

    panel_data = df_hl7[
        df_hl7["Panel"] == panel_name
    ]

    test_order_id = panel_data["Order_ID"].iloc[0]

    order_data = df_hl7[
        df_hl7["Order_ID"] == test_order_id
    ].copy()

    # Generate a unique test message ID
    message_id = f"TESTMSG{index:06d}"

    # Generate HL7 message
    result = generate_hl7_message(
        order_data,
        message_id
    )

    # Count OBX segments in generated message
    segments = result["HL7_Message"].split("\r")

    obx_count = sum(
        segment.startswith("OBX|")
        for segment in segments
    )

    result["OBX_Count"] = obx_count

    # Compare OBX count against source results
    result["Result_Count_Match"] = (
        obx_count == len(order_data)
    )

    test_messages.append(result)

# Create test summary
test_summary = pd.DataFrame([
    {
        "Panel": result["Panel"],
        "Order_ID": result["Order_ID"],
        "Message_ID": result["Message_ID"],
        "Expected_Results": result["Result_Count"],
        "OBX_Count": result["OBX_Count"],
        "Count_Match": result["Result_Count_Match"],
        "Validation_Status": result["Validation_Status"]
    }
    for result in test_messages
])

print("\nHL7 Panel Test Results:")
display(test_summary)

# Verify all tests passed
all_valid = (
    test_summary["Validation_Status"].eq("PASSED").all()
    and test_summary["Count_Match"].all()
    and test_summary["Message_ID"].is_unique
)

print("\nOverall Test Status:",
      "PASSED" if all_valid else "REVIEW REQUIRED")

# Display validation errors, if any
for result in test_messages:
    if result["Validation_Status"] != "PASSED":
        print(
            result["Panel"],
            "Validation Error:",
            result["Validation_Error"]
        )


Step 6G-4: HL7 Panel Testing
---------------------------

HL7 Panel Test Results:


,Panel,Order_ID,Message_ID,Expected_Results,OBX_Count,Count_Match,Validation_Status
0,CBC,ORD000002,TESTMSG000001,9,9,True,PASSED
1,BMP,ORD000003,TESTMSG000002,8,8,True,PASSED
2,CMP,ORD000001,TESTMSG000003,14,14,True,PASSED



Overall Test Status: PASSED



#### Step 6G-5: Batch HL7 Message Generation and Validation

The reusable HL7 message-generation function is applied to all 2,000 laboratory orders in the synthetic dataset. Each order is transformed into an HL7 v2.5.1 ORU^R01 message containing the corresponding patient demographics, laboratory order information, and individual test results.

Each message receives a unique message control identifier and undergoes structural validation using the hl7apy library. Message identifiers, patient identifiers, order numbers, panel types, observation counts, and validation outcomes are recorded for future database integration.

The batch-generation process also verifies that all laboratory orders have been processed and that the total number of HL7 observation segments matches the 20,612 results in the original dataset.

The generated messages and validation metadata will be preserved for subsequent HL7 parsing, PostgreSQL integration, and data reconciliation.


In [22]:

# Step 6G-5: Batch HL7 Message Generation

import pandas as pd
from time import perf_counter

print("Step 6G-5: Batch HL7 Message Generation")
print("--------------------------------------")

# Store generated messages and any processing errors
hl7_messages = []
generation_errors = []

start_time = perf_counter()

# Generate one message per laboratory order
for index, (order_id, order_data) in enumerate(
    df_hl7.groupby("Order_ID", sort=True),
    start=1
):

    message_id = f"MSG{index:06d}"

    try:
        result = generate_hl7_message(
            order_data,
            message_id
        )

        # Count OBX segments in the generated message
        segments = [
            s for s in result["HL7_Message"].split("\r")
            if s
        ]

        result["OBX_Count"] = sum(
            s.startswith("OBX|") for s in segments
        )

        result["Result_Count_Match"] = (
            result["OBX_Count"] == len(order_data)
        )

        hl7_messages.append(result)

    except Exception as error:
        generation_errors.append({
            "Order_ID": order_id,
            "Message_ID": message_id,
            "Error": str(error)
        })

# Measure processing time
processing_seconds = perf_counter() - start_time

# Create batch summary DataFrame
df_hl7_summary = pd.DataFrame(hl7_messages)

# Calculate validation statistics
total_messages = len(df_hl7_summary)

passed_messages = (
    df_hl7_summary["Validation_Status"]
    .eq("PASSED").sum()
)

failed_messages = total_messages - passed_messages

total_obx = df_hl7_summary["OBX_Count"].sum()

# Verify all expected orders and results
expected_orders = df_hl7["Order_ID"].nunique()
expected_results = len(df_hl7)

batch_passed = (
    total_messages == expected_orders
    and passed_messages == expected_orders
    and total_obx == expected_results
    and df_hl7_summary["Message_ID"].is_unique
    and df_hl7_summary["Order_ID"].is_unique
    and df_hl7_summary["Result_Count_Match"].all()
    and len(generation_errors) == 0
)

# Display results
print("\nBatch Generation Summary")
print("------------------------")

print("Expected Orders:", expected_orders)
print("Messages Generated:", total_messages)
print("Messages Passed Validation:", passed_messages)
print("Messages Failed Validation:", failed_messages)
print("Generation Errors:", len(generation_errors))
print("Total OBX Segments:", total_obx)
print("Expected Laboratory Results:", expected_results)

print(
    "Processing Time (seconds):",
    round(processing_seconds, 2)
)

print(
    "\nOverall Batch Status:",
    "PASSED" if batch_passed else "REVIEW REQUIRED"
)

# Display first five message records
print("\nFirst Five Message Records:")

display(
    df_hl7_summary[
        [
            "Message_ID",
            "Order_ID",
            "Patient_ID",
            "Panel",
            "OBX_Count",
            "Validation_Status"
        ]
    ].head()
)

# Display errors if any occurred
if generation_errors:
    print("\nGeneration Errors:")
    display(pd.DataFrame(generation_errors).head(10))

if failed_messages > 0:
    print("\nStructural Validation Errors:")
    display(
        df_hl7_summary[
            df_hl7_summary["Validation_Status"] != "PASSED"
        ][
            ["Message_ID", "Order_ID", "Validation_Error"]
        ].head(10)
    )


Step 6G-5: Batch HL7 Message Generation
--------------------------------------

Batch Generation Summary
------------------------
Expected Orders: 2000
Messages Generated: 2000
Messages Passed Validation: 2000
Messages Failed Validation: 0
Generation Errors: 0
Total OBX Segments: 20612
Expected Laboratory Results: 20612
Processing Time (seconds): 66.1

Overall Batch Status: PASSED

First Five Message Records:


,Message_ID,Order_ID,Patient_ID,Panel,OBX_Count,Validation_Status
0,MSG000001,ORD000001,P00001,CMP,14,PASSED
1,MSG000002,ORD000002,P00001,CBC,9,PASSED
2,MSG000003,ORD000003,P00002,BMP,8,PASSED
3,MSG000004,ORD000004,P00002,CBC,9,PASSED
4,MSG000005,ORD000005,P00003,CBC,9,PASSED



#### Step 6G-6: Exporting HL7 Messages and Validation Metadata

Following successful batch generation, all 2,000 HL7 v2.5.1 ORU^R01 messages are exported to persistent storage for future processing and database integration.

Each message is saved as an individual HL7 file, preserving its patient identification, laboratory order, panel information, and observation results. A separate CSV file records message control identifiers, patient identifiers, laboratory order numbers, panel types, observation counts, and structural validation outcomes.

These exported files provide the foundation for subsequent HL7 parsing, PostgreSQL relational database development, message tracking, and data reconciliation. The original laboratory and patient datasets remain unchanged.


In [23]:

# Step 6G-6: Export HL7 Messages and Validation Metadata

from pathlib import Path
import pandas as pd

print("Step 6G-6: HL7 Message Export")
print("----------------------------")

# Define project directory
project_dir = Path(
    r"C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project"
)

# Create output folders
hl7_dir = project_dir / "hl7_messages"
validation_dir = project_dir / "validation"

hl7_dir.mkdir(parents=True, exist_ok=True)
validation_dir.mkdir(parents=True, exist_ok=True)

# Prevent export if batch validation did not pass
if not batch_passed:
    raise RuntimeError(
        "Batch validation must pass before exporting messages."
    )

# Export individual HL7 messages
for record in hl7_messages:

    message_id = record["Message_ID"]

    output_path = hl7_dir / f"{message_id}.hl7"

    with open(output_path, "w", encoding="utf-8", newline="") as file:
        file.write(record["HL7_Message"] + "\r")

# Export validation metadata
metadata_columns = [
    "Message_ID",
    "Order_ID",
    "Patient_ID",
    "Panel",
    "Result_Count",
    "OBX_Count",
    "Result_Count_Match",
    "Validation_Status",
    "Validation_Error"
]

metadata_path = (
    validation_dir / "hl7_validation_summary.csv"
)

df_hl7_summary[metadata_columns].to_csv(
    metadata_path,
    index=False
)

# Verify exported files
saved_hl7_files = list(hl7_dir.glob("MSG*.hl7"))

saved_metadata = pd.read_csv(metadata_path)

print("\nHL7 Files Saved:", len(saved_hl7_files))
print("Validation Records Saved:", len(saved_metadata))

print("Messages Passed Validation:",
      saved_metadata["Validation_Status"].eq("PASSED").sum())

print("\nHL7 Folder:", hl7_dir)
print("Validation File:", metadata_path)

# Confirm export counts
export_passed = (
    len(saved_hl7_files) == 2000
    and len(saved_metadata) == 2000
    and saved_metadata["Validation_Status"].eq("PASSED").all()
)

print(
    "\nExport Status:",
    "PASSED" if export_passed else "REVIEW REQUIRED"
)


Step 6G-6: HL7 Message Export
----------------------------

HL7 Files Saved: 2000
Validation Records Saved: 2000
Messages Passed Validation: 2000

HL7 Folder: C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project\hl7_messages
Validation File: C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project\validation\hl7_validation_summary.csv

Export Status: PASSED



### Step 6 Results: HL7 Message Generation and Validation Summary

The HL7 v2.5.1 ORU^R01 message-generation process was successfully completed using synthetic patient demographics and laboratory results. A total of 2,000 HL7 messages were generated, representing 1,000 patients, 2,000 laboratory orders, and 20,612 individual laboratory observations across CBC, BMP, and CMP panels.

Initial testing confirmed that messages could be generated for all three laboratory panel types. A reusable Python function was then developed to automate message construction, assign unique message control identifiers, and perform structural validation using the hl7apy library.

All 2,000 messages passed structural validation, with no generation errors or mismatches between the expected laboratory result counts and the generated OBX segment counts. Batch processing required approximately 66.1 seconds.

The messages were exported as individual HL7 files, and a separate CSV file was created to preserve message identifiers, laboratory order associations, observation counts, and validation outcomes.

These results demonstrate successful synthetic HL7 message generation and structural validation. The next phase will examine whether the exported messages can be parsed accurately and whether their contents can be reconciled with the original source datasets before integration into PostgreSQL.



## Step 7: HL7 Message Parsing and Data Extraction

This step simulates the receiving side of healthcare interoperability by reading and parsing HL7 v2.5.1 ORU^R01 laboratory messages previously generated from the synthetic Laboratory Information System (LIS).

The parsing process extracts patient identifiers, demographic information, laboratory order details, test identifiers, numerical results, and timestamps from the HL7 messages. These records will later be transformed into relational datasets for PostgreSQL integration.

The extracted information will also be compared against the original synthetic source datasets to evaluate data completeness and consistency throughout the simulated exchange process.

### Step 7A: Reading and Inspecting a Saved HL7 Message

The first exported HL7 message is loaded from persistent storage and parsed using the hl7apy Python library. The process verifies that the message can be read successfully, identifies its message type and version, and examines the individual HL7 segments.

This initial test establishes that the exported message can be reconstructed from its saved ER7 format before expanding the parsing process to all 2,000 messages.


In [24]:

# Step 7A: Read and Parse a Saved HL7 Message

from pathlib import Path
from hl7apy.parser import parse_message
from hl7apy.exceptions import ValidationError

print("Step 7A: HL7 Message Parsing")
print("---------------------------")

# Locate saved HL7 message
project_dir = Path(
    r"C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project"
)

hl7_file = project_dir / "hl7_messages" / "MSG000001.hl7"

# Read the saved HL7 message
with open(hl7_file, "r", encoding="utf-8", newline="") as file:
    hl7_text = file.read()

# Remove only trailing segment delimiters
hl7_text = hl7_text.rstrip("\r\n")

# Parse HL7 v2.5.1 message
parsed_message = parse_message(
    hl7_text,
    find_groups=True,
    validation_level=2
)

# Display message information
print("File:", hl7_file.name)
print("Message Type:", parsed_message.msh.msh_9.to_er7())
print("HL7 Version:", parsed_message.msh.msh_12.to_er7())
print("Message Control ID:", parsed_message.msh.msh_10.to_er7())

# Count HL7 segments in saved message
segments = [
    segment for segment in hl7_text.split("\r")
    if segment
]

print("\nSegment Counts:")

for segment_type in ["MSH", "PID", "OBR", "OBX"]:
    count = sum(
        segment.startswith(segment_type + "|")
        for segment in segments
    )
    print(f"{segment_type}: {count}")

# Validate reconstructed message
print("\nParsed Message Validation:")

try:
    parsed_message.validate()
    print("Validation Status: PASSED")

except ValidationError as error:
    print("Validation Status: REVIEW REQUIRED")
    print("Validation Details:", error)


Step 7A: HL7 Message Parsing
---------------------------
File: MSG000001.hl7
Message Type: ORU^R01^ORU_R01
HL7 Version: 2.5.1
Message Control ID: MSG000001

Segment Counts:
MSH: 1
PID: 1
OBR: 1
OBX: 14

Parsed Message Validation:
Validation Status: PASSED



### Step 7B: Extracting Structured Patient, Order, and Laboratory Result Data

This step extracts clinical information from the parsed HL7 v2.5.1 ORU^R01 message and transforms it into structured Python datasets.

Patient demographics are retrieved from the PID segment, laboratory order information from the OBR segment, and individual laboratory observations from the OBX segments. The message control identifier is also preserved to maintain traceability between the original HL7 message and its extracted records.

The extracted information is organized into separate patient, laboratory order, and laboratory result DataFrames. This structure supports future PostgreSQL relational database development, including primary and foreign key relationships.

The extracted records will be reviewed for completeness and accuracy before expanding the process to all 2,000 saved HL7 messages.


In [25]:

# Step 7B: Extract Structured Data from HL7 Message

import pandas as pd

print("Step 7B: HL7 Data Extraction")
print("---------------------------")

# Access HL7 patient and order groups
patient_result = parsed_message.oru_r01_patient_result

patient_group = patient_result.oru_r01_patient
pid = patient_group.pid

order_group = patient_result.oru_r01_order_observation
obr = order_group.obr

# Extract message control identifier
message_id = parsed_message.msh.msh_10.value

# Extract patient demographics
patient_record = {
    "Patient_ID": pid.pid_3.cx_1.value,
    "Last_Name": pid.pid_5.xpn_1.value,
    "First_Name": pid.pid_5.xpn_2.value,
    "Date_of_Birth": pid.pid_7.value,
    "Sex": pid.pid_8.value
}

# Extract laboratory order information
order_record = {
    "Order_ID": obr.obr_2.ei_1.value,
    "Patient_ID": patient_record["Patient_ID"],
    "Message_ID": message_id,
    "Panel_LOINC": obr.obr_4.ce_1.value,
    "Panel_Name": obr.obr_4.ce_2.value,
    "Collection_Time": obr.obr_7.value,
    "Result_Time": obr.obr_22.value,
    "Order_Status": obr.obr_25.value
}

# Extract individual laboratory observations
result_records = []

for observation in order_group.oru_r01_observation:

    obx = observation.obx

    result_records.append({
        "Order_ID": order_record["Order_ID"],
        "Message_ID": message_id,
        "Observation_Number": obx.obx_1.value,
        "LOINC_Code": obx.obx_3.ce_1.value,
        "Test_Name": obx.obx_3.ce_2.value,
        "Result": obx.obx_5.value,
        "Unit": obx.obx_6.value,
        "Result_Status": obx.obx_11.value
    })

# Convert extracted records into DataFrames
df_patient_extracted = pd.DataFrame([patient_record])
df_order_extracted = pd.DataFrame([order_record])
df_results_extracted = pd.DataFrame(result_records)

# Display extracted data
print("\nExtracted Patient Record:")
display(df_patient_extracted)

print("\nExtracted Laboratory Order:")
display(df_order_extracted)

print("\nExtracted Laboratory Results:")
display(df_results_extracted)

# Verify extraction counts
print("\nExtraction Verification:")
print("Patient Records:", len(df_patient_extracted))
print("Laboratory Orders:", len(df_order_extracted))
print("Laboratory Results:", len(df_results_extracted))


Step 7B: HL7 Data Extraction
---------------------------

Extracted Patient Record:


,Patient_ID,Last_Name,First_Name,Date_of_Birth,Sex
0,P00001,TESTPATIENT0001,SYNTHETIC,19970513,F



Extracted Laboratory Order:


,Order_ID,Patient_ID,Message_ID,Panel_LOINC,Panel_Name,Collection_Time,Result_Time,Order_Status
0,ORD000001,P00001,MSG000001,24323-8,Comprehensive metabolic panel,20261016080200,20261016083800,F



Extracted Laboratory Results:


,Order_ID,Message_ID,Observation_Number,LOINC_Code,Test_Name,Result,Unit,Result_Status
0,ORD000001,MSG000001,1,2951-2,Sodium,144.8,mmol/L,F
1,ORD000001,MSG000001,2,2823-3,Potassium,3.7,mmol/L,F
2,ORD000001,MSG000001,3,2075-0,Chloride,97.8,mmol/L,F
3,ORD000001,MSG000001,4,2028-9,CO2,19.7,mmol/L,F
4,ORD000001,MSG000001,5,2345-7,Glucose,163.7,mg/dL,F
5,ORD000001,MSG000001,6,3094-0,BUN,24.1,mg/dL,F
6,ORD000001,MSG000001,7,2160-0,Creatinine,1.7,mg/dL,F
7,ORD000001,MSG000001,8,17861-6,Calcium,7.6,mg/dL,F
8,ORD000001,MSG000001,9,1751-7,Albumin,2.8,g/dL,F
9,ORD000001,MSG000001,10,2885-2,Total Protein,5.9,g/dL,F



Extraction Verification:
Patient Records: 1
Laboratory Orders: 1
Laboratory Results: 14



### Step 7C: Batch HL7 Message Parsing and Data Extraction

This step expands the HL7 parsing process to all 2,000 previously generated ORU^R01 laboratory messages.

Each saved HL7 file is read and parsed using the hl7apy Python library. Patient demographics, laboratory order information, and individual test results are extracted from the PID, OBR, and OBX segments and organized into structured datasets.

The batch process records message identifiers, processing outcomes, and any parsing errors. Duplicate patient records are consolidated using their unique patient identifiers, while laboratory orders and observations retain their original relationships.

The resulting datasets will be used for data reconciliation and PostgreSQL integration.


In [26]:

# Step 7C: Batch HL7 Parsing and Data Extraction

from pathlib import Path
from hl7apy.parser import parse_message
from time import perf_counter
import pandas as pd

print("Step 7C: Batch HL7 Parsing")
print("-------------------------")

# Locate saved HL7 messages
hl7_dir = project_dir / "hl7_messages"

hl7_files = sorted(hl7_dir.glob("MSG*.hl7"))

# Prepare extraction containers
patient_records = []
order_records = []
result_records = []
parsing_log = []

start_time = perf_counter()

# Process each saved HL7 message
for hl7_file in hl7_files:

    try:
        # Read saved message
        with open(
            hl7_file, "r", encoding="utf-8", newline=""
        ) as file:
            hl7_text = file.read().rstrip("\r\n")

        # Parse HL7 message
        message = parse_message(
            hl7_text,
            find_groups=True,
            validation_level=2
        )

        # Extract message and patient information
        message_id = message.msh.msh_10.value

        patient_result = message.oru_r01_patient_result
        pid = patient_result.oru_r01_patient.pid

        patient_id = pid.pid_3.cx_1.value

        patient_record = {
            "Patient_ID": patient_id,
            "Last_Name": pid.pid_5.xpn_1.value,
            "First_Name": pid.pid_5.xpn_2.value,
            "Date_of_Birth": pid.pid_7.value,
            "Sex": pid.pid_8.value
        }

        # Extract laboratory order
        order_group = (
            patient_result.oru_r01_order_observation
        )

        obr = order_group.obr

        order_id = obr.obr_2.ei_1.value

        order_record = {
            "Order_ID": order_id,
            "Patient_ID": patient_id,
            "Message_ID": message_id,
            "Panel_LOINC": obr.obr_4.ce_1.value,
            "Panel_Name": obr.obr_4.ce_2.value,
            "Collection_Time": obr.obr_7.value,
            "Result_Time": obr.obr_22.value,
            "Order_Status": obr.obr_25.value
        }

        # Extract individual laboratory results
        message_results = []

        for observation in order_group.oru_r01_observation:

            obx = observation.obx

            message_results.append({
                "Order_ID": order_id,
                "Message_ID": message_id,
                "Observation_Number": int(obx.obx_1.value),
                "LOINC_Code": obx.obx_3.ce_1.value,
                "Test_Name": obx.obx_3.ce_2.value,
                "Result": float(obx.obx_5.value),
                "Unit": obx.obx_6.value,
                "Result_Status": obx.obx_11.value
            })

        # Append only after complete extraction
        patient_records.append(patient_record)
        order_records.append(order_record)
        result_records.extend(message_results)

        parsing_log.append({
            "Message_ID": message_id,
            "File_Name": hl7_file.name,
            "Parsing_Status": "PASSED",
            "Results_Extracted": len(message_results),
            "Error": None
        })

    except Exception as error:

        parsing_log.append({
            "Message_ID": hl7_file.stem,
            "File_Name": hl7_file.name,
            "Parsing_Status": "FAILED",
            "Results_Extracted": 0,
            "Error": str(error)
        })

# Calculate processing time
processing_seconds = perf_counter() - start_time

# Create structured DataFrames
df_patients_parsed = (
    pd.DataFrame(patient_records)
    .drop_duplicates(subset=["Patient_ID"])
)

df_orders_parsed = pd.DataFrame(order_records)

df_results_parsed = pd.DataFrame(result_records)

df_parsing_log = pd.DataFrame(parsing_log)

# Calculate processing statistics
passed = df_parsing_log["Parsing_Status"].eq("PASSED").sum()
failed = df_parsing_log["Parsing_Status"].eq("FAILED").sum()

# Verify expected record counts
batch_parsing_passed = (
    len(hl7_files) == 2000
    and passed == 2000
    and failed == 0
    and len(df_patients_parsed) == 1000
    and len(df_orders_parsed) == 2000
    and len(df_results_parsed) == 20612
    and df_orders_parsed["Order_ID"].is_unique
    and df_orders_parsed["Message_ID"].is_unique
)

# Display summary
print("\nBatch Parsing Summary")
print("---------------------")

print("HL7 Files Processed:", len(hl7_files))
print("Messages Parsed Successfully:", passed)
print("Messages Failed:", failed)

print("Unique Patients:", len(df_patients_parsed))
print("Laboratory Orders:", len(df_orders_parsed))
print("Laboratory Results:", len(df_results_parsed))

print(
    "Processing Time (seconds):",
    round(processing_seconds, 2)
)

print(
    "\nOverall Parsing Status:",
    "PASSED" if batch_parsing_passed else "REVIEW REQUIRED"
)

# Display first five orders
print("\nFirst Five Parsed Laboratory Orders:")
display(df_orders_parsed.head())

# Display parsing errors if present
if failed > 0:
    print("\nParsing Errors:")
    display(
        df_parsing_log[
            df_parsing_log["Parsing_Status"] == "FAILED"
        ].head(10)
    )


Step 7C: Batch HL7 Parsing
-------------------------

Batch Parsing Summary
---------------------
HL7 Files Processed: 2000
Messages Parsed Successfully: 2000
Messages Failed: 0
Unique Patients: 1000
Laboratory Orders: 2000
Laboratory Results: 20612
Processing Time (seconds): 62.2

Overall Parsing Status: PASSED

First Five Parsed Laboratory Orders:


,Order_ID,Patient_ID,Message_ID,Panel_LOINC,Panel_Name,Collection_Time,Result_Time,Order_Status
0,ORD000001,P00001,MSG000001,24323-8,Comprehensive metabolic panel,20261016080200,20261016083800,F
1,ORD000002,P00001,MSG000002,58410-2,CBC panel,20261016080400,20261016102100,F
2,ORD000003,P00002,MSG000003,24321-2,Basic metabolic panel,20261016080600,20261016085900,F
3,ORD000004,P00002,MSG000004,58410-2,CBC panel,20261016080800,20261016085800,F
4,ORD000005,P00003,MSG000005,58410-2,CBC panel,20261016081000,20261016090500,F



### Step 7D: Data Reconciliation and Integrity Validation

This step evaluates whether the clinical information extracted from the HL7 messages matches the original synthetic source datasets.

Patient demographics, laboratory orders, and individual test results are compared to identify missing records, duplicate identifiers, mismatched values, and potential data integrity issues introduced during message generation or parsing.

The reconciliation process provides an additional level of verification beyond HL7 structural validation by examining the accuracy and completeness of the transmitted information.

#### Step 7D-1: Patient Demographics Reconciliation

The 1,000 patient records extracted from HL7 messages are compared against the original synthetic patient demographics dataset using Patient_ID as the unique identifier.

Patient names, dates of birth, and administrative sex values are standardized and compared to identify missing patients or inconsistencies between the source and parsed datasets.

Successful reconciliation will confirm that patient demographic information was preserved during HL7 message generation and parsing.


In [27]:

# Step 7D-1: Patient Demographics Reconciliation

import pandas as pd

print("Step 7D-1: Patient Demographics Reconciliation")
print("---------------------------------------------")

# Prepare original patient data
df_source_patients = df_patients.copy()

# Prepare parsed patient data
df_received_patients = df_patients_parsed.copy()

# Standardize date formats for comparison
df_source_patients["Date_of_Birth"] = pd.to_datetime(
    df_source_patients["Date_of_Birth"]
).dt.strftime("%Y%m%d")

df_received_patients["Date_of_Birth"] = pd.to_datetime(
    df_received_patients["Date_of_Birth"],
    format="%Y%m%d"
).dt.strftime("%Y%m%d")

# Check patient identifiers
source_ids = set(df_source_patients["Patient_ID"])
received_ids = set(df_received_patients["Patient_ID"])

missing_patients = source_ids - received_ids
unexpected_patients = received_ids - source_ids

# Check for duplicate patient identifiers
duplicate_patients = (
    df_received_patients["Patient_ID"].duplicated().sum()
)

# Compare demographic values
comparison = df_source_patients.merge(
    df_received_patients,
    on="Patient_ID",
    how="inner",
    suffixes=("_Source", "_HL7"),
    validate="one_to_one"
)

demographic_fields = [
    "Last_Name",
    "First_Name",
    "Date_of_Birth",
    "Sex"
]

# Count mismatches by field
mismatch_summary = {}

for field in demographic_fields:

    mismatch_summary[field] = (
        comparison[f"{field}_Source"]
        != comparison[f"{field}_HL7"]
    ).sum()

# Create reconciliation summary
df_patient_reconciliation = pd.DataFrame([
    {
        "Validation_Check": "Missing Patients",
        "Issues_Found": len(missing_patients)
    },
    {
        "Validation_Check": "Unexpected Patients",
        "Issues_Found": len(unexpected_patients)
    },
    {
        "Validation_Check": "Duplicate Patient IDs",
        "Issues_Found": duplicate_patients
    },
    *[
        {
            "Validation_Check": f"{field} Mismatches",
            "Issues_Found": int(count)
        }
        for field, count in mismatch_summary.items()
    ]
])

print("\nPatient Reconciliation Results:")
display(df_patient_reconciliation)

total_issues = df_patient_reconciliation["Issues_Found"].sum()

print("\nSource Patient Records:", len(df_source_patients))
print("Parsed Patient Records:", len(df_received_patients))
print("Total Issues Identified:", total_issues)

print(
    "Reconciliation Status:",
    "PASSED" if total_issues == 0 else "REVIEW REQUIRED"
)


Step 7D-1: Patient Demographics Reconciliation
---------------------------------------------

Patient Reconciliation Results:


,Validation_Check,Issues_Found
0,Missing Patients,0
1,Unexpected Patients,0
2,Duplicate Patient IDs,0
3,Last_Name Mismatches,0
4,First_Name Mismatches,0
5,Date_of_Birth Mismatches,0
6,Sex Mismatches,0



Source Patient Records: 1000
Parsed Patient Records: 1000
Total Issues Identified: 0
Reconciliation Status: PASSED



#### Step 7D-2: Laboratory Order Reconciliation

This step compares the 2,000 laboratory orders extracted from the HL7 messages against the original synthetic laboratory dataset.

Each order is evaluated for missing or unexpected identifiers, duplicate records, patient associations, laboratory panel assignments, and collection and result timestamps. Panel-level LOINC identifiers are also compared against the established laboratory panel mappings.

The reconciliation process determines whether laboratory order information was preserved during HL7 message generation and parsing. Identifying discrepancies at this stage helps prevent incorrect patient-order relationships or incomplete laboratory information from being transferred into PostgreSQL.


In [28]:

# Step 7D-2: Laboratory Order Reconciliation

import pandas as pd

print("Step 7D-2: Laboratory Order Reconciliation")
print("-----------------------------------------")

# Prepare original laboratory orders
df_source_orders = (
    df_hl7[
        ["Order_ID", "Patient_ID", "Panel",
         "Collection_Time", "Result_Time"]
    ]
    .drop_duplicates(subset=["Order_ID"])
    .copy()
)

# Convert source timestamps to HL7 format
for column in ["Collection_Time", "Result_Time"]:
    df_source_orders[column] = pd.to_datetime(
        df_source_orders[column]
    ).dt.strftime("%Y%m%d%H%M%S")

# Add expected panel LOINC identifiers
df_source_orders["Panel_LOINC"] = (
    df_source_orders["Panel"].map(
        lambda panel: panel_loinc[panel]["code"]
    )
)

# Prepare parsed laboratory orders
df_received_orders = df_orders_parsed.copy()

# Check order identifiers
source_ids = set(df_source_orders["Order_ID"])
received_ids = set(df_received_orders["Order_ID"])

missing_orders = source_ids - received_ids
unexpected_orders = received_ids - source_ids

duplicate_orders = (
    df_received_orders["Order_ID"].duplicated().sum()
)

# Compare source and received orders
order_comparison = df_source_orders.merge(
    df_received_orders,
    on="Order_ID",
    how="inner",
    suffixes=("_Source", "_HL7"),
    validate="one_to_one"
)

# Compare matching fields
order_fields = [
    "Patient_ID",
    "Panel_LOINC",
    "Collection_Time",
    "Result_Time"
]

mismatch_summary = {}

for field in order_fields:
    mismatch_summary[field] = int(
        (
            order_comparison[f"{field}_Source"].astype(str)
            != order_comparison[f"{field}_HL7"].astype(str)
        ).sum()
    )

# Verify that the parsed panel name matches its LOINC
expected_panel_names = {
    details["code"]: details["description"]
    for details in panel_loinc.values()
}

panel_name_mismatches = int(
    (
        df_received_orders["Panel_Name"]
        != df_received_orders["Panel_LOINC"].map(
            expected_panel_names
        )
    ).sum()
)

# Create reconciliation summary
df_order_reconciliation = pd.DataFrame([
    {
        "Validation_Check": "Missing Orders",
        "Issues_Found": len(missing_orders)
    },
    {
        "Validation_Check": "Unexpected Orders",
        "Issues_Found": len(unexpected_orders)
    },
    {
        "Validation_Check": "Duplicate Order IDs",
        "Issues_Found": int(duplicate_orders)
    },
    *[
        {
            "Validation_Check": f"{field} Mismatches",
            "Issues_Found": count
        }
        for field, count in mismatch_summary.items()
    ],
    {
        "Validation_Check": "Panel Name Mismatches",
        "Issues_Found": panel_name_mismatches
    }
])

# Calculate total issues
total_order_issues = int(
    df_order_reconciliation["Issues_Found"].sum()
)

# Display results
print("\nLaboratory Order Reconciliation Results:")
display(df_order_reconciliation)

print("\nSource Laboratory Orders:", len(df_source_orders))
print("Parsed Laboratory Orders:", len(df_received_orders))
print("Total Issues Identified:", total_order_issues)

print(
    "Reconciliation Status:",
    "PASSED" if total_order_issues == 0 else "REVIEW REQUIRED"
)


Step 7D-2: Laboratory Order Reconciliation
-----------------------------------------

Laboratory Order Reconciliation Results:


,Validation_Check,Issues_Found
0,Missing Orders,0
1,Unexpected Orders,0
2,Duplicate Order IDs,0
3,Patient_ID Mismatches,0
4,Panel_LOINC Mismatches,0
5,Collection_Time Mismatches,0
6,Result_Time Mismatches,0
7,Panel Name Mismatches,0



Source Laboratory Orders: 2000
Parsed Laboratory Orders: 2000
Total Issues Identified: 0
Reconciliation Status: PASSED



#### Step 7D-3: Individual Laboratory Result Reconciliation

This step compares all 20,612 laboratory observations extracted from the HL7 messages against the original synthetic laboratory dataset.

Each observation is matched using its laboratory order identifier and LOINC code. The reconciliation evaluates missing or unexpected observations, duplicate order-test combinations, test names, numerical results, measurement units, and result statuses.

Numerical values are compared using a small floating-point tolerance to account for potential differences in numerical representation during HL7 serialization and parsing.

This validation determines whether individual laboratory results were preserved accurately throughout the simulated LIS-to-EHR data exchange process. Successful reconciliation will establish a verified dataset for subsequent PostgreSQL integration.


In [29]:

# Step 7D-3: Laboratory Result Reconciliation

import pandas as pd
import numpy as np

print("Step 7D-3: Laboratory Result Reconciliation")
print("------------------------------------------")

# Prepare original laboratory results
df_source_results = df_hl7[
    ["Order_ID", "LOINC_Code", "Test_Name",
     "Result", "Unit", "Result_Status"]
].copy()

# Prepare HL7-extracted laboratory results
df_received_results = df_results_parsed[
    ["Order_ID", "LOINC_Code", "Test_Name",
     "Result", "Unit", "Result_Status"]
].copy()

# Standardize result status values
status_mapping = {
    "Final": "F",
    "Preliminary": "P",
    "Corrected": "C"
}

df_source_results["Result_Status"] = (
    df_source_results["Result_Status"]
    .replace(status_mapping)
)

# Define observation identifiers
key_columns = ["Order_ID", "LOINC_Code"]

# Identify duplicate observations
source_duplicates = int(
    df_source_results.duplicated(
        subset=key_columns
    ).sum()
)

received_duplicates = int(
    df_received_results.duplicated(
        subset=key_columns
    ).sum()
)

# Compare source and received observation identifiers
source_keys = pd.MultiIndex.from_frame(
    df_source_results[key_columns]
)

received_keys = pd.MultiIndex.from_frame(
    df_received_results[key_columns]
)

missing_results = source_keys.difference(received_keys)
unexpected_results = received_keys.difference(source_keys)

# Merge laboratory results for comparison
result_comparison = df_source_results.merge(
    df_received_results,
    on=key_columns,
    how="inner",
    suffixes=("_Source", "_HL7"),
    validate="one_to_one"
)

# Compare categorical fields
categorical_fields = [
    "Test_Name",
    "Unit",
    "Result_Status"
]

mismatch_summary = {}

for field in categorical_fields:
    mismatch_summary[field] = int(
        (
            result_comparison[f"{field}_Source"].astype(str)
            != result_comparison[f"{field}_HL7"].astype(str)
        ).sum()
    )

# Compare numerical laboratory results
source_values = pd.to_numeric(
    result_comparison["Result_Source"],
    errors="coerce"
)

received_values = pd.to_numeric(
    result_comparison["Result_HL7"],
    errors="coerce"
)

numeric_matches = np.isclose(
    source_values,
    received_values,
    rtol=1e-9,
    atol=1e-9,
    equal_nan=False
)

numeric_mismatches = int((~numeric_matches).sum())

# Create reconciliation summary
df_result_reconciliation = pd.DataFrame([
    {
        "Validation_Check": "Missing Laboratory Results",
        "Issues_Found": len(missing_results)
    },
    {
        "Validation_Check": "Unexpected Laboratory Results",
        "Issues_Found": len(unexpected_results)
    },
    {
        "Validation_Check": "Source Duplicate Observations",
        "Issues_Found": source_duplicates
    },
    {
        "Validation_Check": "HL7 Duplicate Observations",
        "Issues_Found": received_duplicates
    },
    *[
        {
            "Validation_Check": f"{field} Mismatches",
            "Issues_Found": count
        }
        for field, count in mismatch_summary.items()
    ],
    {
        "Validation_Check": "Numerical Result Mismatches",
        "Issues_Found": numeric_mismatches
    }
])

# Calculate total issues
total_result_issues = int(
    df_result_reconciliation["Issues_Found"].sum()
)

# Display results
print("\nLaboratory Result Reconciliation Results:")
display(df_result_reconciliation)

print("\nSource Laboratory Results:", len(df_source_results))
print("Parsed Laboratory Results:", len(df_received_results))
print("Total Issues Identified:", total_result_issues)

print(
    "Reconciliation Status:",
    "PASSED" if total_result_issues == 0 else "REVIEW REQUIRED"
)


Step 7D-3: Laboratory Result Reconciliation
------------------------------------------

Laboratory Result Reconciliation Results:


,Validation_Check,Issues_Found
0,Missing Laboratory Results,0
1,Unexpected Laboratory Results,0
2,Source Duplicate Observations,0
3,HL7 Duplicate Observations,0
4,Test_Name Mismatches,0
5,Unit Mismatches,0
6,Result_Status Mismatches,0
7,Numerical Result Mismatches,0



Source Laboratory Results: 20612
Parsed Laboratory Results: 20612
Total Issues Identified: 0
Reconciliation Status: PASSED



#### Step 7D-4: Patient Demographic Consistency Across HL7 Messages

This step evaluates the consistency of patient demographic information across all 2,000 parsed HL7 messages.

Because each synthetic patient is associated with two laboratory orders, patient identifiers and demographic values should remain consistent across both messages. The extracted patient records are grouped by Patient_ID and evaluated for differences in patient names, dates of birth, and administrative sex values.

This additional validation helps identify conflicting patient information that could otherwise be overlooked when duplicate patient records are consolidated. Successful verification supports accurate patient identification and reliable relational database integration.


In [30]:

# Step 7D-4: Patient Consistency Across HL7 Messages

import pandas as pd

print("Step 7D-4: Patient Demographic Consistency")
print("----------------------------------------")

# Reconstruct all patient records before deduplication
df_all_patient_messages = pd.DataFrame(patient_records)

# Demographic fields to compare
demographic_fields = [
    "Last_Name",
    "First_Name",
    "Date_of_Birth",
    "Sex"
]

# Count distinct values for each field per patient
consistency = (
    df_all_patient_messages
    .groupby("Patient_ID")[demographic_fields]
    .nunique(dropna=False)
)

# Identify patients with conflicting demographic values
conflicting_patients = consistency[
    (consistency > 1).any(axis=1)
]

# Verify each patient appears in exactly two messages
message_counts = (
    df_all_patient_messages
    .groupby("Patient_ID")
    .size()
)

incorrect_message_counts = int(
    (message_counts != 2).sum()
)

# Check for missing demographic values
missing_demographics = int(
    df_all_patient_messages[
        ["Patient_ID"] + demographic_fields
    ].isna().sum().sum()
)

# Display results
print("\nPatient Consistency Results")
print("---------------------------")

print("Patient Records Evaluated:",
      len(df_all_patient_messages))

print("Unique Patients:",
      df_all_patient_messages["Patient_ID"].nunique())

print("Patients With Conflicting Demographics:",
      len(conflicting_patients))

print("Patients Without Exactly Two Messages:",
      incorrect_message_counts)

print("Missing Demographic Values:",
      missing_demographics)

# Determine overall status
consistency_passed = (
    len(df_all_patient_messages) == 2000
    and df_all_patient_messages["Patient_ID"].nunique() == 1000
    and len(conflicting_patients) == 0
    and incorrect_message_counts == 0
    and missing_demographics == 0
)

print(
    "\nConsistency Status:",
    "PASSED" if consistency_passed else "REVIEW REQUIRED"
)

# Display conflicts if identified
if len(conflicting_patients) > 0:
    print("\nConflicting Patient Records:")
    display(conflicting_patients.head(10))


Step 7D-4: Patient Demographic Consistency
----------------------------------------

Patient Consistency Results
---------------------------
Patient Records Evaluated: 2000
Unique Patients: 1000
Patients With Conflicting Demographics: 0
Patients Without Exactly Two Messages: 0
Missing Demographic Values: 0

Consistency Status: PASSED



### Step 7E: Exporting Reconciled HL7 Data for PostgreSQL Integration

Following successful HL7 parsing and data reconciliation, the extracted patient demographics, laboratory orders, and individual laboratory results are exported into separate CSV files for PostgreSQL integration.

The exported datasets preserve the relationships between patients, laboratory orders, HL7 message identifiers, and laboratory observations. Parsing logs and reconciliation summaries are also retained to document message-processing outcomes and data integrity checks.

The original synthetic source datasets and HL7 message files remain unchanged, allowing future comparisons between the source information, transmitted messages, and relational database records.

These exports establish the foundation for PostgreSQL table creation, primary and foreign key relationships, SQL-based validation, and subsequent Power BI reporting.


In [31]:

# Step 7E: Export Reconciled HL7 Data

from pathlib import Path
import pandas as pd

print("Step 7E: PostgreSQL Data Preparation")
print("-----------------------------------")

# Require successful validation before export
validation_checks = {
    "Batch Parsing": batch_parsing_passed,
    "Patient Reconciliation": total_issues == 0,
    "Order Reconciliation": total_order_issues == 0,
    "Result Reconciliation": total_result_issues == 0,
    "Patient Consistency": consistency_passed
}

if not all(validation_checks.values()):
    raise RuntimeError(
        "One or more reconciliation checks did not pass."
    )

# Create export directories
parsed_dir = project_dir / "parsed_data"
validation_dir = project_dir / "validation"

parsed_dir.mkdir(parents=True, exist_ok=True)
validation_dir.mkdir(parents=True, exist_ok=True)

# Define export datasets
export_datasets = {
    "hl7_patients_parsed.csv": df_patients_parsed,
    "hl7_orders_parsed.csv": df_orders_parsed,
    "hl7_results_parsed.csv": df_results_parsed
}

# Save parsed datasets
for filename, dataframe in export_datasets.items():

    output_path = parsed_dir / filename

    dataframe.to_csv(output_path, index=False)

    print(f"Saved: {filename} ({len(dataframe):,} records)")

# Save parsing log
parsing_log_path = (
    validation_dir / "hl7_parsing_log.csv"
)

df_parsing_log.to_csv(
    parsing_log_path,
    index=False
)

# Combine reconciliation summaries
reconciliation_tables = {
    "Patient Demographics": df_patient_reconciliation,
    "Laboratory Orders": df_order_reconciliation,
    "Laboratory Results": df_result_reconciliation
}

combined_reconciliation = pd.concat(
    [
        dataframe.assign(Category=category)
        for category, dataframe in reconciliation_tables.items()
    ],
    ignore_index=True
)

# Save reconciliation summary
reconciliation_path = (
    validation_dir / "hl7_reconciliation_summary.csv"
)

combined_reconciliation.to_csv(
    reconciliation_path,
    index=False
)

# Verify saved datasets
print("\nExport Verification")
print("-------------------")

export_valid = True

for filename, dataframe in export_datasets.items():

    saved = pd.read_csv(parsed_dir / filename)

    count_match = len(saved) == len(dataframe)

    export_valid = export_valid and count_match

    print(
        f"{filename}:",
        "PASSED" if count_match else "FAILED"
    )

print("\nParsing Log Records:", len(df_parsing_log))
print("Reconciliation Checks:", len(combined_reconciliation))

print(
    "\nOverall Export Status:",
    "PASSED" if export_valid else "REVIEW REQUIRED"
)


Step 7E: PostgreSQL Data Preparation
-----------------------------------
Saved: hl7_patients_parsed.csv (1,000 records)
Saved: hl7_orders_parsed.csv (2,000 records)
Saved: hl7_results_parsed.csv (20,612 records)

Export Verification
-------------------
hl7_patients_parsed.csv: PASSED
hl7_orders_parsed.csv: PASSED
hl7_results_parsed.csv: PASSED

Parsing Log Records: 2000
Reconciliation Checks: 23

Overall Export Status: PASSED



### Step 7 Results: HL7 Parsing, Data Extraction, and Reconciliation

The HL7 parsing and data extraction process was successfully completed for all 2,000 synthetic HL7 v2.5.1 ORU^R01 laboratory messages. Using Python and the hl7apy library, each saved message was read and parsed to extract patient demographics, laboratory order information, and individual test results.

The process produced structured datasets containing 1,000 unique patients, 2,000 laboratory orders, and 20,612 laboratory observations. All messages were parsed successfully, with no processing failures. Batch parsing required approximately 62.2 seconds.

Data reconciliation was performed to compare the extracted information against the original synthetic source datasets. All 23 patient, laboratory order, and laboratory result validation checks identified zero discrepancies. An additional consistency assessment confirmed that patient demographics remained unchanged across the two HL7 messages associated with each patient.

Following successful reconciliation, the extracted datasets were exported into separate CSV files for PostgreSQL integration. Parsing logs and reconciliation summaries were also saved to preserve processing outcomes and support future data quality analysis.

These results demonstrate that the tested patient, order, and laboratory result information was preserved throughout the simulated LIS-to-EHR message exchange. The next phase will focus on developing a PostgreSQL relational database to store the extracted records, establish data relationships, and support SQL-based analysis and Power BI reporting.



## Step 8: PostgreSQL Relational Database Development

This phase focuses on developing a PostgreSQL relational database to store and manage the patient demographics, laboratory orders, and test results extracted from the HL7 messages.

The database will establish relationships between patients, laboratory orders, laboratory tests, and individual observations. SQL queries will be used to verify data integrity, evaluate laboratory workflows, and prepare structured information for Power BI reporting.

### Step 8A: PostgreSQL Environment Verification

The PostgreSQL environment is evaluated to identify the installed database version and confirm the availability of the database management tools before establishing a connection through Python.

This initial verification helps ensure that the existing PostgreSQL installation can support the relational database development phase.


In [32]:

# Step 8A: PostgreSQL Installation Verification

from pathlib import Path
import subprocess

print("Step 8A: PostgreSQL Environment Verification")
print("--------------------------------------------")

# Check common PostgreSQL installation locations
postgres_base = Path(r"C:\Program Files\PostgreSQL")

if postgres_base.exists():

    installations = sorted(
        [folder for folder in postgres_base.iterdir()
         if folder.is_dir()]
    )

    print("\nPostgreSQL Installation Directories:")

    for folder in installations:
        print("-", folder.name)

        psql_path = folder / "bin" / "psql.exe"

        if psql_path.exists():

            result = subprocess.run(
                [str(psql_path), "--version"],
                capture_output=True,
                text=True
            )

            print("  ", result.stdout.strip())

else:
    print("PostgreSQL not found in the default directory.")
    print("It may be installed in another location.")

print("\nEnvironment Verification Complete")


Step 8A: PostgreSQL Environment Verification
--------------------------------------------

PostgreSQL Installation Directories:
- 17
   psql (PostgreSQL) 17.4

Environment Verification Complete



### Step 8B: PostgreSQL Server Verification

Following confirmation of the PostgreSQL 17.4 installation, the local database server is evaluated to determine whether it is running and accepting connections.

The verification process checks the PostgreSQL server's availability through the pg_isready utility. This establishes whether the database environment is accessible before configuring the Python database connection and developing the relational schema.

The verification does not modify existing databases or database configurations.


In [33]:

# Step 8B: PostgreSQL Server Verification

from pathlib import Path
import subprocess

print("Step 8B: PostgreSQL Server Verification")
print("--------------------------------------")

# Locate PostgreSQL server readiness utility
pg_isready = Path(
    r"C:\Program Files\PostgreSQL\17\bin\pg_isready.exe"
)

if pg_isready.exists():

    result = subprocess.run(
        [
            str(pg_isready),
            "-h", "localhost",
            "-p", "5432"
        ],
        capture_output=True,
        text=True,
        timeout=10
    )

    print("\nServer Response:")
    print(result.stdout.strip())

    if result.returncode == 0:
        print("\nServer Connection Status: READY")
    else:
        print("\nServer Connection Status: REVIEW REQUIRED")
        print(result.stderr.strip())

else:
    print("PostgreSQL readiness utility not found.")
    print("Verification Status: REVIEW REQUIRED")


Step 8B: PostgreSQL Server Verification
--------------------------------------

Server Response:
localhost:5432 - accepting connections

Server Connection Status: READY



### Step 8C: Python and PostgreSQL Connection Setup

This step establishes an authenticated connection between the Jupyter Notebook environment and the local PostgreSQL database server.

The psycopg Python library is used to support database connectivity and SQL execution. The connection will be verified before creating the relational database schema or importing the HL7-extracted datasets.

Database credentials will be entered securely during execution rather than stored directly in the notebook to avoid exposing sensitive connection information.


In [34]:

# Step 8C-1: Install PostgreSQL Python Connector

%pip install "psycopg[binary]"

import psycopg
from importlib.metadata import version

print("Step 8C-1: PostgreSQL Connector Verification")
print("-------------------------------------------")
print("Psycopg Version:", version("psycopg"))
print("Connector Status: READY")


   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.7 MB 4.8 MB/s eta 0:00:01
   -------------------- ------------------- 1.8/3.7 MB 4.8 MB/s eta 0:00:01
   ---------------------------- ----------- 2.6/3.7 MB 4.6 MB/s eta 0:00:01
   ---------------------------------- ----- 3.1/3.7 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 3.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Step 8C-1: PostgreSQL Connector Verification
-------------------------------------------
Psycopg Version: 3.3.6
Connector Status: READY


In [35]:

# Step 8C-2: Authenticated PostgreSQL Connection

import psycopg
from getpass import getpass

print("Step 8C-2: PostgreSQL Connection Verification")
print("---------------------------------------------")

# Request password without displaying it
db_password = getpass("Enter PostgreSQL password: ")

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="postgres",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            cursor.execute("""
                SELECT
                    current_database(),
                    current_user,
                    current_setting('server_version');
            """)

            database, username, server_version = cursor.fetchone()

            print("\nDatabase:", database)
            print("Connected User:", username)
            print("PostgreSQL Server Version:", server_version)
            print("Connection Status: PASSED")

except psycopg.Error as error:
    print("\nConnection Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Check the username, password, and database settings.")

finally:
    # Avoid retaining the password in a notebook variable
    del db_password


Step 8C-2: PostgreSQL Connection Verification
---------------------------------------------


Enter PostgreSQL password:  ········



Database: postgres
Connected User: postgres
PostgreSQL Server Version: 17.4
Connection Status: PASSED



### Step 8D: Relational Database Schema Design

A relational database schema is designed to organize the patient demographics, laboratory orders, laboratory test definitions, individual laboratory results, and HL7 message metadata extracted during the interoperability simulation.

The proposed PostgreSQL database contains five primary tables:

1. **patients:** Stores unique synthetic patient identifiers and demographic information.
2. **hl7_messages:** Stores HL7 message control identifiers, processing statuses, and validation information.
3. **laboratory_orders:** Stores laboratory order identifiers, patient associations, panel information, collection timestamps, and result timestamps.
4. **laboratory_tests:** Stores standardized laboratory test identifiers, test names, and measurement units.
5. **laboratory_results:** Stores individual laboratory observations and their associated laboratory orders and tests.

Primary keys will uniquely identify records within each table, while foreign keys will preserve relationships between patients, orders, messages, and results.

This structure reduces unnecessary duplication and supports SQL-based data integrity validation, laboratory turnaround time analysis, HL7 message monitoring, and future Power BI reporting.



### Step 8E: PostgreSQL Database Creation

A dedicated PostgreSQL database is created to store the structured healthcare information extracted from the synthetic HL7 v2.5.1 laboratory messages.

The database, named hl7_healthcare_interoperability, will support relational storage of patient demographics, laboratory orders, laboratory test definitions, individual results, and HL7 message-processing metadata.

The database is created separately from PostgreSQL's default administrative database to maintain an organized project environment. Its existence is verified before proceeding with relational table creation, primary and foreign key constraints, and data import.

All records used in this project are synthetic and contain no actual patient information or protected health information (PHI).


In [36]:

# Step 8E: Create PostgreSQL Project Database

import psycopg
from psycopg import sql
from getpass import getpass

print("Step 8E: PostgreSQL Database Creation")
print("-------------------------------------")

database_name = "hl7_healthcare_interoperability"

# Request PostgreSQL password securely
db_password = getpass("Enter PostgreSQL password: ")

try:
    # Connect to the default PostgreSQL database
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="postgres",
        user="postgres",
        password=db_password,
        autocommit=True,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Check whether project database already exists
            cursor.execute(
                "SELECT 1 FROM pg_database WHERE datname = %s",
                (database_name,)
            )

            database_exists = cursor.fetchone() is not None

            if database_exists:
                print("\nProject Database Already Exists")
            else:
                # Create dedicated project database
                cursor.execute(
                    sql.SQL("CREATE DATABASE {}").format(
                        sql.Identifier(database_name)
                    )
                )

                print("\nProject Database Created Successfully")

            # Verify database exists
            cursor.execute(
                "SELECT datname FROM pg_database WHERE datname = %s",
                (database_name,)
            )

            verified_database = cursor.fetchone()

            if verified_database:
                print("Database Name:", verified_database[0])
                print("Database Verification: PASSED")
            else:
                print("Database Verification: REVIEW REQUIRED")

except psycopg.Error as error:
    print("\nDatabase Creation Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    # Avoid retaining the password in a notebook variable
    del db_password


Step 8E: PostgreSQL Database Creation
-------------------------------------


Enter PostgreSQL password:  ········



Project Database Created Successfully
Database Name: hl7_healthcare_interoperability
Database Verification: PASSED



### Step 8F: PostgreSQL Relational Table Creation

Five relational tables are created within the hl7_healthcare_interoperability database to organize the synthetic patient demographics, laboratory orders, laboratory test definitions, individual laboratory results, and HL7 message-processing metadata.

Primary keys uniquely identify records within each table, while foreign keys establish relationships between patients, laboratory orders, laboratory tests, laboratory results, and HL7 messages.

The database schema uses appropriate PostgreSQL data types, including VARCHAR for identifiers, DATE for patient birth dates, TIMESTAMP for laboratory collection and result times, and NUMERIC for laboratory measurements.

The schema also includes constraints to support referential integrity and prevent duplicate or invalid records. Table creation will be verified before importing the HL7-extracted datasets.


In [37]:

# Step 8F: Create PostgreSQL Relational Tables

import psycopg
from getpass import getpass

print("Step 8F: PostgreSQL Relational Table Creation")
print("--------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

table_definitions = [
    """
    CREATE TABLE IF NOT EXISTS patients (
        patient_id VARCHAR(20) PRIMARY KEY,
        last_name VARCHAR(100) NOT NULL,
        first_name VARCHAR(100) NOT NULL,
        date_of_birth DATE NOT NULL,
        sex VARCHAR(10) NOT NULL
    );
    """,

    """
    CREATE TABLE IF NOT EXISTS hl7_messages (
        message_id VARCHAR(30) PRIMARY KEY,
        order_id VARCHAR(30) NOT NULL UNIQUE,
        patient_id VARCHAR(20) NOT NULL,
        panel VARCHAR(10) NOT NULL,
        result_count INTEGER NOT NULL CHECK (result_count >= 0),
        obx_count INTEGER NOT NULL CHECK (obx_count >= 0),
        validation_status VARCHAR(30) NOT NULL,
        parsing_status VARCHAR(30),
        validation_error TEXT
    );
    """,

    """
    CREATE TABLE IF NOT EXISTS laboratory_tests (
        loinc_code VARCHAR(20) PRIMARY KEY,
        test_name VARCHAR(100) NOT NULL,
        unit VARCHAR(30) NOT NULL
    );
    """,

    """
    CREATE TABLE IF NOT EXISTS laboratory_orders (
        order_id VARCHAR(30) PRIMARY KEY,
        patient_id VARCHAR(20) NOT NULL,
        message_id VARCHAR(30) NOT NULL UNIQUE,
        panel_loinc VARCHAR(20) NOT NULL,
        panel_name VARCHAR(100) NOT NULL,
        collection_time TIMESTAMP NOT NULL,
        result_time TIMESTAMP NOT NULL,
        order_status VARCHAR(10) NOT NULL,

        CONSTRAINT fk_order_patient
            FOREIGN KEY (patient_id)
            REFERENCES patients(patient_id),

        CONSTRAINT fk_order_message
            FOREIGN KEY (message_id)
            REFERENCES hl7_messages(message_id),

        CONSTRAINT chk_order_times
            CHECK (result_time > collection_time)
    );
    """,

    """
    CREATE TABLE IF NOT EXISTS laboratory_results (
        result_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        order_id VARCHAR(30) NOT NULL,
        message_id VARCHAR(30) NOT NULL,
        observation_number INTEGER NOT NULL,
        loinc_code VARCHAR(20) NOT NULL,
        result_value NUMERIC(12,4) NOT NULL,
        unit VARCHAR(30) NOT NULL,
        result_status VARCHAR(10) NOT NULL,

        CONSTRAINT fk_result_order
            FOREIGN KEY (order_id)
            REFERENCES laboratory_orders(order_id),

        CONSTRAINT fk_result_test
            FOREIGN KEY (loinc_code)
            REFERENCES laboratory_tests(loinc_code),

        CONSTRAINT uq_order_observation
            UNIQUE (order_id, observation_number),

        CONSTRAINT uq_order_test
            UNIQUE (order_id, loinc_code)
    );
    """
]

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            for definition in table_definitions:
                cursor.execute(definition)

        connection.commit()

    print("\nTable Creation Commands Executed")

    # Verify tables in database
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            cursor.execute("""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = 'public'
                  AND table_type = 'BASE TABLE'
                ORDER BY table_name;
            """)

            created_tables = [
                row[0] for row in cursor.fetchall()
            ]

    expected_tables = {
        "patients",
        "hl7_messages",
        "laboratory_tests",
        "laboratory_orders",
        "laboratory_results"
    }

    print("\nPostgreSQL Tables:")

    for table in created_tables:
        print("-", table)

    missing_tables = expected_tables - set(created_tables)

    print("\nExpected Tables:", len(expected_tables))
    print("Required Tables Found:",
          len(expected_tables - missing_tables))

    print(
        "\nTable Creation Status:",
        "PASSED" if not missing_tables else "REVIEW REQUIRED"
    )

except psycopg.Error as error:
    print("\nTable Creation Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 8F: PostgreSQL Relational Table Creation
--------------------------------------------


Enter PostgreSQL password:  ········



Table Creation Commands Executed

PostgreSQL Tables:
- hl7_messages
- laboratory_orders
- laboratory_results
- laboratory_tests
- patients

Expected Tables: 5
Required Tables Found: 5

Table Creation Status: PASSED



### Step 8G: PostgreSQL Schema and Constraint Verification

Following successful creation of the five relational tables, the PostgreSQL database schema is examined to verify primary keys, foreign keys, unique constraints, and data integrity rules.

The verification process queries PostgreSQL system metadata to identify the constraints associated with each table and confirm that the expected relationships were established.

This assessment helps ensure that patient identifiers, laboratory order identifiers, laboratory test codes, and HL7 message identifiers are appropriately linked before importing the extracted clinical data.

The verification is performed without modifying existing tables or records.


In [38]:

# Step 8G: PostgreSQL Schema and Constraint Verification

import psycopg
import pandas as pd
from getpass import getpass

print("Step 8G: PostgreSQL Schema Verification")
print("--------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            cursor.execute("""
                SELECT
                    cls.relname AS table_name,
                    con.conname AS constraint_name,
                    CASE con.contype
                        WHEN 'p' THEN 'PRIMARY KEY'
                        WHEN 'f' THEN 'FOREIGN KEY'
                        WHEN 'u' THEN 'UNIQUE'
                        WHEN 'c' THEN 'CHECK'
                        ELSE 'OTHER'
                    END AS constraint_type,
                    pg_get_constraintdef(con.oid)
                        AS constraint_definition
                FROM pg_constraint con
                JOIN pg_class cls
                    ON cls.oid = con.conrelid
                JOIN pg_namespace ns
                    ON ns.oid = cls.relnamespace
                WHERE ns.nspname = 'public'
                  AND cls.relname IN (
                      'patients',
                      'hl7_messages',
                      'laboratory_tests',
                      'laboratory_orders',
                      'laboratory_results'
                  )
                ORDER BY
                    cls.relname,
                    constraint_type,
                    con.conname;
            """)

            records = cursor.fetchall()

            columns = [
                "Table",
                "Constraint_Name",
                "Constraint_Type",
                "Definition"
            ]

            df_constraints = pd.DataFrame(
                records,
                columns=columns
            )

    print("\nDatabase Constraints:")
    display(df_constraints)

    print("\nConstraint Summary:")

    constraint_summary = (
        df_constraints["Constraint_Type"]
        .value_counts()
        .rename_axis("Constraint_Type")
        .reset_index(name="Count")
    )

    display(constraint_summary)

    expected_primary_keys = 5
    expected_foreign_keys = 4

    actual_primary_keys = (
        df_constraints["Constraint_Type"]
        .eq("PRIMARY KEY").sum()
    )

    actual_foreign_keys = (
        df_constraints["Constraint_Type"]
        .eq("FOREIGN KEY").sum()
    )

    schema_passed = (
        actual_primary_keys == expected_primary_keys
        and actual_foreign_keys == expected_foreign_keys
    )

    print("\nPrimary Keys Found:", actual_primary_keys)
    print("Foreign Keys Found:", actual_foreign_keys)

    print(
        "\nSchema Verification Status:",
        "PASSED" if schema_passed else "REVIEW REQUIRED"
    )

except psycopg.Error as error:
    print("\nSchema Verification Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 8G: PostgreSQL Schema Verification
--------------------------------------


Enter PostgreSQL password:  ········



Database Constraints:


,Table,Constraint_Name,Constraint_Type,Definition
0,hl7_messages,hl7_messages_obx_count_check,CHECK,CHECK ((obx_count >= 0))
1,hl7_messages,hl7_messages_result_count_check,CHECK,CHECK ((result_count >= 0))
2,hl7_messages,hl7_messages_pkey,PRIMARY KEY,PRIMARY KEY (message_id)
3,hl7_messages,hl7_messages_order_id_key,UNIQUE,UNIQUE (order_id)
4,laboratory_orders,chk_order_times,CHECK,CHECK ((result_time > collection_time))
5,laboratory_orders,fk_order_message,FOREIGN KEY,FOREIGN KEY (message_id) REFERENCES hl7_messag...
6,laboratory_orders,fk_order_patient,FOREIGN KEY,FOREIGN KEY (patient_id) REFERENCES patients(p...
7,laboratory_orders,laboratory_orders_pkey,PRIMARY KEY,PRIMARY KEY (order_id)
8,laboratory_orders,laboratory_orders_message_id_key,UNIQUE,UNIQUE (message_id)
9,laboratory_results,fk_result_order,FOREIGN KEY,FOREIGN KEY (order_id) REFERENCES laboratory_o...



Constraint Summary:


,Constraint_Type,Count
0,PRIMARY KEY,5
1,UNIQUE,4
2,FOREIGN KEY,4
3,CHECK,3



Primary Keys Found: 5
Foreign Keys Found: 4

Schema Verification Status: PASSED



### Step 8H: Strengthening Referential Integrity

Additional relational constraints are implemented to strengthen data integrity across the PostgreSQL database.

The HL7 message table is linked directly to the patient table through a foreign key. Composite foreign-key relationships are also established to verify that laboratory orders reference the correct patient-message combination and that individual laboratory results reference the correct order-message combination.

These constraints provide additional protection against mismatched patient identifiers, incorrect laboratory order associations, and inconsistencies between HL7 message metadata and extracted clinical records.

The database relationships are verified before importing the reconciled HL7 datasets.


In [39]:

# Step 8H: Strengthen PostgreSQL Referential Integrity

import psycopg
import pandas as pd
from getpass import getpass

print("Step 8H: Referential Integrity Enhancement")
print("-----------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

constraint_commands = [
    """
    ALTER TABLE hl7_messages
    ADD CONSTRAINT fk_message_patient
    FOREIGN KEY (patient_id)
    REFERENCES patients(patient_id);
    """,

    """
    ALTER TABLE hl7_messages
    ADD CONSTRAINT uq_message_patient
    UNIQUE (message_id, patient_id);
    """,

    """
    ALTER TABLE laboratory_orders
    ADD CONSTRAINT fk_order_message_patient
    FOREIGN KEY (message_id, patient_id)
    REFERENCES hl7_messages(message_id, patient_id);
    """,

    """
    ALTER TABLE laboratory_orders
    ADD CONSTRAINT uq_order_message
    UNIQUE (order_id, message_id);
    """,

    """
    ALTER TABLE laboratory_results
    ADD CONSTRAINT fk_result_order_message
    FOREIGN KEY (order_id, message_id)
    REFERENCES laboratory_orders(order_id, message_id);
    """
]

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Verify all five tables are empty
            table_names = [
                "patients",
                "hl7_messages",
                "laboratory_tests",
                "laboratory_orders",
                "laboratory_results"
            ]

            for table in table_names:
                cursor.execute(
                    f"SELECT COUNT(*) FROM {table}"
                )

                if cursor.fetchone()[0] != 0:
                    raise RuntimeError(
                        "Tables must be empty before this step."
                    )

            # Add additional constraints
            for command in constraint_commands:
                cursor.execute(command)

        connection.commit()

    print("\nAdditional Constraints Created Successfully")

    # Verify new constraints
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            cursor.execute("""
                SELECT
                    con.conname,
                    pg_get_constraintdef(con.oid)
                FROM pg_constraint con
                JOIN pg_namespace ns
                    ON ns.oid = con.connamespace
                WHERE ns.nspname = 'public'
                  AND con.conname IN (
                      'fk_message_patient',
                      'uq_message_patient',
                      'fk_order_message_patient',
                      'uq_order_message',
                      'fk_result_order_message'
                  )
                ORDER BY con.conname;
            """)

            constraints = cursor.fetchall()

    df_new_constraints = pd.DataFrame(
        constraints,
        columns=["Constraint_Name", "Definition"]
    )

    print("\nNew Database Constraints:")
    display(df_new_constraints)

    print("\nExpected New Constraints: 5")
    print("Constraints Found:", len(df_new_constraints))

    print(
        "\nReferential Integrity Status:",
        "PASSED" if len(df_new_constraints) == 5
        else "REVIEW REQUIRED"
    )

except (psycopg.Error, RuntimeError) as error:
    print("\nReferential Integrity Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 8H: Referential Integrity Enhancement
-----------------------------------------


Enter PostgreSQL password:  ········



Additional Constraints Created Successfully

New Database Constraints:


,Constraint_Name,Definition
0,fk_message_patient,FOREIGN KEY (patient_id) REFERENCES patients(p...
1,fk_order_message_patient,"FOREIGN KEY (message_id, patient_id) REFERENCE..."
2,fk_result_order_message,"FOREIGN KEY (order_id, message_id) REFERENCES ..."
3,uq_message_patient,"UNIQUE (message_id, patient_id)"
4,uq_order_message,"UNIQUE (order_id, message_id)"



Expected New Constraints: 5
Constraints Found: 5

Referential Integrity Status: PASSED



### Step 8I: HL7 Data Preparation and PostgreSQL Import

This step prepares the validated HL7-extracted datasets for integration into the PostgreSQL relational database.

The process includes verifying the exported source files, examining column structures and data types, preparing records to match the database schema, and importing the information in an order that preserves primary and foreign key relationships.

Patient demographics, laboratory test definitions, HL7 message metadata, laboratory orders, and individual laboratory results will be evaluated before insertion.

Following the import, SQL queries will be used to verify record counts, relational integrity, and consistency with the original HL7-extracted datasets.

#### Step 8I-1: Import File and Column Verification

The exported HL7 datasets and message validation summary are examined to confirm file availability, record counts, and column names before PostgreSQL import.

This verification identifies potential differences between the exported data structures and the relational database schema. The source files remain unchanged during this process.


In [40]:

# Step 8I-1: PostgreSQL Import File Verification

from pathlib import Path
import pandas as pd

print("Step 8I-1: PostgreSQL Import File Verification")
print("---------------------------------------------")

project_path = Path(
    r"C:\Users\angel\OneDrive\Projects\HL7 Healthcare Interoperability Project"
)

# Define required import files and expected record counts
import_files = {
    "Patients": (
        project_path / "parsed_data" / "hl7_patients_parsed.csv",
        1000
    ),
    "Laboratory Orders": (
        project_path / "parsed_data" / "hl7_orders_parsed.csv",
        2000
    ),
    "Laboratory Results": (
        project_path / "parsed_data" / "hl7_results_parsed.csv",
        20612
    ),
    "HL7 Validation Summary": (
        project_path / "validation" / "hl7_validation_summary.csv",
        2000
    )
}

loaded_datasets = {}
verification_records = []

for dataset_name, (file_path, expected_count) in import_files.items():

    if file_path.exists():

        df = pd.read_csv(file_path, dtype=str)

        loaded_datasets[dataset_name] = df

        actual_count = len(df)

        status = (
            "PASSED"
            if actual_count == expected_count
            else "REVIEW REQUIRED"
        )

        verification_records.append({
            "Dataset": dataset_name,
            "Expected_Records": expected_count,
            "Actual_Records": actual_count,
            "Status": status
        })

        print(f"\n{dataset_name}")
        print("-" * len(dataset_name))
        print("File:", file_path.name)
        print("Records:", actual_count)
        print("Columns:", list(df.columns))

    else:

        verification_records.append({
            "Dataset": dataset_name,
            "Expected_Records": expected_count,
            "Actual_Records": 0,
            "Status": "FILE NOT FOUND"
        })

# Display verification summary
df_import_verification = pd.DataFrame(
    verification_records
)

print("\nImport File Verification Summary:")
display(df_import_verification)

all_files_valid = (
    df_import_verification["Status"] == "PASSED"
).all()

print(
    "\nImport File Verification Status:",
    "PASSED" if all_files_valid else "REVIEW REQUIRED"
)


Step 8I-1: PostgreSQL Import File Verification
---------------------------------------------

Patients
--------
File: hl7_patients_parsed.csv
Records: 1000
Columns: ['Patient_ID', 'Last_Name', 'First_Name', 'Date_of_Birth', 'Sex']

Laboratory Orders
-----------------
File: hl7_orders_parsed.csv
Records: 2000
Columns: ['Order_ID', 'Patient_ID', 'Message_ID', 'Panel_LOINC', 'Panel_Name', 'Collection_Time', 'Result_Time', 'Order_Status']

Laboratory Results
------------------
File: hl7_results_parsed.csv
Records: 20612
Columns: ['Order_ID', 'Message_ID', 'Observation_Number', 'LOINC_Code', 'Test_Name', 'Result', 'Unit', 'Result_Status']

HL7 Validation Summary
----------------------
File: hl7_validation_summary.csv
Records: 2000
Columns: ['Message_ID', 'Order_ID', 'Patient_ID', 'Panel', 'Result_Count', 'OBX_Count', 'Result_Count_Match', 'Validation_Status', 'Validation_Error']

Import File Verification Summary:


,Dataset,Expected_Records,Actual_Records,Status
0,Patients,1000,1000,PASSED
1,Laboratory Orders,2000,2000,PASSED
2,Laboratory Results,20612,20612,PASSED
3,HL7 Validation Summary,2000,2000,PASSED



Import File Verification Status: PASSED



#### Step 8I-2: Patient Data Preparation for PostgreSQL

The 1,000 patient records extracted from the HL7 messages are prepared for insertion into the PostgreSQL patients table.

The preparation process standardizes column names, converts HL7-formatted dates of birth into PostgreSQL-compatible DATE values, and verifies patient identifiers and demographic information.

Additional checks evaluate missing values, duplicate patient identifiers, valid date formats, and field lengths to ensure compatibility with the database schema.

The original extracted CSV file remains unchanged. The prepared dataset will be used during the PostgreSQL import process.


In [41]:

# Step 8I-2: Patient Data Preparation

import pandas as pd

print("Step 8I-2: Patient Data Preparation")
print("----------------------------------")

# Load verified patient dataset
df_patients_sql = loaded_datasets["Patients"].copy()

# Standardize column names
df_patients_sql = df_patients_sql.rename(columns={
    "Patient_ID": "patient_id",
    "Last_Name": "last_name",
    "First_Name": "first_name",
    "Date_of_Birth": "date_of_birth",
    "Sex": "sex"
})

# Convert HL7 date format (YYYYMMDD) to date
df_patients_sql["date_of_birth"] = pd.to_datetime(
    df_patients_sql["date_of_birth"],
    format="%Y%m%d",
    errors="coerce"
).dt.date

# Validate required fields
required_columns = [
    "patient_id",
    "last_name",
    "first_name",
    "date_of_birth",
    "sex"
]

missing_values = int(
    df_patients_sql[required_columns].isna().sum().sum()
)

# Check blank text fields
text_columns = [
    "patient_id",
    "last_name",
    "first_name",
    "sex"
]

blank_values = int(
    df_patients_sql[text_columns]
    .apply(lambda col: col.str.strip().eq(""))
    .sum().sum()
)

# Check duplicate patient identifiers
duplicate_patients = int(
    df_patients_sql["patient_id"].duplicated().sum()
)

# Verify database field lengths
field_limits = {
    "patient_id": 20,
    "last_name": 100,
    "first_name": 100,
    "sex": 10
}

length_issues = sum(
    int(df_patients_sql[column].str.len().gt(limit).sum())
    for column, limit in field_limits.items()
)

# Verify expected patient count
record_count_valid = len(df_patients_sql) == 1000

# Determine overall preparation status
patient_preparation_passed = (
    record_count_valid
    and missing_values == 0
    and blank_values == 0
    and duplicate_patients == 0
    and length_issues == 0
)

# Display verification results
print("\nPatient Data Verification")
print("-------------------------")
print("Total Patient Records:", len(df_patients_sql))
print("Unique Patient IDs:",
      df_patients_sql["patient_id"].nunique())
print("Missing Values:", missing_values)
print("Blank Text Values:", blank_values)
print("Duplicate Patient IDs:", duplicate_patients)
print("Field Length Issues:", length_issues)

print(
    "\nPatient Preparation Status:",
    "PASSED" if patient_preparation_passed
    else "REVIEW REQUIRED"
)

print("\nFirst Five Prepared Patient Records:")
display(df_patients_sql.head())


Step 8I-2: Patient Data Preparation
----------------------------------

Patient Data Verification
-------------------------
Total Patient Records: 1000
Unique Patient IDs: 1000
Missing Values: 0
Blank Text Values: 0
Duplicate Patient IDs: 0
Field Length Issues: 0

Patient Preparation Status: PASSED

First Five Prepared Patient Records:


,patient_id,last_name,first_name,date_of_birth,sex
0,P00001,TESTPATIENT0001,SYNTHETIC,1997-05-13,F
1,P00002,TESTPATIENT0002,SYNTHETIC,1942-03-30,M
2,P00003,TESTPATIENT0003,SYNTHETIC,1961-12-20,F
3,P00004,TESTPATIENT0004,SYNTHETIC,1952-07-08,F
4,P00005,TESTPATIENT0005,SYNTHETIC,2000-09-16,F



#### Step 8I-3: Laboratory Test Definition Preparation

The laboratory test definitions are extracted from the 20,612 parsed HL7 observations to create a standardized reference dataset for PostgreSQL.

Each laboratory test is identified by its LOINC code and associated with a test name and measurement unit. Duplicate observations are consolidated into unique test definitions, reducing unnecessary repetition within the relational database.

The preparation process verifies that each LOINC code corresponds to a consistent test name and measurement unit. Missing values, duplicate identifiers, and field lengths are also evaluated to ensure compatibility with the PostgreSQL laboratory_tests table.

The resulting reference dataset supports consistent laboratory test identification and establishes the relationship between standardized test definitions and individual laboratory results.


In [42]:

# Step 8I-3: Laboratory Test Definition Preparation

import pandas as pd

print("Step 8I-3: Laboratory Test Definition Preparation")
print("------------------------------------------------")

# Load parsed HL7 laboratory results
df_test_source = loaded_datasets["Laboratory Results"].copy()

# Extract laboratory test reference fields
df_tests_sql = (
    df_test_source[
        ["LOINC_Code", "Test_Name", "Unit"]
    ]
    .drop_duplicates()
    .rename(columns={
        "LOINC_Code": "loinc_code",
        "Test_Name": "test_name",
        "Unit": "unit"
    })
    .sort_values("loinc_code")
    .reset_index(drop=True)
)

# Check for inconsistent LOINC mappings
mapping_counts = (
    df_tests_sql.groupby("loinc_code")
    .agg(
        Test_Names=("test_name", "nunique"),
        Units=("unit", "nunique")
    )
)

inconsistent_mappings = int(
    (
        (mapping_counts["Test_Names"] > 1)
        | (mapping_counts["Units"] > 1)
    ).sum()
)

# Check missing values
missing_values = int(
    df_tests_sql.isna().sum().sum()
)

# Check blank text values
blank_values = int(
    df_tests_sql
    .apply(lambda col: col.str.strip().eq(""))
    .sum().sum()
)

# Check duplicate LOINC identifiers
duplicate_loinc = int(
    df_tests_sql["loinc_code"].duplicated().sum()
)

# Verify PostgreSQL field lengths
field_limits = {
    "loinc_code": 20,
    "test_name": 100,
    "unit": 30
}

length_issues = sum(
    int(df_tests_sql[column].str.len().gt(limit).sum())
    for column, limit in field_limits.items()
)

# Verify expected number of laboratory tests
expected_tests = 23

test_preparation_passed = (
    len(df_tests_sql) == expected_tests
    and inconsistent_mappings == 0
    and missing_values == 0
    and blank_values == 0
    and duplicate_loinc == 0
    and length_issues == 0
)

# Display verification results
print("\nLaboratory Test Verification")
print("----------------------------")
print("Total Test Definitions:", len(df_tests_sql))
print("Unique LOINC Codes:",
      df_tests_sql["loinc_code"].nunique())
print("Inconsistent LOINC Mappings:", inconsistent_mappings)
print("Missing Values:", missing_values)
print("Blank Text Values:", blank_values)
print("Duplicate LOINC Codes:", duplicate_loinc)
print("Field Length Issues:", length_issues)

print(
    "\nTest Preparation Status:",
    "PASSED" if test_preparation_passed
    else "REVIEW REQUIRED"
)

print("\nPrepared Laboratory Test Definitions:")
display(df_tests_sql)


Step 8I-3: Laboratory Test Definition Preparation
------------------------------------------------

Laboratory Test Verification
----------------------------
Total Test Definitions: 23
Unique LOINC Codes: 23
Inconsistent LOINC Mappings: 0
Missing Values: 0
Blank Text Values: 0
Duplicate LOINC Codes: 0
Field Length Issues: 0

Test Preparation Status: PASSED

Prepared Laboratory Test Definitions:


,loinc_code,test_name,unit
0,1742-6,ALT,U/L
1,1751-7,Albumin,g/dL
2,17861-6,Calcium,mg/dL
3,1920-8,AST,U/L
4,1975-2,Total Bilirubin,mg/dL
5,2028-9,CO2,mmol/L
6,2075-0,Chloride,mmol/L
7,2160-0,Creatinine,mg/dL
8,2345-7,Glucose,mg/dL
9,2823-3,Potassium,mmol/L



#### Step 8I-4: HL7 Message Metadata Preparation

The 2,000 HL7 message validation records are prepared for insertion into the PostgreSQL hl7_messages table.

Each record contains a unique message control identifier, laboratory order identifier, patient identifier, laboratory panel, expected result count, actual OBX segment count, and validation status.

The preparation process standardizes column names, converts numerical fields to appropriate integer types, and evaluates missing identifiers, duplicate records, field lengths, and inconsistencies between expected and actual observation counts.

Message validation outcomes are retained to support traceability, database integrity, and future analysis of interoperability performance. The original validation summary remains unchanged.


In [43]:

# Step 8I-4: HL7 Message Metadata Preparation

import pandas as pd

print("Step 8I-4: HL7 Message Metadata Preparation")
print("------------------------------------------")

# Load verified HL7 validation summary
df_messages_sql = loaded_datasets[
    "HL7 Validation Summary"
].copy()

# Standardize column names
df_messages_sql = df_messages_sql.rename(columns={
    "Message_ID": "message_id",
    "Order_ID": "order_id",
    "Patient_ID": "patient_id",
    "Panel": "panel",
    "Result_Count": "result_count",
    "OBX_Count": "obx_count",
    "Validation_Status": "validation_status",
    "Validation_Error": "validation_error"
})

# Convert numerical fields
for column in ["result_count", "obx_count"]:
    df_messages_sql[column] = pd.to_numeric(
        df_messages_sql[column],
        errors="coerce"
    )

# Add parsing status from completed batch verification
# All 2,000 messages were successfully parsed in Step 7C
df_messages_sql["parsing_status"] = "PASSED"

# Select columns matching PostgreSQL schema
sql_columns = [
    "message_id",
    "order_id",
    "patient_id",
    "panel",
    "result_count",
    "obx_count",
    "validation_status",
    "parsing_status",
    "validation_error"
]

df_messages_sql = df_messages_sql[sql_columns]

# Validate required database fields
required_columns = [
    column for column in sql_columns
    if column != "validation_error"
]

missing_values = int(
    df_messages_sql[required_columns].isna().sum().sum()
)

# Check blank required text fields
text_columns = [
    "message_id",
    "order_id",
    "patient_id",
    "panel",
    "validation_status",
    "parsing_status"
]

blank_values = int(
    df_messages_sql[text_columns]
    .apply(lambda col: col.str.strip().eq(""))
    .sum().sum()
)

# Check duplicate identifiers
duplicate_messages = int(
    df_messages_sql["message_id"].duplicated().sum()
)

duplicate_orders = int(
    df_messages_sql["order_id"].duplicated().sum()
)

# Verify result counts match OBX segment counts
count_mismatches = int(
    (
        df_messages_sql["result_count"]
        != df_messages_sql["obx_count"]
    ).sum()
)

# Check nonnegative integer values
invalid_counts = int(
    (
        df_messages_sql[["result_count", "obx_count"]]
        .lt(0)
        | df_messages_sql[["result_count", "obx_count"]]
        .mod(1).ne(0)
    ).sum().sum()
)

# Verify PostgreSQL field lengths
field_limits = {
    "message_id": 30,
    "order_id": 30,
    "patient_id": 20,
    "panel": 10,
    "validation_status": 30,
    "parsing_status": 30
}

length_issues = sum(
    int(df_messages_sql[column].str.len().gt(limit).sum())
    for column, limit in field_limits.items()
)

# Verify expected message count
expected_messages = 2000

message_preparation_passed = (
    len(df_messages_sql) == expected_messages
    and missing_values == 0
    and blank_values == 0
    and duplicate_messages == 0
    and duplicate_orders == 0
    and count_mismatches == 0
    and invalid_counts == 0
    and length_issues == 0
)

# Display verification results
print("\nHL7 Message Verification")
print("------------------------")
print("Total Messages:", len(df_messages_sql))
print("Unique Message IDs:",
      df_messages_sql["message_id"].nunique())
print("Duplicate Message IDs:", duplicate_messages)
print("Duplicate Order IDs:", duplicate_orders)
print("Missing Required Values:", missing_values)
print("Blank Required Values:", blank_values)
print("Result/OBX Count Mismatches:", count_mismatches)
print("Invalid Numerical Counts:", invalid_counts)
print("Field Length Issues:", length_issues)

print(
    "\nMessage Preparation Status:",
    "PASSED" if message_preparation_passed
    else "REVIEW REQUIRED"
)

print("\nFirst Five Prepared HL7 Message Records:")
display(df_messages_sql.head())


Step 8I-4: HL7 Message Metadata Preparation
------------------------------------------

HL7 Message Verification
------------------------
Total Messages: 2000
Unique Message IDs: 2000
Duplicate Message IDs: 0
Duplicate Order IDs: 0
Missing Required Values: 0
Blank Required Values: 0
Result/OBX Count Mismatches: 0
Invalid Numerical Counts: 0
Field Length Issues: 0

Message Preparation Status: PASSED

First Five Prepared HL7 Message Records:


,message_id,order_id,patient_id,panel,result_count,obx_count,validation_status,parsing_status,validation_error
0,MSG000001,ORD000001,P00001,CMP,14,14,PASSED,PASSED,NaN
1,MSG000002,ORD000002,P00001,CBC,9,9,PASSED,PASSED,NaN
2,MSG000003,ORD000003,P00002,BMP,8,8,PASSED,PASSED,NaN
3,MSG000004,ORD000004,P00002,CBC,9,9,PASSED,PASSED,NaN
4,MSG000005,ORD000005,P00003,CBC,9,9,PASSED,PASSED,NaN



#### Step 8I-5: Laboratory Order Preparation for PostgreSQL

The 2,000 laboratory orders extracted from the HL7 messages are prepared for insertion into the PostgreSQL laboratory_orders table.

The preparation process standardizes column names, converts HL7-formatted collection and result timestamps into PostgreSQL-compatible datetime values, and verifies unique order identifiers.

Additional validation checks confirm that each laboratory order is associated with the correct patient and HL7 message, that panel identifiers are consistent with the message metadata, and that laboratory result timestamps occur after specimen collection.

These checks support the relational integrity of the database and help ensure that laboratory order information remains accurate before import. The original HL7-extracted CSV file remains unchanged.


In [44]:

# Step 8I-5: Laboratory Order Preparation

import pandas as pd

print("Step 8I-5: Laboratory Order Preparation")
print("--------------------------------------")

# Load verified HL7-extracted laboratory orders
df_orders_sql = loaded_datasets["Laboratory Orders"].copy()

# Standardize column names
df_orders_sql = df_orders_sql.rename(columns={
    "Order_ID": "order_id",
    "Patient_ID": "patient_id",
    "Message_ID": "message_id",
    "Panel_LOINC": "panel_loinc",
    "Panel_Name": "panel_name",
    "Collection_Time": "collection_time",
    "Result_Time": "result_time",
    "Order_Status": "order_status"
})

# Convert HL7 timestamps (YYYYMMDDHHMMSS)
for column in ["collection_time", "result_time"]:
    df_orders_sql[column] = pd.to_datetime(
        df_orders_sql[column],
        format="%Y%m%d%H%M%S",
        errors="coerce"
    )

# Verify required fields
missing_values = int(
    df_orders_sql.isna().sum().sum()
)

# Check blank text values
text_columns = [
    "order_id",
    "patient_id",
    "message_id",
    "panel_loinc",
    "panel_name",
    "order_status"
]

blank_values = int(
    df_orders_sql[text_columns]
    .apply(lambda col: col.str.strip().eq(""))
    .sum().sum()
)

# Check duplicate identifiers
duplicate_orders = int(
    df_orders_sql["order_id"].duplicated().sum()
)

duplicate_messages = int(
    df_orders_sql["message_id"].duplicated().sum()
)

# Verify timestamps
invalid_timestamps = int(
    (
        df_orders_sql["result_time"]
        <= df_orders_sql["collection_time"]
    ).sum()
)

# Verify patient identifiers exist
unknown_patients = int(
    (~df_orders_sql["patient_id"].isin(
        df_patients_sql["patient_id"]
    )).sum()
)

# Verify order-message-patient relationships
message_reference = df_messages_sql[
    ["message_id", "order_id", "patient_id", "panel"]
].rename(columns={
    "order_id": "message_order_id",
    "patient_id": "message_patient_id"
})

order_comparison = df_orders_sql.merge(
    message_reference,
    on="message_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

missing_messages = int(
    order_comparison["_merge"].ne("both").sum()
)

order_mismatches = int(
    (
        order_comparison["order_id"]
        != order_comparison["message_order_id"]
    ).sum()
)

patient_mismatches = int(
    (
        order_comparison["patient_id"]
        != order_comparison["message_patient_id"]
    ).sum()
)

# Verify panel mappings
expected_panel_loinc = {
    "CBC": "58410-2",
    "BMP": "24321-2",
    "CMP": "24323-8"
}

panel_mismatches = int(
    (
        order_comparison["panel_loinc"]
        != order_comparison["panel"].map(
            expected_panel_loinc
        )
    ).sum()
)

# Verify PostgreSQL field lengths
field_limits = {
    "order_id": 30,
    "patient_id": 20,
    "message_id": 30,
    "panel_loinc": 20,
    "panel_name": 100,
    "order_status": 10
}

length_issues = sum(
    int(df_orders_sql[column].str.len().gt(limit).sum())
    for column, limit in field_limits.items()
)

# Determine preparation status
order_preparation_passed = (
    len(df_orders_sql) == 2000
    and missing_values == 0
    and blank_values == 0
    and duplicate_orders == 0
    and duplicate_messages == 0
    and invalid_timestamps == 0
    and unknown_patients == 0
    and missing_messages == 0
    and order_mismatches == 0
    and patient_mismatches == 0
    and panel_mismatches == 0
    and length_issues == 0
)

# Display verification results
print("\nLaboratory Order Verification")
print("-----------------------------")
print("Total Laboratory Orders:", len(df_orders_sql))
print("Duplicate Order IDs:", duplicate_orders)
print("Duplicate Message IDs:", duplicate_messages)
print("Missing Values:", missing_values)
print("Blank Text Values:", blank_values)
print("Invalid Timestamps:", invalid_timestamps)
print("Unknown Patient IDs:", unknown_patients)
print("Missing HL7 Messages:", missing_messages)
print("Order ID Mismatches:", order_mismatches)
print("Patient ID Mismatches:", patient_mismatches)
print("Panel LOINC Mismatches:", panel_mismatches)
print("Field Length Issues:", length_issues)

print(
    "\nOrder Preparation Status:",
    "PASSED" if order_preparation_passed
    else "REVIEW REQUIRED"
)

print("\nFirst Five Prepared Laboratory Orders:")
display(df_orders_sql.head())


Step 8I-5: Laboratory Order Preparation
--------------------------------------

Laboratory Order Verification
-----------------------------
Total Laboratory Orders: 2000
Duplicate Order IDs: 0
Duplicate Message IDs: 0
Missing Values: 0
Blank Text Values: 0
Invalid Timestamps: 0
Unknown Patient IDs: 0
Missing HL7 Messages: 0
Order ID Mismatches: 0
Patient ID Mismatches: 0
Panel LOINC Mismatches: 0
Field Length Issues: 0

Order Preparation Status: PASSED

First Five Prepared Laboratory Orders:


,order_id,patient_id,message_id,panel_loinc,panel_name,collection_time,result_time,order_status
0,ORD000001,P00001,MSG000001,24323-8,Comprehensive metabolic panel,2026-10-16 08:02:00,2026-10-16 08:38:00,F
1,ORD000002,P00001,MSG000002,58410-2,CBC panel,2026-10-16 08:04:00,2026-10-16 10:21:00,F
2,ORD000003,P00002,MSG000003,24321-2,Basic metabolic panel,2026-10-16 08:06:00,2026-10-16 08:59:00,F
3,ORD000004,P00002,MSG000004,58410-2,CBC panel,2026-10-16 08:08:00,2026-10-16 08:58:00,F
4,ORD000005,P00003,MSG000005,58410-2,CBC panel,2026-10-16 08:10:00,2026-10-16 09:05:00,F



#### Step 8I-6: Laboratory Result Preparation for PostgreSQL

The 20,612 laboratory observations extracted from the HL7 messages are prepared for insertion into the PostgreSQL laboratory_results table.

The preparation process standardizes column names, converts laboratory measurements and observation sequence numbers into appropriate numerical data types, and verifies the relationships between laboratory results, orders, HL7 messages, and standardized LOINC test definitions.

Additional checks evaluate missing values, duplicate observations, invalid numerical measurements, inconsistent test identifiers, measurement units, and result statuses.

These validations help ensure that individual laboratory observations retain their original values and associations throughout the interoperability workflow. The prepared dataset will support relational database integration, laboratory data quality analysis, and subsequent Power BI reporting.


In [45]:

# Step 8I-6: Laboratory Result Preparation

import pandas as pd
import numpy as np

print("Step 8I-6: Laboratory Result Preparation")
print("---------------------------------------")

# Load verified HL7-extracted laboratory results
df_results_sql = loaded_datasets[
    "Laboratory Results"
].copy()

# Standardize column names
df_results_sql = df_results_sql.rename(columns={
    "Order_ID": "order_id",
    "Message_ID": "message_id",
    "Observation_Number": "observation_number",
    "LOINC_Code": "loinc_code",
    "Test_Name": "test_name",
    "Result": "result_value",
    "Unit": "unit",
    "Result_Status": "result_status"
})

# Convert numerical values
df_results_sql["observation_number"] = pd.to_numeric(
    df_results_sql["observation_number"],
    errors="coerce"
)

df_results_sql["result_value"] = pd.to_numeric(
    df_results_sql["result_value"],
    errors="coerce"
)

# Check required fields
missing_values = int(
    df_results_sql.isna().sum().sum()
)

# Check blank text values
text_columns = [
    "order_id",
    "message_id",
    "loinc_code",
    "test_name",
    "unit",
    "result_status"
]

blank_values = int(
    df_results_sql[text_columns]
    .apply(lambda col: col.str.strip().eq(""))
    .sum().sum()
)

# Check duplicate observation identifiers
duplicate_observations = int(
    df_results_sql.duplicated(
        subset=["order_id", "observation_number"]
    ).sum()
)

# Check duplicate order-test combinations
duplicate_tests = int(
    df_results_sql.duplicated(
        subset=["order_id", "loinc_code"]
    ).sum()
)

# Check numerical values
invalid_results = int(
    (~np.isfinite(df_results_sql["result_value"])).sum()
)

invalid_observation_numbers = int(
    (
        df_results_sql["observation_number"].lt(1)
        | df_results_sql["observation_number"].mod(1).ne(0)
    ).sum()
)

# Verify order and message relationships
order_reference = df_orders_sql[
    ["order_id", "message_id"]
].rename(columns={
    "message_id": "expected_message_id"
})

result_comparison = df_results_sql.merge(
    order_reference,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

unknown_orders = int(
    result_comparison["_merge"].ne("both").sum()
)

message_mismatches = int(
    (
        result_comparison["message_id"]
        != result_comparison["expected_message_id"]
    ).sum()
)

# Verify LOINC, test name, and unit mappings
test_reference = df_tests_sql.rename(columns={
    "test_name": "expected_test_name",
    "unit": "expected_unit"
})

test_comparison = df_results_sql.merge(
    test_reference,
    on="loinc_code",
    how="left",
    validate="many_to_one",
    indicator=True
)

unknown_loinc = int(
    test_comparison["_merge"].ne("both").sum()
)

test_name_mismatches = int(
    (
        test_comparison["test_name"]
        != test_comparison["expected_test_name"]
    ).sum()
)

unit_mismatches = int(
    (
        test_comparison["unit"]
        != test_comparison["expected_unit"]
    ).sum()
)

# Verify PostgreSQL field lengths
field_limits = {
    "order_id": 30,
    "message_id": 30,
    "loinc_code": 20,
    "unit": 30,
    "result_status": 10
}

length_issues = sum(
    int(df_results_sql[column].str.len().gt(limit).sum())
    for column, limit in field_limits.items()
)

# Verify expected record count
expected_results = 20612

result_preparation_passed = (
    len(df_results_sql) == expected_results
    and missing_values == 0
    and blank_values == 0
    and duplicate_observations == 0
    and duplicate_tests == 0
    and invalid_results == 0
    and invalid_observation_numbers == 0
    and unknown_orders == 0
    and message_mismatches == 0
    and unknown_loinc == 0
    and test_name_mismatches == 0
    and unit_mismatches == 0
    and length_issues == 0
)

# Display verification results
print("\nLaboratory Result Verification")
print("------------------------------")
print("Total Laboratory Results:", len(df_results_sql))
print("Duplicate Observations:", duplicate_observations)
print("Duplicate Order-Test Combinations:", duplicate_tests)
print("Missing Values:", missing_values)
print("Blank Text Values:", blank_values)
print("Invalid Numerical Results:", invalid_results)
print("Invalid Observation Numbers:", invalid_observation_numbers)
print("Unknown Laboratory Orders:", unknown_orders)
print("Order-Message Mismatches:", message_mismatches)
print("Unknown LOINC Codes:", unknown_loinc)
print("Test Name Mismatches:", test_name_mismatches)
print("Measurement Unit Mismatches:", unit_mismatches)
print("Field Length Issues:", length_issues)

print(
    "\nResult Preparation Status:",
    "PASSED" if result_preparation_passed
    else "REVIEW REQUIRED"
)

print("\nFirst Five Prepared Laboratory Results:")
display(df_results_sql.head())


Step 8I-6: Laboratory Result Preparation
---------------------------------------

Laboratory Result Verification
------------------------------
Total Laboratory Results: 20612
Duplicate Observations: 0
Duplicate Order-Test Combinations: 0
Missing Values: 0
Blank Text Values: 0
Invalid Numerical Results: 0
Invalid Observation Numbers: 0
Unknown Laboratory Orders: 0
Order-Message Mismatches: 0
Unknown LOINC Codes: 0
Test Name Mismatches: 0
Measurement Unit Mismatches: 0
Field Length Issues: 0

Result Preparation Status: PASSED

First Five Prepared Laboratory Results:


,order_id,message_id,observation_number,loinc_code,test_name,result_value,unit,result_status
0,ORD000001,MSG000001,1,2951-2,Sodium,144.8,mmol/L,F
1,ORD000001,MSG000001,2,2823-3,Potassium,3.7,mmol/L,F
2,ORD000001,MSG000001,3,2075-0,Chloride,97.8,mmol/L,F
3,ORD000001,MSG000001,4,2028-9,CO2,19.7,mmol/L,F
4,ORD000001,MSG000001,5,2345-7,Glucose,163.7,mg/dL,F



#### Step 8I-7: PostgreSQL Data Import

Following successful data preparation and validation, the five structured datasets are imported into the PostgreSQL relational database.

The import follows the established primary and foreign key relationships, beginning with patient demographics and laboratory test definitions, followed by HL7 message metadata, laboratory orders, and individual laboratory observations.

Python and psycopg are used to execute parameterized SQL INSERT statements. The import is performed within a database transaction to maintain data integrity and prevent partial insertion if an error occurs.

After insertion, SQL queries verify the number of records stored in each table against the expected dataset counts. The completed import establishes the relational foundation for subsequent SQL analysis, interoperability monitoring, laboratory turnaround time evaluation, and Power BI reporting.


In [46]:

# Step 8I-7: PostgreSQL Data Import

import psycopg
import pandas as pd
from decimal import Decimal
from getpass import getpass

print("Step 8I-7: PostgreSQL Data Import")
print("--------------------------------")

# Verify all preparation steps passed
preparation_checks = {
    "Patients": patient_preparation_passed,
    "Laboratory Tests": test_preparation_passed,
    "HL7 Messages": message_preparation_passed,
    "Laboratory Orders": order_preparation_passed,
    "Laboratory Results": result_preparation_passed
}

if not all(preparation_checks.values()):
    raise RuntimeError(
        "Data preparation checks must pass before import."
    )

# Define expected PostgreSQL table counts
expected_counts = {
    "patients": 1000,
    "laboratory_tests": 23,
    "hl7_messages": 2000,
    "laboratory_orders": 2000,
    "laboratory_results": 20612
}

# Prepare datasets in foreign-key dependency order
import_plan = [
    (
        "patients",
        df_patients_sql,
        ["patient_id", "last_name", "first_name",
         "date_of_birth", "sex"]
    ),
    (
        "laboratory_tests",
        df_tests_sql,
        ["loinc_code", "test_name", "unit"]
    ),
    (
        "hl7_messages",
        df_messages_sql,
        ["message_id", "order_id", "patient_id",
         "panel", "result_count", "obx_count",
         "validation_status", "parsing_status",
         "validation_error"]
    ),
    (
        "laboratory_orders",
        df_orders_sql,
        ["order_id", "patient_id", "message_id",
         "panel_loinc", "panel_name",
         "collection_time", "result_time",
         "order_status"]
    ),
    (
        "laboratory_results",
        df_results_sql,
        ["order_id", "message_id",
         "observation_number", "loinc_code",
         "result_value", "unit", "result_status"]
    )
]

# Ensure numerical values are compatible with PostgreSQL
df_messages_sql["result_count"] = (
    df_messages_sql["result_count"].astype(int)
)
df_messages_sql["obx_count"] = (
    df_messages_sql["obx_count"].astype(int)
)
df_results_sql["observation_number"] = (
    df_results_sql["observation_number"].astype(int)
)

# Convert pandas values to database-compatible Python values
def prepare_records(dataframe, columns):
    records = []

    for row in dataframe[columns].itertuples(
        index=False, name=None
    ):
        converted = []

        for value in row:
            if pd.isna(value):
                converted.append(None)
            elif isinstance(value, pd.Timestamp):
                converted.append(value.to_pydatetime())
            elif hasattr(value, "item"):
                converted.append(value.item())
            else:
                converted.append(value)

        records.append(tuple(converted))

    return records

# Convert laboratory measurements to Decimal
df_results_import = df_results_sql.copy()
df_results_import["result_value"] = (
    df_results_import["result_value"]
    .map(lambda value: Decimal(str(value)))
)

# Replace result dataset with Decimal-compatible copy
import_plan[-1] = (
    "laboratory_results",
    df_results_import,
    import_plan[-1][2]
)

# Request password securely
db_password = getpass("Enter PostgreSQL password: ")

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Check existing records before import
            for table, expected in expected_counts.items():
                cursor.execute(
                    f"SELECT COUNT(*) FROM {table}"
                )

                existing_count = cursor.fetchone()[0]

                if existing_count != 0:
                    raise RuntimeError(
                        f"{table} already contains "
                        f"{existing_count} records. "
                        "Import stopped to prevent duplicates."
                    )

            print("\nImporting Prepared Datasets")
            print("---------------------------")

            # Insert each dataset
            for table, dataframe, columns in import_plan:

                column_names = ", ".join(columns)
                placeholders = ", ".join(
                    ["%s"] * len(columns)
                )

                insert_sql = (
                    f"INSERT INTO {table} "
                    f"({column_names}) "
                    f"VALUES ({placeholders})"
                )

                records = prepare_records(
                    dataframe, columns
                )

                cursor.executemany(
                    insert_sql,
                    records
                )

                print(
                    f"{table}: {len(records):,} "
                    "records inserted"
                )

            # Verify counts before committing
            verification_records = []

            for table, expected in expected_counts.items():

                cursor.execute(
                    f"SELECT COUNT(*) FROM {table}"
                )

                actual = cursor.fetchone()[0]

                verification_records.append({
                    "Table": table,
                    "Expected": expected,
                    "Actual": actual,
                    "Status": (
                        "PASSED" if actual == expected
                        else "FAILED"
                    )
                })

            df_sql_import_verification = pd.DataFrame(
                verification_records
            )

            import_passed = (
                df_sql_import_verification["Status"]
                .eq("PASSED").all()
            )

            if not import_passed:
                raise RuntimeError(
                    "PostgreSQL record count verification failed."
                )

        # Commit only after all checks succeed
        connection.commit()

    print("\nPostgreSQL Import Verification:")
    display(df_sql_import_verification)

    print(
        "\nTotal Records Imported:",
        df_sql_import_verification["Actual"].sum()
    )

    print("\nOverall Import Status: PASSED")

except (psycopg.Error, RuntimeError) as error:
    print("\nOverall Import Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 8I-7: PostgreSQL Data Import
--------------------------------


Enter PostgreSQL password:  ········



Importing Prepared Datasets
---------------------------
patients: 1,000 records inserted
laboratory_tests: 23 records inserted
hl7_messages: 2,000 records inserted
laboratory_orders: 2,000 records inserted
laboratory_results: 20,612 records inserted

PostgreSQL Import Verification:


,Table,Expected,Actual,Status
0,patients,1000,1000,PASSED
1,laboratory_tests,23,23,PASSED
2,hl7_messages,2000,2000,PASSED
3,laboratory_orders,2000,2000,PASSED
4,laboratory_results,20612,20612,PASSED



Total Records Imported: 25635

Overall Import Status: PASSED



### Step 8J: PostgreSQL Data Integrity Verification

Following the successful import of 25,635 records, SQL queries are executed against the PostgreSQL database to verify record counts and relational data integrity.

The verification examines the relationships between patients, HL7 messages, laboratory orders, laboratory test definitions, and individual laboratory results. Checks are performed to identify orphaned records, mismatched identifiers, duplicate observations, and inconsistencies between HL7 message counts and stored laboratory results.

These database-level checks provide additional assurance that the information extracted from the synthetic HL7 messages was preserved during relational database integration.

The results establish a verified foundation for subsequent SQL analysis, laboratory turnaround time evaluation, and Power BI visualization.


In [47]:

# Step 8J: PostgreSQL Data Integrity Verification

import psycopg
import pandas as pd
from getpass import getpass

print("Step 8J: PostgreSQL Data Integrity Verification")
print("-----------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

# Define database integrity checks
integrity_queries = {
    "Orders Without Valid Patients": """
        SELECT COUNT(*)
        FROM laboratory_orders o
        LEFT JOIN patients p
            ON o.patient_id = p.patient_id
        WHERE p.patient_id IS NULL
    """,

    "Messages Without Valid Patients": """
        SELECT COUNT(*)
        FROM hl7_messages m
        LEFT JOIN patients p
            ON m.patient_id = p.patient_id
        WHERE p.patient_id IS NULL
    """,

    "Orders Without Valid Messages": """
        SELECT COUNT(*)
        FROM laboratory_orders o
        LEFT JOIN hl7_messages m
            ON o.message_id = m.message_id
        WHERE m.message_id IS NULL
    """,

    "Order-Message Patient Mismatches": """
        SELECT COUNT(*)
        FROM laboratory_orders o
        JOIN hl7_messages m
            ON o.message_id = m.message_id
        WHERE o.patient_id <> m.patient_id
           OR o.order_id <> m.order_id
    """,

    "Results Without Valid Orders": """
        SELECT COUNT(*)
        FROM laboratory_results r
        LEFT JOIN laboratory_orders o
            ON r.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "Result-Message Mismatches": """
        SELECT COUNT(*)
        FROM laboratory_results r
        JOIN laboratory_orders o
            ON r.order_id = o.order_id
        WHERE r.message_id <> o.message_id
    """,

    "Results Without Valid LOINC Codes": """
        SELECT COUNT(*)
        FROM laboratory_results r
        LEFT JOIN laboratory_tests t
            ON r.loinc_code = t.loinc_code
        WHERE t.loinc_code IS NULL
    """,

    "Duplicate Order Observations": """
        SELECT COUNT(*)
        FROM (
            SELECT order_id, observation_number
            FROM laboratory_results
            GROUP BY order_id, observation_number
            HAVING COUNT(*) > 1
        ) duplicates
    """,

    "Duplicate Order-Test Combinations": """
        SELECT COUNT(*)
        FROM (
            SELECT order_id, loinc_code
            FROM laboratory_results
            GROUP BY order_id, loinc_code
            HAVING COUNT(*) > 1
        ) duplicates
    """,

    "HL7 Observation Count Mismatches": """
        SELECT COUNT(*)
        FROM hl7_messages m
        LEFT JOIN (
            SELECT message_id, COUNT(*) AS actual_results
            FROM laboratory_results
            GROUP BY message_id
        ) r
            ON m.message_id = r.message_id
        WHERE m.obx_count <> COALESCE(r.actual_results, 0)
           OR m.result_count <> COALESCE(r.actual_results, 0)
    """,

    "Invalid Laboratory Timestamps": """
        SELECT COUNT(*)
        FROM laboratory_orders
        WHERE result_time <= collection_time
    """
}

try:
    verification_records = []

    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            for check_name, query in integrity_queries.items():

                cursor.execute(query)
                issues = cursor.fetchone()[0]

                verification_records.append({
                    "Validation_Check": check_name,
                    "Issues_Found": issues,
                    "Status": (
                        "PASSED" if issues == 0
                        else "REVIEW REQUIRED"
                    )
                })

    df_sql_integrity = pd.DataFrame(
        verification_records
    )

    print("\nDatabase Integrity Results:")
    display(df_sql_integrity)

    total_issues = int(
        df_sql_integrity["Issues_Found"].sum()
    )

    integrity_passed = total_issues == 0

    print("\nTotal Integrity Checks:",
          len(df_sql_integrity))
    print("Total Issues Identified:", total_issues)

    print(
        "\nDatabase Integrity Status:",
        "PASSED" if integrity_passed
        else "REVIEW REQUIRED"
    )

except psycopg.Error as error:
    print("\nDatabase Integrity Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 8J: PostgreSQL Data Integrity Verification
-----------------------------------------------


Enter PostgreSQL password:  ········



Database Integrity Results:


,Validation_Check,Issues_Found,Status
0,Orders Without Valid Patients,0,PASSED
1,Messages Without Valid Patients,0,PASSED
2,Orders Without Valid Messages,0,PASSED
3,Order-Message Patient Mismatches,0,PASSED
4,Results Without Valid Orders,0,PASSED
5,Result-Message Mismatches,0,PASSED
6,Results Without Valid LOINC Codes,0,PASSED
7,Duplicate Order Observations,0,PASSED
8,Duplicate Order-Test Combinations,0,PASSED
9,HL7 Observation Count Mismatches,0,PASSED



Total Integrity Checks: 11
Total Issues Identified: 0

Database Integrity Status: PASSED



### Step 8K: PostgreSQL Database Inspection

Following successful database import and integrity verification, the PostgreSQL database is inspected using pgAdmin 4.

The inspection confirms that the five relational tables are accessible through PostgreSQL and contain the expected patient, laboratory test, HL7 message, laboratory order, and laboratory result records.

The database structure and sample records are reviewed to confirm successful integration. This provides an additional verification of the relational database before proceeding with SQL analysis and Power BI reporting.



## Step 9: SQL Laboratory Analytics

This phase uses PostgreSQL to analyze the synthetic laboratory data generated and transmitted through HL7 v2.5.1 messages.

SQL queries are executed through Python in Jupyter Notebook to examine laboratory testing volumes, turnaround times, test utilization, HL7 message processing, and patient-level laboratory activity.

The analysis demonstrates how structured data extracted from HL7 messages can support laboratory operations, data quality monitoring, and healthcare business intelligence.

The resulting SQL queries and analytical datasets will also provide the foundation for Power BI dashboards.

### Step 9A: Laboratory Testing Volume by Panel

This analysis examines laboratory testing activity across Complete Blood Count (CBC), Basic Metabolic Panel (BMP), and Comprehensive Metabolic Panel (CMP) orders.

PostgreSQL queries are used to calculate the number of laboratory orders, individual test results, percentage of total testing volume, and average number of results per order for each panel.

These measures help demonstrate how laboratory order volume and individual test volume differ, particularly when comparing panels containing different numbers of tests.

All results are based on synthetic laboratory data and do not represent actual hospital testing volumes.


In [48]:

# Step 9A: Laboratory Testing Volume by Panel

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9A: Laboratory Testing Volume by Panel")
print("------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

sql_query = """
WITH panel_summary AS (
    SELECT
        o.panel_name,
        o.panel_loinc,
        COUNT(DISTINCT o.order_id) AS total_orders,
        COUNT(r.result_id) AS total_results
    FROM laboratory_orders o
    LEFT JOIN laboratory_results r
        ON o.order_id = r.order_id
    GROUP BY
        o.panel_name,
        o.panel_loinc
)
SELECT
    panel_name,
    panel_loinc,
    total_orders,
    total_results,

    ROUND(
        100.0 * total_orders
        / NULLIF(SUM(total_orders) OVER (), 0),
        2
    ) AS order_percentage,

    ROUND(
        100.0 * total_results
        / NULLIF(SUM(total_results) OVER (), 0),
        2
    ) AS result_percentage,

    ROUND(
        total_results::NUMERIC
        / NULLIF(total_orders, 0),
        2
    ) AS avg_results_per_order

FROM panel_summary
ORDER BY total_orders DESC;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:
            cursor.execute(sql_query)

            records = cursor.fetchall()

            columns = [
                description.name
                for description in cursor.description
            ]

    df_panel_volume = pd.DataFrame(
        records,
        columns=columns
    )

    print("\nLaboratory Testing Volume Summary:")
    display(df_panel_volume)

    total_orders = int(
        df_panel_volume["total_orders"].sum()
    )

    total_results = int(
        df_panel_volume["total_results"].sum()
    )

    print("\nTotal Laboratory Orders:", total_orders)
    print("Total Laboratory Results:", total_results)

    volume_passed = (
        total_orders == 2000
        and total_results == 20612
        and len(df_panel_volume) == 3
    )

    print(
        "\nPanel Volume Verification:",
        "PASSED" if volume_passed
        else "REVIEW REQUIRED"
    )

except (psycopg.Error, KeyError, ValueError) as error:
    print("\nPanel Volume Verification: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9A: Laboratory Testing Volume by Panel
------------------------------------------


Enter PostgreSQL password:  ········



Laboratory Testing Volume Summary:


,panel_name,panel_loinc,total_orders,total_results,order_percentage,result_percentage,avg_results_per_order
0,Basic metabolic panel,24321-2,673,5384,33.65,26.12,8.00
1,CBC panel,58410-2,670,6030,33.50,29.25,9.00
2,Comprehensive metabolic panel,24323-8,657,9198,32.85,44.62,14.00



Total Laboratory Orders: 2000
Total Laboratory Results: 20612

Panel Volume Verification: PASSED



#### Step 9A: Results and Interpretation

The SQL analysis identified 2,000 laboratory orders containing 20,612 individual test results across three laboratory panels. Basic metabolic panels (BMP) accounted for the largest number of orders at 673 (33.65%), followed by complete blood counts (CBC) at 670 (33.50%) and comprehensive metabolic panels (CMP) at 657 (32.85%).

Although order volumes were distributed almost evenly across the three panels, the number of individual results varied considerably. CMP generated 9,198 results, representing 44.62% of all observations, despite having the fewest orders. This difference reflects the 14 individual tests included in each CMP, compared with nine for CBC and eight for BMP.

These findings demonstrate why laboratory order volume and individual test volume should be evaluated separately. A laboratory may receive a similar number of orders for different panels, but the number of tests associated with those orders can vary significantly. Understanding these differences can support testing utilization analysis, workload planning, and future cost evaluations when additional operational data is available.

All findings represent synthetic laboratory activity and are intended to demonstrate SQL-based healthcare analytics.



### Step 9B: Laboratory Turnaround Time Analysis

Laboratory turnaround time (TAT) is evaluated using specimen collection and result timestamps stored in PostgreSQL.

SQL queries calculate the mean, median, minimum, maximum, and standard deviation of turnaround times for CBC, BMP, and CMP laboratory panels. Turnaround time is measured in minutes from specimen collection to the recorded laboratory result time.

The analysis uses individual laboratory orders rather than individual test observations to avoid duplicating turnaround time measurements for panels containing multiple tests.

Comparing turnaround times across panels provides insight into simulated laboratory workflow performance and establishes metrics for future Power BI reporting.

Because the dataset is synthetic, the findings demonstrate analytical methods rather than actual clinical laboratory performance.


In [49]:

# Step 9B: Laboratory Turnaround Time Analysis

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9B: Laboratory Turnaround Time Analysis")
print("-------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

sql_query = """
WITH turnaround_data AS (
    SELECT
        order_id,
        panel_name,
        EXTRACT(
            EPOCH FROM (result_time - collection_time)
        ) / 60.0 AS turnaround_minutes
    FROM laboratory_orders
)
SELECT
    panel_name,
    COUNT(*) AS total_orders,

    ROUND(
        AVG(turnaround_minutes)::NUMERIC, 2
    ) AS mean_tat_minutes,

    ROUND(
        PERCENTILE_CONT(0.5)
        WITHIN GROUP (
            ORDER BY turnaround_minutes
        )::NUMERIC, 2
    ) AS median_tat_minutes,

    ROUND(
        MIN(turnaround_minutes)::NUMERIC, 2
    ) AS min_tat_minutes,

    ROUND(
        MAX(turnaround_minutes)::NUMERIC, 2
    ) AS max_tat_minutes,

    ROUND(
        STDDEV_SAMP(turnaround_minutes)::NUMERIC, 2
    ) AS std_tat_minutes,

    ROUND(
        PERCENTILE_CONT(0.25)
        WITHIN GROUP (
            ORDER BY turnaround_minutes
        )::NUMERIC, 2
    ) AS q1_tat_minutes,

    ROUND(
        PERCENTILE_CONT(0.75)
        WITHIN GROUP (
            ORDER BY turnaround_minutes
        )::NUMERIC, 2
    ) AS q3_tat_minutes

FROM turnaround_data
GROUP BY panel_name
ORDER BY mean_tat_minutes DESC;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:
            cursor.execute(sql_query)

            records = cursor.fetchall()

            columns = [
                description.name
                for description in cursor.description
            ]

    df_tat_summary = pd.DataFrame(
        records,
        columns=columns
    )

    print("\nLaboratory Turnaround Time Summary:")
    display(df_tat_summary)

    total_orders = int(
        df_tat_summary["total_orders"].sum()
    )

    invalid_tat = int(
        (
            df_tat_summary["min_tat_minutes"] <= 0
        ).sum()
    )

    tat_passed = (
        total_orders == 2000
        and len(df_tat_summary) == 3
        and invalid_tat == 0
    )

    print("\nTotal Orders Analyzed:", total_orders)
    print("Panels Analyzed:", len(df_tat_summary))
    print("Panels With Invalid TAT:", invalid_tat)

    print(
        "\nTurnaround Time Verification:",
        "PASSED" if tat_passed
        else "REVIEW REQUIRED"
    )

except (psycopg.Error, KeyError, ValueError) as error:
    print("\nTurnaround Time Verification: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9B: Laboratory Turnaround Time Analysis
-------------------------------------------


Enter PostgreSQL password:  ········



Laboratory Turnaround Time Summary:


,panel_name,total_orders,mean_tat_minutes,median_tat_minutes,min_tat_minutes,max_tat_minutes,std_tat_minutes,q1_tat_minutes,q3_tat_minutes
0,Basic metabolic panel,673,106.90,107.00,30.00,180.00,45.32,66.00,148.00
1,CBC panel,670,106.21,107.00,30.00,180.00,44.39,67.00,144.00
2,Comprehensive metabolic panel,657,105.89,107.00,30.00,180.00,44.63,67.00,146.00



Total Orders Analyzed: 2000
Panels Analyzed: 3
Panels With Invalid TAT: 0

Turnaround Time Verification: PASSED



#### Step 9B: Results and Interpretation

The turnaround time analysis evaluated 2,000 laboratory orders across BMP, CBC, and CMP panels. Mean turnaround times were similar across all three groups, ranging from 105.89 minutes for CMP to 106.90 minutes for BMP. CBC had an average turnaround time of 106.21 minutes, while all three panels shared a median of 107 minutes.

Despite the similar averages, turnaround times varied considerably within each panel. The minimum turnaround time was 30 minutes, and the maximum was 180 minutes across all three groups. Standard deviations ranged from 44.39 to 45.32 minutes, indicating substantial variation around the average. The interquartile ranges also showed that the middle 50% of orders covered a relatively wide range of turnaround times.

One interesting finding is that CMP, which contains the largest number of individual tests, had the lowest mean turnaround time. However, the difference between panels was only about one minute, and the synthetic data does not establish a relationship between panel complexity and processing time.

These results demonstrate why laboratory turnaround time should be evaluated using multiple statistical measures rather than averages alone. In an actual clinical laboratory, examining variation and unusually long turnaround times could help identify workflow delays, evaluate performance targets, and support quality improvement initiatives.

The turnaround times represent specimen collection to recorded result time and do not distinguish between transportation, accessioning, analytical processing, or result verification.



### Step 9C: Laboratory Test Utilization Analysis

Laboratory test utilization is examined using SQL queries against the PostgreSQL laboratory_results and laboratory_tests tables.

The analysis calculates the number of observations associated with each standardized LOINC test definition, the percentage of total laboratory results represented by each test, and the number of distinct laboratory orders containing that test.

These measures help identify frequently performed laboratory tests and demonstrate how individual analytes contribute to overall testing activity across CBC, BMP, and CMP panels.

The analysis also verifies that all 23 laboratory test definitions are represented in the dataset and that individual observation counts reconcile with the 20,612 laboratory results stored in PostgreSQL.

The findings are based on synthetic data and demonstrate laboratory utilization analysis rather than actual clinical testing demand.



### Step 9C: Laboratory Test Utilization Analysis

Laboratory test utilization is examined using SQL queries against the PostgreSQL laboratory_results and laboratory_tests tables.

The analysis calculates the number of observations associated with each standardized LOINC test definition, the percentage of total laboratory results represented by each test, and the number of distinct laboratory orders containing that test.

These measures help identify frequently performed laboratory tests and demonstrate how individual analytes contribute to overall testing activity across CBC, BMP, and CMP panels.

The analysis also verifies that all 23 laboratory test definitions are represented in the dataset and that individual observation counts reconcile with the 20,612 laboratory results stored in PostgreSQL.

The findings are based on synthetic data and demonstrate laboratory utilization analysis rather than actual clinical testing demand.


In [50]:

# Step 9C: Laboratory Test Utilization Analysis

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9C: Laboratory Test Utilization Analysis")
print("--------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

sql_query = """
SELECT
    t.test_name,
    t.loinc_code,
    t.unit,

    COUNT(r.result_id) AS total_observations,

    COUNT(DISTINCT r.order_id) AS total_orders,

    ROUND(
        100.0 * COUNT(r.result_id)
        / NULLIF(
            SUM(COUNT(r.result_id)) OVER (), 0
        ),
        2
    ) AS result_percentage

FROM laboratory_tests t

LEFT JOIN laboratory_results r
    ON t.loinc_code = r.loinc_code

GROUP BY
    t.test_name,
    t.loinc_code,
    t.unit

ORDER BY
    total_observations DESC,
    t.test_name;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:
            cursor.execute(sql_query)

            records = cursor.fetchall()

            columns = [
                description.name
                for description in cursor.description
            ]

    df_test_utilization = pd.DataFrame(
        records,
        columns=columns
    )

    print("\nLaboratory Test Utilization Summary:")
    display(df_test_utilization)

    total_tests = len(df_test_utilization)

    total_observations = int(
        df_test_utilization["total_observations"].sum()
    )

    unused_tests = int(
        df_test_utilization["total_observations"]
        .eq(0).sum()
    )

    utilization_passed = (
        total_tests == 23
        and total_observations == 20612
        and unused_tests == 0
    )

    print("\nTotal Test Definitions:", total_tests)
    print("Total Observations:", total_observations)
    print("Unused Test Definitions:", unused_tests)

    print(
        "\nTest Utilization Verification:",
        "PASSED" if utilization_passed
        else "REVIEW REQUIRED"
    )

except (psycopg.Error, KeyError, ValueError) as error:
    print("\nTest Utilization Verification: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9C: Laboratory Test Utilization Analysis
--------------------------------------------


Enter PostgreSQL password:  ········



Laboratory Test Utilization Summary:


,test_name,loinc_code,unit,total_observations,total_orders,result_percentage
0,BUN,3094-0,mg/dL,1330,1330,6.45
1,Calcium,17861-6,mg/dL,1330,1330,6.45
2,Chloride,2075-0,mmol/L,1330,1330,6.45
3,CO2,2028-9,mmol/L,1330,1330,6.45
4,Creatinine,2160-0,mg/dL,1330,1330,6.45
5,Glucose,2345-7,mg/dL,1330,1330,6.45
6,Potassium,2823-3,mmol/L,1330,1330,6.45
7,Sodium,2951-2,mmol/L,1330,1330,6.45
8,Hematocrit,4544-3,%,670,670,3.25
9,Hemoglobin,718-7,g/dL,670,670,3.25



Total Test Definitions: 23
Total Observations: 20612
Unused Test Definitions: 0

Test Utilization Verification: PASSED



#### Step 9C: Results and Interpretation

The SQL utilization analysis confirmed that all 23 laboratory test definitions were represented in the PostgreSQL database, accounting for 20,612 individual observations. No unused test definitions were identified, and the total observation count matched the previously validated laboratory dataset.

The most frequently recorded tests were BUN, calcium, chloride, CO2, creatinine, glucose, potassium, and sodium, each appearing in 1,330 laboratory orders and representing 6.45% of all results individually. These tests are included in both BMP and CMP panels, which explains their higher utilization compared with tests associated with only one panel.

The nine CBC tests each appeared 670 times, accounting for 3.25% of individual results per test. The six tests unique to CMP, including albumin, ALP, ALT, AST, total bilirubin, and total protein, each appeared 657 times, representing 3.19% of results individually.

These findings demonstrate how laboratory test utilization is influenced by panel composition and overlapping analytes. Although BMP, CBC, and CMP order volumes were relatively similar, tests included in multiple panels appeared more frequently in the overall dataset.

From a laboratory operations perspective, analyzing individual test utilization alongside panel volumes can provide a clearer understanding of testing activity. This information could support reagent utilization analysis, inventory planning, and workload evaluation when combined with actual laboratory resource and cost data.

Because the dataset is synthetic, the findings illustrate SQL-based utilization analysis rather than actual clinical testing patterns.



### Step 9D: HL7 Message Processing and Validation Analysis

This analysis evaluates the processing and validation results of 2,000 synthetic HL7 v2.5.1 laboratory messages transmitted through the simulated LIS-to-EHR integration workflow.

PostgreSQL queries are used to examine message validation and parsing statuses, compare expected laboratory results with OBX observation counts, and evaluate message processing outcomes across CBC, BMP, and CMP panels.

The analysis also calculates overall validation and parsing success rates and identifies any messages with inconsistent observation counts or unsuccessful processing statuses.

These measures demonstrate how SQL can support healthcare interoperability monitoring by identifying potential message processing errors, incomplete laboratory results, and data exchange inconsistencies.

All messages were generated and processed within a simulated environment. The findings demonstrate interoperability data quality monitoring rather than actual hospital interface performance.


In [51]:

# Step 9D: HL7 Message Processing and Validation Analysis

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9D: HL7 Message Processing and Validation Analysis")
print("------------------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

sql_query = """
SELECT
    panel,
    COUNT(*) AS total_messages,

    COUNT(*) FILTER (
        WHERE validation_status = 'PASSED'
    ) AS validated_messages,

    COUNT(*) FILTER (
        WHERE parsing_status = 'PASSED'
    ) AS parsed_messages,

    COUNT(*) FILTER (
        WHERE result_count = obx_count
    ) AS matching_obx_counts,

    COUNT(*) FILTER (
        WHERE result_count <> obx_count
    ) AS mismatched_obx_counts,

    SUM(result_count) AS expected_results,

    SUM(obx_count) AS total_obx_segments,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE validation_status = 'PASSED'
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS validation_success_pct,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE parsing_status = 'PASSED'
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS parsing_success_pct

FROM hl7_messages
GROUP BY panel
ORDER BY total_messages DESC;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:
            cursor.execute(sql_query)

            records = cursor.fetchall()

            columns = [
                description.name
                for description in cursor.description
            ]

    df_hl7_performance = pd.DataFrame(
        records,
        columns=columns
    )

    print("\nHL7 Message Processing Summary:")
    display(df_hl7_performance)

    total_messages = int(
        df_hl7_performance["total_messages"].sum()
    )

    validated_messages = int(
        df_hl7_performance["validated_messages"].sum()
    )

    parsed_messages = int(
        df_hl7_performance["parsed_messages"].sum()
    )

    mismatched_counts = int(
        df_hl7_performance["mismatched_obx_counts"].sum()
    )

    expected_results = int(
        df_hl7_performance["expected_results"].sum()
    )

    total_obx = int(
        df_hl7_performance["total_obx_segments"].sum()
    )

    hl7_performance_passed = (
        total_messages == 2000
        and validated_messages == 2000
        and parsed_messages == 2000
        and mismatched_counts == 0
        and expected_results == 20612
        and total_obx == 20612
        and len(df_hl7_performance) == 3
    )

    print("\nTotal HL7 Messages:", total_messages)
    print("Messages Passed Validation:", validated_messages)
    print("Messages Parsed Successfully:", parsed_messages)
    print("Messages With OBX Count Mismatches:", mismatched_counts)
    print("Expected Laboratory Results:", expected_results)
    print("Total OBX Segments:", total_obx)

    print(
        "\nHL7 Processing Verification:",
        "PASSED" if hl7_performance_passed
        else "REVIEW REQUIRED"
    )

except (psycopg.Error, KeyError, ValueError) as error:
    print("\nHL7 Processing Verification: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9D: HL7 Message Processing and Validation Analysis
------------------------------------------------------


Enter PostgreSQL password:  ········



HL7 Message Processing Summary:


,panel,total_messages,validated_messages,parsed_messages,matching_obx_counts,mismatched_obx_counts,expected_results,total_obx_segments,validation_success_pct,parsing_success_pct
0,BMP,673,673,673,673,0,5384,5384,100.00,100.00
1,CBC,670,670,670,670,0,6030,6030,100.00,100.00
2,CMP,657,657,657,657,0,9198,9198,100.00,100.00



Total HL7 Messages: 2000
Messages Passed Validation: 2000
Messages Parsed Successfully: 2000
Messages With OBX Count Mismatches: 0
Expected Laboratory Results: 20612
Total OBX Segments: 20612

HL7 Processing Verification: PASSED



#### Step 9D: Results and Interpretation

The SQL analysis confirmed that all 2,000 HL7 messages were successfully validated and parsed, resulting in a 100% success rate across BMP, CBC, and CMP panels. No message processing failures or observation count mismatches were identified.

BMP accounted for 673 messages containing 5,384 OBX segments, while CBC accounted for 670 messages containing 6,030 segments. CMP generated 657 messages with 9,198 OBX segments. Although the number of messages was relatively similar across the three panels, CMP contained the largest number of individual observations because each panel included 14 laboratory tests.

The analysis also confirmed that the expected laboratory result counts matched the OBX segment counts for every panel. All 20,612 observations were accounted for, supporting the earlier HL7 parsing and PostgreSQL data integrity verification results.

From a healthcare interoperability perspective, these findings demonstrate the importance of monitoring both message processing success and the completeness of the information contained within each message. A message may be received and processed, but verifying that all expected laboratory observations are present provides an additional measure of data quality.

In an actual hospital environment, monitoring these measures could help identify incomplete transmissions, interface processing errors, or discrepancies between laboratory orders and reported results. These issues could affect the availability and accuracy of laboratory information within the electronic health record.

Because this project uses synthetic HL7 messages processed in a controlled environment, the 100% success rate reflects the simulated workflow rather than actual LIS-to-EHR interface performance.



### Step 9E: Patient-Level Laboratory Utilization Analysis

This analysis examines laboratory testing activity across the 1,000 synthetic patients represented in the PostgreSQL database.

SQL queries are used to evaluate the number of laboratory orders and individual test results associated with each patient. The analysis also examines the distribution of laboratory panel combinations, including CBC, BMP, and CMP, to determine how different testing patterns influence the number of results generated per patient.

Patient-level utilization is evaluated by joining the patients, laboratory_orders, and laboratory_results tables using their established relational identifiers. The analysis verifies that each patient has exactly two laboratory orders and that all laboratory observations are correctly associated with the corresponding patient.

These measures demonstrate how relational healthcare data can be used to examine laboratory utilization, identify testing patterns, and support operational reporting.

Because the dataset is synthetic and each patient was assigned two orders during data generation, the findings illustrate SQL-based utilization analysis rather than actual clinical testing demand.


In [52]:

# Step 9E: Patient-Level Laboratory Utilization Analysis

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9E: Patient-Level Laboratory Utilization Analysis")
print("-----------------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

# Query 1: Patient-level laboratory utilization
sql_patient_query = """
SELECT
    p.patient_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(r.result_id) AS total_results,
    COUNT(DISTINCT o.panel_loinc) AS unique_panel_types,

    STRING_AGG(
        DISTINCT m.panel,
        ', ' ORDER BY m.panel
    ) AS panel_types

FROM patients p

LEFT JOIN laboratory_orders o
    ON p.patient_id = o.patient_id

LEFT JOIN hl7_messages m
    ON o.message_id = m.message_id

LEFT JOIN laboratory_results r
    ON o.order_id = r.order_id

GROUP BY p.patient_id
ORDER BY p.patient_id;
"""

# Query 2: Patient panel combination summary
sql_combination_query = """
WITH patient_panels AS (
    SELECT
        p.patient_id,
        COUNT(DISTINCT o.order_id) AS total_orders,
        COUNT(r.result_id) AS total_results,

        STRING_AGG(
            DISTINCT m.panel,
            ', ' ORDER BY m.panel
        ) AS panel_combination

    FROM patients p

    LEFT JOIN laboratory_orders o
        ON p.patient_id = o.patient_id

    LEFT JOIN hl7_messages m
        ON o.message_id = m.message_id

    LEFT JOIN laboratory_results r
        ON o.order_id = r.order_id

    GROUP BY p.patient_id
)

SELECT
    panel_combination,
    COUNT(*) AS total_patients,
    SUM(total_orders) AS total_orders,
    SUM(total_results) AS total_results,

    ROUND(
        AVG(total_results)::NUMERIC, 2
    ) AS avg_results_per_patient,

    ROUND(
        100.0 * COUNT(*)
        / NULLIF(SUM(COUNT(*)) OVER (), 0),
        2
    ) AS patient_percentage

FROM patient_panels

GROUP BY panel_combination
ORDER BY total_patients DESC, panel_combination;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        # Read both queries into pandas
        df_patient_utilization = pd.read_sql_query(
            sql_patient_query,
            connection
        )

        df_patient_combinations = pd.read_sql_query(
            sql_combination_query,
            connection
        )

    print("\nPatient-Level Laboratory Utilization:")
    display(df_patient_utilization.head(10))

    print("\nPatient Panel Combination Summary:")
    display(df_patient_combinations)

    # Verification checks
    total_patients = len(df_patient_utilization)

    total_orders = int(
        df_patient_utilization["total_orders"].sum()
    )

    total_results = int(
        df_patient_utilization["total_results"].sum()
    )

    patients_without_two_orders = int(
        df_patient_utilization["total_orders"]
        .ne(2).sum()
    )

    patients_without_results = int(
        df_patient_utilization["total_results"]
        .eq(0).sum()
    )

    combination_patients = int(
        df_patient_combinations["total_patients"].sum()
    )

    combination_results = int(
        df_patient_combinations["total_results"].sum()
    )

    utilization_passed = (
        total_patients == 1000
        and total_orders == 2000
        and total_results == 20612
        and patients_without_two_orders == 0
        and patients_without_results == 0
        and combination_patients == 1000
        and combination_results == 20612
    )

    print("\nPatient Utilization Verification")
    print("--------------------------------")
    print("Total Patients:", total_patients)
    print("Total Laboratory Orders:", total_orders)
    print("Total Laboratory Results:", total_results)
    print(
        "Patients Without Exactly Two Orders:",
        patients_without_two_orders
    )
    print(
        "Patients Without Laboratory Results:",
        patients_without_results
    )
    print(
        "Patients in Combination Summary:",
        combination_patients
    )

    print(
        "\nPatient Utilization Verification:",
        "PASSED" if utilization_passed
        else "REVIEW REQUIRED"
    )

except (psycopg.Error, KeyError, ValueError) as error:
    print("\nPatient Utilization Verification: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9E: Patient-Level Laboratory Utilization Analysis
-----------------------------------------------------


Enter PostgreSQL password:  ········



Patient-Level Laboratory Utilization:


C:\Users\angel\AppData\Local\Temp\ipykernel_7316\1161572298.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_patient_utilization = pd.read_sql_query(
C:\Users\angel\AppData\Local\Temp\ipykernel_7316\1161572298.py:105: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_patient_combinations = pd.read_sql_query(


,patient_id,total_orders,total_results,unique_panel_types,panel_types
0,P00001,2,23,2,"CBC, CMP"
1,P00002,2,17,2,"BMP, CBC"
2,P00003,2,23,2,"CBC, CMP"
3,P00004,2,22,2,"BMP, CMP"
4,P00005,2,22,2,"BMP, CMP"
5,P00006,2,23,2,"CBC, CMP"
6,P00007,2,22,2,"BMP, CMP"
7,P00008,2,17,2,"BMP, CBC"
8,P00009,2,23,2,"CBC, CMP"
9,P00010,2,23,2,"CBC, CMP"



Patient Panel Combination Summary:


,panel_combination,total_patients,total_orders,total_results,avg_results_per_patient,patient_percentage
0,"BMP, CBC",343,686.0,5831.0,17.0,34.3
1,"BMP, CMP",330,660.0,7260.0,22.0,33.0
2,"CBC, CMP",327,654.0,7521.0,23.0,32.7



Patient Utilization Verification
--------------------------------
Total Patients: 1000
Total Laboratory Orders: 2000
Total Laboratory Results: 20612
Patients Without Exactly Two Orders: 0
Patients Without Laboratory Results: 0
Patients in Combination Summary: 1000

Patient Utilization Verification: PASSED



#### Step 9E: Results and Interpretation

The patient-level utilization analysis confirmed that all 1,000 synthetic patients had exactly two laboratory orders, accounting for 2,000 orders and 20,612 individual laboratory results. No patients were missing laboratory results, and all patient records were included in the panel combination analysis.

The most common panel combination was BMP and CBC, representing 343 patients (34.3%). Each patient in this group had 17 laboratory results, producing 5,831 observations. The BMP and CMP combination accounted for 330 patients (33.0%), with 22 results per patient and 7,260 total observations. The remaining 327 patients (32.7%) received CBC and CMP panels, generating 23 results per patient and 7,521 observations.

Although the three combinations were distributed relatively evenly across patients, the total number of results differed because each panel contains a different number of tests. The CBC and CMP combination generated the greatest number of individual results despite having the fewest patients.

These findings demonstrate why patient counts and laboratory order counts do not always reflect the total volume of laboratory testing. Evaluating individual results alongside patient utilization provides a more complete picture of testing activity.

From a healthcare operations perspective, this type of analysis could support laboratory workload evaluation, test utilization monitoring, and resource planning. It also demonstrates how relational SQL queries can connect patient demographics, laboratory orders, and individual observations to produce meaningful operational information.

Because the dataset is synthetic and each patient was assigned two different laboratory panels, the findings illustrate the analytical workflow rather than actual patient testing patterns.



### Step 9F: PostgreSQL Analytical Views for Power BI

This phase creates reusable PostgreSQL analytical views based on the validated synthetic laboratory dataset.

The views organize laboratory testing volumes, turnaround times, test utilization, HL7 message processing, and patient-level activity into structured datasets that can be accessed directly by Power BI.

Rather than duplicating the original laboratory records, PostgreSQL views store SQL queries that retrieve information from the underlying relational tables. This approach supports consistent reporting, reduces repeated query development, and provides a foundation for interactive healthcare analytics dashboards.

Each view is created and verified individually to confirm that its results match the previously completed SQL analyses.

#### Step 9F-1: Laboratory Panel Volume View

The first analytical view summarizes laboratory orders and individual test results by panel.

The view calculates total orders, total observations, order percentages, result percentages, and average results per order for BMP, CBC, and CMP panels.

These measures will support Power BI visualizations comparing laboratory order volume with individual test volume.


In [53]:

# Step 9F-1: Laboratory Panel Volume View

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9F-1: Laboratory Panel Volume View")
print("--------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

create_view_sql = """
CREATE OR REPLACE VIEW vw_panel_volume AS

WITH panel_summary AS (
    SELECT
        o.panel_name,
        o.panel_loinc,
        COUNT(DISTINCT o.order_id) AS total_orders,
        COUNT(r.result_id) AS total_results

    FROM laboratory_orders o

    LEFT JOIN laboratory_results r
        ON o.order_id = r.order_id

    GROUP BY
        o.panel_name,
        o.panel_loinc
)

SELECT
    panel_name,
    panel_loinc,
    total_orders,
    total_results,

    ROUND(
        100.0 * total_orders
        / NULLIF(SUM(total_orders) OVER (), 0),
        2
    ) AS order_percentage,

    ROUND(
        100.0 * total_results
        / NULLIF(SUM(total_results) OVER (), 0),
        2
    ) AS result_percentage,

    ROUND(
        total_results::NUMERIC
        / NULLIF(total_orders, 0),
        2
    ) AS avg_results_per_order

FROM panel_summary;
"""

verify_view_sql = """
SELECT *
FROM vw_panel_volume
ORDER BY total_orders DESC;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Create analytical view
            cursor.execute(create_view_sql)

            # Query newly created view
            cursor.execute(verify_view_sql)

            records = cursor.fetchall()

            columns = [
                description.name
                for description in cursor.description
            ]

            df_view_panel_volume = pd.DataFrame(
                records,
                columns=columns
            )

            # Verify record counts
            total_orders = int(
                df_view_panel_volume["total_orders"].sum()
            )

            total_results = int(
                df_view_panel_volume["total_results"].sum()
            )

            view_passed = (
                len(df_view_panel_volume) == 3
                and total_orders == 2000
                and total_results == 20612
            )

            if not view_passed:
                raise RuntimeError(
                    "Panel volume view verification failed."
                )

        # Commit only after verification succeeds
        connection.commit()

    print("\nPostgreSQL View Created: vw_panel_volume")

    print("\nLaboratory Panel Volume View:")
    display(df_view_panel_volume)

    print("\nPanels:", len(df_view_panel_volume))
    print("Total Orders:", total_orders)
    print("Total Results:", total_results)

    print("\nView Verification Status: PASSED")

except (psycopg.Error, RuntimeError) as error:
    print("\nView Verification Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9F-1: Laboratory Panel Volume View
--------------------------------------


Enter PostgreSQL password:  ········



PostgreSQL View Created: vw_panel_volume

Laboratory Panel Volume View:


,panel_name,panel_loinc,total_orders,total_results,order_percentage,result_percentage,avg_results_per_order
0,Basic metabolic panel,24321-2,673,5384,33.65,26.12,8.00
1,CBC panel,58410-2,670,6030,33.50,29.25,9.00
2,Comprehensive metabolic panel,24323-8,657,9198,32.85,44.62,14.00



Panels: 3
Total Orders: 2000
Total Results: 20612

View Verification Status: PASSED



#### Step 9F-1: Results and Interpretation

The PostgreSQL analytical view `vw_panel_volume` was successfully created and verified against the earlier laboratory testing volume analysis. The view returned all three laboratory panels, accounting for 2,000 orders and 20,612 individual test results, with no differences in the expected record counts.

BMP represented the largest proportion of laboratory orders at 33.65%, followed by CBC at 33.50% and CMP at 32.85%. However, CMP contributed the greatest proportion of individual test results at 44.62%, compared with 29.25% for CBC and 26.12% for BMP. These differences reflect the number of tests included in each panel.

The successful verification confirms that the analytical view can reproduce the results of the original SQL query without creating a separate copy of the underlying laboratory records. This provides a consistent source for reporting and reduces the need to repeat complex SQL queries when developing dashboards.

For Power BI, the view will support comparisons between laboratory order volume and individual test volume, helping demonstrate how panel composition influences testing activity. Because the view retrieves information directly from PostgreSQL, it can also reflect changes to the underlying data when refreshed.

This establishes the first reusable reporting dataset for the laboratory interoperability dashboard.



#### Step 9F-2: Laboratory Turnaround Time View

This step creates a PostgreSQL analytical view to support laboratory turnaround time reporting in Power BI.

The view retrieves individual laboratory orders and calculates the elapsed time between specimen collection and the recorded laboratory result. Turnaround time is expressed in both minutes and hours, allowing the data to be used for different reporting and visualization requirements.

Unlike the earlier summarized turnaround time analysis, this view preserves individual order-level records. This allows Power BI to calculate averages, medians, distributions, and other performance measures while filtering by laboratory panel.

The view also retains patient and order identifiers, panel information, and collection and result timestamps to support future analysis.

After creation, SQL queries verify that all 2,000 laboratory orders are represented, that turnaround times are valid, and that the calculated averages match the earlier PostgreSQL analysis.

The measurements represent collection-to-result turnaround time within a synthetic laboratory environment rather than actual clinical laboratory processing performance.


In [54]:

# Step 9F-2: Laboratory Turnaround Time View

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9F-2: Laboratory Turnaround Time View")
print("----------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

create_view_sql = """
CREATE OR REPLACE VIEW vw_turnaround_time AS

SELECT
    order_id,
    patient_id,
    message_id,
    panel_name,
    panel_loinc,
    collection_time,
    result_time,

    ROUND(
        (
            EXTRACT(
                EPOCH FROM (result_time - collection_time)
            ) / 60.0
        )::NUMERIC,
        2
    ) AS turnaround_minutes,

    ROUND(
        (
            EXTRACT(
                EPOCH FROM (result_time - collection_time)
            ) / 3600.0
        )::NUMERIC,
        2
    ) AS turnaround_hours

FROM laboratory_orders;
"""

verify_summary_sql = """
SELECT
    panel_name,
    COUNT(*) AS total_orders,

    ROUND(
        AVG(turnaround_minutes), 2
    ) AS mean_tat_minutes,

    ROUND(
        PERCENTILE_CONT(0.5)
        WITHIN GROUP (
            ORDER BY turnaround_minutes
        )::NUMERIC,
        2
    ) AS median_tat_minutes,

    MIN(turnaround_minutes) AS min_tat_minutes,
    MAX(turnaround_minutes) AS max_tat_minutes

FROM vw_turnaround_time
GROUP BY panel_name
ORDER BY mean_tat_minutes DESC;
"""

verify_integrity_sql = """
SELECT
    COUNT(*) AS total_orders,

    COUNT(DISTINCT order_id) AS unique_orders,

    COUNT(*) FILTER (
        WHERE turnaround_minutes <= 0
           OR turnaround_minutes IS NULL
    ) AS invalid_turnaround_times,

    MIN(turnaround_minutes) AS minimum_tat,
    MAX(turnaround_minutes) AS maximum_tat

FROM vw_turnaround_time;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Create analytical view
            cursor.execute(create_view_sql)

            # Verify panel statistics
            cursor.execute(verify_summary_sql)

            summary_records = cursor.fetchall()

            summary_columns = [
                description.name
                for description in cursor.description
            ]

            df_view_tat_summary = pd.DataFrame(
                summary_records,
                columns=summary_columns
            )

            # Verify order-level integrity
            cursor.execute(verify_integrity_sql)

            integrity_record = cursor.fetchone()

            integrity_columns = [
                description.name
                for description in cursor.description
            ]

            df_view_tat_integrity = pd.DataFrame(
                [integrity_record],
                columns=integrity_columns
            )

            # Verify expected results
            total_orders = int(
                df_view_tat_integrity.loc[
                    0, "total_orders"
                ]
            )

            unique_orders = int(
                df_view_tat_integrity.loc[
                    0, "unique_orders"
                ]
            )

            invalid_tat = int(
                df_view_tat_integrity.loc[
                    0, "invalid_turnaround_times"
                ]
            )

            # Compare against previously calculated means
            expected_means = {
                "Basic metabolic panel": 106.90,
                "CBC panel": 106.21,
                "Comprehensive metabolic panel": 105.89
            }

            mean_matches = all(
                abs(
                    float(row["mean_tat_minutes"])
                    - expected_means[row["panel_name"]]
                ) <= 0.01
                for _, row in df_view_tat_summary.iterrows()
            )

            view_passed = (
                total_orders == 2000
                and unique_orders == 2000
                and invalid_tat == 0
                and len(df_view_tat_summary) == 3
                and mean_matches
            )

            if not view_passed:
                raise RuntimeError(
                    "Turnaround time view verification failed."
                )

        connection.commit()

    print("\nPostgreSQL View Created: vw_turnaround_time")

    print("\nTurnaround Time Summary:")
    display(df_view_tat_summary)

    print("\nOrder-Level Integrity Verification:")
    display(df_view_tat_integrity)

    print("\nTotal Orders:", total_orders)
    print("Unique Orders:", unique_orders)
    print("Invalid Turnaround Times:", invalid_tat)
    print("Mean TAT Matches Step 9B:", mean_matches)

    print("\nView Verification Status: PASSED")

except (psycopg.Error, RuntimeError, KeyError) as error:
    print("\nView Verification Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9F-2: Laboratory Turnaround Time View
----------------------------------------


Enter PostgreSQL password:  ········



PostgreSQL View Created: vw_turnaround_time

Turnaround Time Summary:


,panel_name,total_orders,mean_tat_minutes,median_tat_minutes,min_tat_minutes,max_tat_minutes
0,Basic metabolic panel,673,106.90,107.00,30.00,180.00
1,CBC panel,670,106.21,107.00,30.00,180.00
2,Comprehensive metabolic panel,657,105.89,107.00,30.00,180.00



Order-Level Integrity Verification:


,total_orders,unique_orders,invalid_turnaround_times,minimum_tat,maximum_tat
0,2000,2000,0,30.00,180.00



Total Orders: 2000
Unique Orders: 2000
Invalid Turnaround Times: 0
Mean TAT Matches Step 9B: True

View Verification Status: PASSED



#### Step 9F-2: Results and Interpretation

The PostgreSQL analytical view `vw_turnaround_time` was successfully created and verified, returning 2,000 unique laboratory orders with no duplicate records or invalid turnaround times. The calculated mean turnaround times matched the earlier SQL analysis in Step 9B, confirming that the view preserved the original collection-to-result measurements.

The mean turnaround time was 106.90 minutes for BMP, 106.21 minutes for CBC, and 105.89 minutes for CMP. All three panels had a median turnaround time of 107 minutes, with recorded values ranging from 30 to 180 minutes. These findings indicate that the simulated turnaround times were similar across the different laboratory panels.

Unlike the earlier summary analysis, this view retains individual order identifiers, patient identifiers, laboratory panel information, and collection and result timestamps. Preserving these details allows turnaround time to be evaluated at the order level rather than limiting the analysis to previously calculated averages.

For Power BI, the view will support interactive comparisons of turnaround time across laboratory panels, distributions of individual order times, and the identification of orders exceeding simulated performance thresholds. These measures could help demonstrate how laboratory data can be used to monitor workflow efficiency and identify potential delays.

The successful verification establishes a reusable reporting dataset while maintaining the integrity of the original PostgreSQL records. Because the data is synthetic, the findings illustrate laboratory performance analysis rather than actual clinical laboratory turnaround times.



#### Step 9F-3: Laboratory Test Utilization View

This step creates a PostgreSQL analytical view to summarize individual laboratory test utilization across the synthetic dataset.

The view combines standardized LOINC test definitions with laboratory observations to calculate the number of results associated with each test, the number of distinct laboratory orders containing that test, and its percentage of overall testing volume.

Unlike panel-level utilization, this analysis focuses on individual analytes, including tests that appear in more than one laboratory panel. This distinction is important because shared tests may have higher utilization even when panel order volumes are relatively similar.

The view will support Power BI visualizations of frequently performed tests, utilization patterns, and the contribution of individual analytes to total laboratory testing activity.

After creation, the view is verified against the earlier Step 9C analysis to confirm that all 23 test definitions and 20,612 laboratory observations are represented.


In [55]:

# Step 9F-3: Laboratory Test Utilization View

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9F-3: Laboratory Test Utilization View")
print("-----------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

create_view_sql = """
CREATE OR REPLACE VIEW vw_test_utilization AS

SELECT
    t.test_name,
    t.loinc_code,
    t.unit,

    COUNT(r.result_id) AS total_observations,

    COUNT(DISTINCT r.order_id) AS total_orders,

    ROUND(
        100.0 * COUNT(r.result_id)
        / NULLIF(
            SUM(COUNT(r.result_id)) OVER (), 0
        ),
        2
    ) AS result_percentage

FROM laboratory_tests t

LEFT JOIN laboratory_results r
    ON t.loinc_code = r.loinc_code

GROUP BY
    t.test_name,
    t.loinc_code,
    t.unit;
"""

verify_view_sql = """
SELECT *
FROM vw_test_utilization
ORDER BY total_observations DESC, test_name;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Create analytical view
            cursor.execute(create_view_sql)

            # Retrieve view results
            cursor.execute(verify_view_sql)

            records = cursor.fetchall()

            columns = [
                description.name
                for description in cursor.description
            ]

            df_view_test_utilization = pd.DataFrame(
                records,
                columns=columns
            )

            # Verify test definitions and observations
            total_tests = len(df_view_test_utilization)

            total_observations = int(
                df_view_test_utilization[
                    "total_observations"
                ].sum()
            )

            unused_tests = int(
                df_view_test_utilization[
                    "total_observations"
                ].eq(0).sum()
            )

            # Compare with Step 9C
            previous_results = (
                df_test_utilization[
                    ["loinc_code", "total_observations",
                     "total_orders"]
                ]
                .sort_values("loinc_code")
                .reset_index(drop=True)
            )

            current_results = (
                df_view_test_utilization[
                    ["loinc_code", "total_observations",
                     "total_orders"]
                ]
                .sort_values("loinc_code")
                .reset_index(drop=True)
            )

            matches_step_9c = (
                previous_results.equals(current_results)
            )

            view_passed = (
                total_tests == 23
                and total_observations == 20612
                and unused_tests == 0
                and matches_step_9c
            )

            if not view_passed:
                raise RuntimeError(
                    "Test utilization view verification failed."
                )

        # Commit only after successful verification
        connection.commit()

    print("\nPostgreSQL View Created: vw_test_utilization")

    print("\nLaboratory Test Utilization View:")
    display(df_view_test_utilization)

    print("\nTotal Test Definitions:", total_tests)
    print("Total Laboratory Observations:", total_observations)
    print("Unused Test Definitions:", unused_tests)
    print("Matches Step 9C:", matches_step_9c)

    print("\nView Verification Status: PASSED")

except (psycopg.Error, RuntimeError, KeyError) as error:
    print("\nView Verification Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9F-3: Laboratory Test Utilization View
-----------------------------------------


Enter PostgreSQL password:  ········



PostgreSQL View Created: vw_test_utilization

Laboratory Test Utilization View:


,test_name,loinc_code,unit,total_observations,total_orders,result_percentage
0,BUN,3094-0,mg/dL,1330,1330,6.45
1,Calcium,17861-6,mg/dL,1330,1330,6.45
2,Chloride,2075-0,mmol/L,1330,1330,6.45
3,CO2,2028-9,mmol/L,1330,1330,6.45
4,Creatinine,2160-0,mg/dL,1330,1330,6.45
5,Glucose,2345-7,mg/dL,1330,1330,6.45
6,Potassium,2823-3,mmol/L,1330,1330,6.45
7,Sodium,2951-2,mmol/L,1330,1330,6.45
8,Hematocrit,4544-3,%,670,670,3.25
9,Hemoglobin,718-7,g/dL,670,670,3.25



Total Test Definitions: 23
Total Laboratory Observations: 20612
Unused Test Definitions: 0
Matches Step 9C: True

View Verification Status: PASSED



#### Step 9F-3: Results and Interpretation

The PostgreSQL analytical view `vw_test_utilization` was successfully created and verified against the earlier Step 9C laboratory test utilization analysis. All 23 LOINC-defined laboratory tests were represented, accounting for 20,612 individual observations. No unused test definitions were identified, and the utilization counts matched the original SQL results.

The eight tests shared by BMP and CMP, including sodium, potassium, chloride, CO2, glucose, BUN, creatinine, and calcium, each appeared in 1,330 laboratory orders and represented 6.45% of all observations individually. The nine CBC tests each appeared 670 times, accounting for 3.25% of results per test. The six CMP-specific tests each appeared 657 times, representing 3.19% of observations per test.

These differences reflect how laboratory panel composition influences individual test utilization. Tests included in multiple panels appear more frequently, even when the overall number of orders for each panel is relatively similar.

The successful verification confirms that the PostgreSQL view reproduces the earlier utilization analysis while providing a reusable dataset for Power BI. The view retains standardized LOINC codes, test names, measurement units, observation counts, and utilization percentages, allowing individual analytes to be compared consistently.

From a laboratory operations perspective, this information could support reagent utilization monitoring, inventory planning, and workload evaluation when combined with actual testing costs and resource consumption. In Power BI, the view will support ranked test utilization charts and comparisons of individual analytes across the synthetic laboratory dataset.



#### Step 9F-4: HL7 Message Performance View

This step creates a PostgreSQL analytical view to support HL7 message processing and validation reporting in Power BI.

The view retrieves individual HL7 message records and evaluates their validation status, parsing status, and observation count consistency. Each message is associated with a laboratory order, patient identifier, and panel type, allowing message processing performance to be examined across the simulated LIS-to-EHR workflow.

Additional fields identify whether each message passed validation, was successfully parsed, and contained the expected number of OBX observation segments. These measures provide a foundation for monitoring message processing success rates and identifying potential interoperability data quality issues.

Unlike the earlier panel-level summary, this view preserves individual message records so that Power BI can calculate performance measures dynamically and filter results by laboratory panel.

After creation, the view is verified against the Step 9D analysis to confirm that all 2,000 HL7 messages are represented and that the expected 20,612 laboratory observations match the recorded OBX segment counts.

The analysis reflects a controlled synthetic HL7 workflow rather than actual hospital interface performance.


In [56]:

# Step 9F-4: HL7 Message Performance View

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9F-4: HL7 Message Performance View")
print("--------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

create_view_sql = """
CREATE OR REPLACE VIEW vw_hl7_performance AS

SELECT
    message_id,
    order_id,
    patient_id,
    panel,
    result_count,
    obx_count,
    validation_status,
    parsing_status,
    validation_error,

    (validation_status = 'PASSED')
        AS validation_pass,

    (parsing_status = 'PASSED')
        AS parsing_pass,

    (result_count = obx_count)
        AS obx_count_match

FROM hl7_messages;
"""

verify_summary_sql = """
SELECT
    panel,
    COUNT(*) AS total_messages,

    COUNT(*) FILTER (
        WHERE validation_pass IS TRUE
    ) AS validated_messages,

    COUNT(*) FILTER (
        WHERE parsing_pass IS TRUE
    ) AS parsed_messages,

    COUNT(*) FILTER (
        WHERE obx_count_match IS FALSE
           OR obx_count_match IS NULL
    ) AS obx_mismatches,

    SUM(result_count) AS expected_results,
    SUM(obx_count) AS total_obx_segments,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE validation_pass IS TRUE
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS validation_success_pct,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE parsing_pass IS TRUE
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS parsing_success_pct

FROM vw_hl7_performance
GROUP BY panel
ORDER BY total_messages DESC;
"""

verify_integrity_sql = """
SELECT
    COUNT(*) AS total_messages,

    COUNT(DISTINCT message_id)
        AS unique_messages,

    COUNT(*) FILTER (
        WHERE validation_pass IS NOT TRUE
    ) AS failed_validation,

    COUNT(*) FILTER (
        WHERE parsing_pass IS NOT TRUE
    ) AS failed_parsing,

    COUNT(*) FILTER (
        WHERE obx_count_match IS NOT TRUE
    ) AS mismatched_obx_counts,

    COUNT(*) FILTER (
        WHERE validation_error IS NOT NULL
          AND TRIM(validation_error) <> ''
    ) AS recorded_validation_errors

FROM vw_hl7_performance;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Create analytical view
            cursor.execute(create_view_sql)

            # Verify panel-level processing
            cursor.execute(verify_summary_sql)

            summary_records = cursor.fetchall()

            summary_columns = [
                description.name
                for description in cursor.description
            ]

            df_view_hl7_summary = pd.DataFrame(
                summary_records,
                columns=summary_columns
            )

            # Verify message-level integrity
            cursor.execute(verify_integrity_sql)

            integrity_record = cursor.fetchone()

            integrity_columns = [
                description.name
                for description in cursor.description
            ]

            df_view_hl7_integrity = pd.DataFrame(
                [integrity_record],
                columns=integrity_columns
            )

            total_messages = int(
                df_view_hl7_integrity.loc[
                    0, "total_messages"
                ]
            )

            unique_messages = int(
                df_view_hl7_integrity.loc[
                    0, "unique_messages"
                ]
            )

            failed_validation = int(
                df_view_hl7_integrity.loc[
                    0, "failed_validation"
                ]
            )

            failed_parsing = int(
                df_view_hl7_integrity.loc[
                    0, "failed_parsing"
                ]
            )

            obx_mismatches = int(
                df_view_hl7_integrity.loc[
                    0, "mismatched_obx_counts"
                ]
            )

            validation_errors = int(
                df_view_hl7_integrity.loc[
                    0, "recorded_validation_errors"
                ]
            )

            expected_results = int(
                df_view_hl7_summary[
                    "expected_results"
                ].sum()
            )

            total_obx = int(
                df_view_hl7_summary[
                    "total_obx_segments"
                ].sum()
            )

            view_passed = (
                total_messages == 2000
                and unique_messages == 2000
                and failed_validation == 0
                and failed_parsing == 0
                and obx_mismatches == 0
                and validation_errors == 0
                and expected_results == 20612
                and total_obx == 20612
                and len(df_view_hl7_summary) == 3
            )

            if not view_passed:
                raise RuntimeError(
                    "HL7 performance view verification failed."
                )

        connection.commit()

    print("\nPostgreSQL View Created: vw_hl7_performance")

    print("\nHL7 Message Performance Summary:")
    display(df_view_hl7_summary)

    print("\nMessage-Level Integrity Verification:")
    display(df_view_hl7_integrity)

    print("\nTotal Messages:", total_messages)
    print("Unique Messages:", unique_messages)
    print("Failed Validation:", failed_validation)
    print("Failed Parsing:", failed_parsing)
    print("OBX Count Mismatches:", obx_mismatches)
    print("Recorded Validation Errors:", validation_errors)
    print("Expected Laboratory Results:", expected_results)
    print("Total OBX Segments:", total_obx)

    print("\nView Verification Status: PASSED")

except (psycopg.Error, RuntimeError, KeyError) as error:
    print("\nView Verification Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9F-4: HL7 Message Performance View
--------------------------------------


Enter PostgreSQL password:  ········



PostgreSQL View Created: vw_hl7_performance

HL7 Message Performance Summary:


,panel,total_messages,validated_messages,parsed_messages,obx_mismatches,expected_results,total_obx_segments,validation_success_pct,parsing_success_pct
0,BMP,673,673,673,0,5384,5384,100.00,100.00
1,CBC,670,670,670,0,6030,6030,100.00,100.00
2,CMP,657,657,657,0,9198,9198,100.00,100.00



Message-Level Integrity Verification:


,total_messages,unique_messages,failed_validation,failed_parsing,mismatched_obx_counts,recorded_validation_errors
0,2000,2000,0,0,0,0



Total Messages: 2000
Unique Messages: 2000
Failed Validation: 0
Failed Parsing: 0
OBX Count Mismatches: 0
Recorded Validation Errors: 0
Expected Laboratory Results: 20612
Total OBX Segments: 20612

View Verification Status: PASSED



#### Step 9F-4: Results and Interpretation

The PostgreSQL analytical view `vw_hl7_performance` was successfully created and verified against the earlier Step 9D HL7 message processing analysis. All 2,000 HL7 messages were represented with unique message identifiers, and both validation and parsing achieved a 100% success rate. No failed messages, observation count mismatches, or recorded validation errors were identified.

BMP accounted for 673 messages containing 5,384 OBX segments, while CBC accounted for 670 messages containing 6,030 segments. CMP generated 657 messages containing 9,198 segments. Although message volumes were relatively similar across the three panels, CMP contained the greatest number of individual observations because each panel included 14 laboratory tests.

The analysis also confirmed that the 20,612 expected laboratory results matched the total number of OBX segments recorded in PostgreSQL. This consistency supports the earlier message generation, parsing, and data reconciliation findings.

Unlike the earlier summarized analysis, the new PostgreSQL view preserves individual message records and includes Boolean fields identifying successful validation, parsing, and observation count matching. These fields will allow Power BI to calculate processing success rates and identify messages requiring further investigation.

From a healthcare interoperability perspective, monitoring message processing and observation completeness can help identify potential problems with laboratory data exchange. In an actual hospital environment, incomplete or incorrectly processed HL7 messages could affect the availability of laboratory information within the electronic health record.

The results demonstrate successful processing within the synthetic interoperability workflow. They do not establish successful delivery to a live EHR or confirm clinical interpretation of the transmitted results.



#### Step 9F-5: Patient-Level Laboratory Utilization View

This step creates a PostgreSQL analytical view to support patient-level laboratory utilization reporting in Power BI.

The view combines patient records, laboratory orders, HL7 message information, and individual laboratory results to summarize testing activity for each synthetic patient. It calculates the total number of orders, individual test results, and unique panel types associated with each patient.

Laboratory panel combinations are also identified to examine how different testing patterns influence the number of observations generated. This provides a more detailed understanding of testing activity than patient or order counts alone.

Unlike the earlier panel-level analyses, this view preserves one record per patient, allowing Power BI to evaluate laboratory utilization across the synthetic patient population.

After creation, the view is verified against the Step 9E analysis to confirm that all 1,000 patients are represented, each patient has exactly two laboratory orders, and all 20,612 laboratory observations are accounted for.

The analysis uses synthetic patient and laboratory data to demonstrate relational database reporting rather than actual clinical testing patterns.


In [59]:

# Step 9F-5: Patient-Level Laboratory Utilization View

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9F-5: Patient-Level Laboratory Utilization View")
print("---------------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

# Create patient-level analytical view
create_view_sql = """
CREATE OR REPLACE VIEW vw_patient_utilization AS

SELECT
    p.patient_id,
    p.last_name,
    p.first_name,
    p.date_of_birth,
    p.sex,

    COUNT(DISTINCT o.order_id) AS total_orders,

    COUNT(r.result_id) AS total_results,

    COUNT(DISTINCT o.panel_loinc)
        AS unique_panel_types,

    STRING_AGG(
        DISTINCT m.panel,
        ', ' ORDER BY m.panel
    ) AS panel_combination

FROM patients p

LEFT JOIN laboratory_orders o
    ON p.patient_id = o.patient_id

LEFT JOIN hl7_messages m
    ON o.message_id = m.message_id

LEFT JOIN laboratory_results r
    ON o.order_id = r.order_id

GROUP BY
    p.patient_id,
    p.last_name,
    p.first_name,
    p.date_of_birth,
    p.sex;
"""

# Verify patient panel combinations
verify_summary_sql = """
SELECT
    panel_combination,
    COUNT(*) AS total_patients,
    SUM(total_orders) AS total_orders,
    SUM(total_results) AS total_results,

    ROUND(
        AVG(total_results)::NUMERIC, 2
    ) AS avg_results_per_patient,

    ROUND(
        100.0 * COUNT(*)
        / NULLIF(SUM(COUNT(*)) OVER (), 0),
        2
    ) AS patient_percentage

FROM vw_patient_utilization
GROUP BY panel_combination
ORDER BY total_patients DESC, panel_combination;
"""

# Verify patient-level integrity
verify_integrity_sql = """
SELECT
    COUNT(*) AS total_patients,

    COUNT(DISTINCT patient_id)
        AS unique_patients,

    SUM(total_orders) AS total_orders,

    SUM(total_results) AS total_results,

    COUNT(*) FILTER (
        WHERE total_orders <> 2
    ) AS patients_without_two_orders,

    COUNT(*) FILTER (
        WHERE total_results = 0
    ) AS patients_without_results,

    COUNT(*) FILTER (
        WHERE unique_panel_types <> 2
    ) AS patients_without_two_panel_types

FROM vw_patient_utilization;
"""

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Create analytical view
            cursor.execute(create_view_sql)

            # Retrieve panel combination summary
            cursor.execute(verify_summary_sql)

            summary_records = cursor.fetchall()

            summary_columns = [
                description.name
                for description in cursor.description
            ]

            df_view_patient_summary = pd.DataFrame(
                summary_records,
                columns=summary_columns
            )

            # Retrieve patient-level integrity results
            cursor.execute(verify_integrity_sql)

            integrity_record = cursor.fetchone()

            integrity_columns = [
                description.name
                for description in cursor.description
            ]

            df_view_patient_integrity = pd.DataFrame(
                [integrity_record],
                columns=integrity_columns
            )

            # Extract verification values
            total_patients = int(
                df_view_patient_integrity.loc[
                    0, "total_patients"
                ]
            )

            unique_patients = int(
                df_view_patient_integrity.loc[
                    0, "unique_patients"
                ]
            )

            total_orders = int(
                df_view_patient_integrity.loc[
                    0, "total_orders"
                ]
            )

            total_results = int(
                df_view_patient_integrity.loc[
                    0, "total_results"
                ]
            )

            patients_without_two_orders = int(
                df_view_patient_integrity.loc[
                    0, "patients_without_two_orders"
                ]
            )

            patients_without_results = int(
                df_view_patient_integrity.loc[
                    0, "patients_without_results"
                ]
            )

            patients_without_two_panel_types = int(
                df_view_patient_integrity.loc[
                    0, "patients_without_two_panel_types"
                ]
            )

            # Compare results with Step 9E
            previous_combinations = (
                df_patient_combinations[
                    [
                        "panel_combination",
                        "total_patients",
                        "total_orders",
                        "total_results"
                    ]
                ]
                .sort_values("panel_combination")
                .reset_index(drop=True)
                .copy()
            )

            current_combinations = (
                df_view_patient_summary[
                    [
                        "panel_combination",
                        "total_patients",
                        "total_orders",
                        "total_results"
                    ]
                ]
                .sort_values("panel_combination")
                .reset_index(drop=True)
                .copy()
            )

            # Normalize numeric types before comparison
            numeric_columns = [
                "total_patients",
                "total_orders",
                "total_results"
            ]

            previous_combinations[numeric_columns] = (
                previous_combinations[numeric_columns]
                .astype("int64")
            )

            current_combinations[numeric_columns] = (
                current_combinations[numeric_columns]
                .astype("int64")
            )

            matches_step_9e = (
                previous_combinations.equals(
                    current_combinations
                )
            )

            # Final verification
            view_passed = (
                total_patients == 1000
                and unique_patients == 1000
                and total_orders == 2000
                and total_results == 20612
                and patients_without_two_orders == 0
                and patients_without_results == 0
                and patients_without_two_panel_types == 0
                and matches_step_9e
                and len(df_view_patient_summary) == 3
            )

            if not view_passed:
                raise RuntimeError(
                    "Patient utilization view verification failed."
                )

        # Commit only after successful verification
        connection.commit()

    print("\nPostgreSQL View Created: vw_patient_utilization")

    print("\nPatient Panel Combination Summary:")
    display(df_view_patient_summary)

    print("\nPatient-Level Integrity Verification:")
    display(df_view_patient_integrity)

    print("\nTotal Patients:", total_patients)
    print("Unique Patients:", unique_patients)
    print("Total Laboratory Orders:", total_orders)
    print("Total Laboratory Results:", total_results)

    print(
        "Patients Without Exactly Two Orders:",
        patients_without_two_orders
    )

    print(
        "Patients Without Laboratory Results:",
        patients_without_results
    )

    print(
        "Patients Without Two Panel Types:",
        patients_without_two_panel_types
    )

    print("Matches Step 9E:", matches_step_9e)

    print("\nView Verification Status: PASSED")

except (
    psycopg.Error,
    RuntimeError,
    KeyError,
    ValueError,
    TypeError
) as error:

    print("\nView Verification Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9F-5: Patient-Level Laboratory Utilization View
---------------------------------------------------


Enter PostgreSQL password:  ········



PostgreSQL View Created: vw_patient_utilization

Patient Panel Combination Summary:


,panel_combination,total_patients,total_orders,total_results,avg_results_per_patient,patient_percentage
0,"BMP, CBC",343,686,5831,17.00,34.30
1,"BMP, CMP",330,660,7260,22.00,33.00
2,"CBC, CMP",327,654,7521,23.00,32.70



Patient-Level Integrity Verification:


,total_patients,unique_patients,total_orders,total_results,patients_without_two_orders,patients_without_results,patients_without_two_panel_types
0,1000,1000,2000,20612,0,0,0



Total Patients: 1000
Unique Patients: 1000
Total Laboratory Orders: 2000
Total Laboratory Results: 20612
Patients Without Exactly Two Orders: 0
Patients Without Laboratory Results: 0
Patients Without Two Panel Types: 0
Matches Step 9E: True

View Verification Status: PASSED



#### Step 9F-5: Results and Interpretation

The PostgreSQL analytical view `vw_patient_utilization` was successfully created and verified against the earlier Step 9E patient-level laboratory utilization analysis. All 1,000 synthetic patients were represented with unique identifiers, accounting for 2,000 laboratory orders and 20,612 individual test results. No patients were missing laboratory results, and every patient had exactly two orders representing two different panel types.

The most common laboratory panel combination was BMP and CBC, representing 343 patients (34.3%) and generating 5,831 individual results. BMP and CMP accounted for 330 patients (33.0%) and 7,260 results, while CBC and CMP represented 327 patients (32.7%) and 7,521 results.

Although the three panel combinations were distributed relatively evenly across patients, the number of individual laboratory results differed because each panel contains a different number of tests. Patients receiving BMP and CBC generated 17 results each, compared with 22 for BMP and CMP and 23 for CBC and CMP.

The view successfully reproduced the earlier SQL findings while preserving one record per patient. This structure will allow Power BI to calculate patient-level utilization measures without duplicating patient counts when analyzing individual laboratory observations.

From a healthcare operations perspective, patient-level utilization analysis can help distinguish between the number of patients receiving laboratory services, the number of orders placed, and the total testing workload. When combined with additional clinical and operational data, these measures could support laboratory resource planning, utilization monitoring, and cost analysis.

Because the dataset is synthetic and each patient was assigned two different laboratory panels, these findings demonstrate relational SQL reporting rather than actual patient testing patterns.



#### Step 9F-6: Final PostgreSQL Analytical View Verification

This step performs a consolidated verification of the five PostgreSQL analytical views created to support laboratory operations and HL7 interoperability reporting.

The verification confirms that each view exists, can be queried successfully, and contains the expected number of records. Additional checks compare laboratory orders, test results, patient records, and HL7 message counts against the underlying relational tables.

This final review helps confirm that the analytical views preserve the consistency of the synthetic laboratory dataset and are ready to serve as reporting sources for Power BI.

The verification provides a documented checkpoint between PostgreSQL data preparation and dashboard development.


In [60]:

# Step 9F-6: Final PostgreSQL Analytical View Verification

import psycopg
import pandas as pd
from getpass import getpass

print("Step 9F-6: Final PostgreSQL Analytical View Verification")
print("-------------------------------------------------------")

db_password = getpass("Enter PostgreSQL password: ")

expected_views = {
    "vw_panel_volume": 3,
    "vw_turnaround_time": 2000,
    "vw_test_utilization": 23,
    "vw_hl7_performance": 2000,
    "vw_patient_utilization": 1000
}

try:
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="hl7_healthcare_interoperability",
        user="postgres",
        password=db_password,
        connect_timeout=10
    ) as connection:

        with connection.cursor() as cursor:

            # Verify analytical view existence
            cursor.execute("""
                SELECT table_name
                FROM information_schema.views
                WHERE table_schema = 'public';
            """)

            existing_views = {
                row[0] for row in cursor.fetchall()
            }

            # Verify individual view record counts
            view_results = []

            for view_name, expected_count in expected_views.items():

                view_exists = view_name in existing_views

                if view_exists:
                    cursor.execute(
                        f"SELECT COUNT(*) FROM public.{view_name};"
                    )
                    actual_count = cursor.fetchone()[0]
                else:
                    actual_count = None

                status = (
                    "PASSED"
                    if actual_count == expected_count
                    else "FAILED"
                )

                view_results.append({
                    "View": view_name,
                    "Expected_Records": expected_count,
                    "Actual_Records": actual_count,
                    "Status": status
                })

            df_final_views = pd.DataFrame(view_results)

            # Verify underlying relational table totals
            cursor.execute("""
                SELECT
                    (SELECT COUNT(*) FROM patients)
                        AS patients,

                    (SELECT COUNT(*) FROM laboratory_tests)
                        AS laboratory_tests,

                    (SELECT COUNT(*) FROM hl7_messages)
                        AS hl7_messages,

                    (SELECT COUNT(*) FROM laboratory_orders)
                        AS laboratory_orders,

                    (SELECT COUNT(*) FROM laboratory_results)
                        AS laboratory_results;
            """)

            table_counts = cursor.fetchone()

            table_columns = [
                description.name
                for description in cursor.description
            ]

            df_final_tables = pd.DataFrame(
                [table_counts],
                columns=table_columns
            )

            # Cross-check analytical view totals
            cursor.execute("""
                SELECT
                    (SELECT SUM(total_orders)
                     FROM vw_panel_volume)
                        AS panel_orders,

                    (SELECT SUM(total_results)
                     FROM vw_panel_volume)
                        AS panel_results,

                    (SELECT COUNT(*)
                     FROM vw_turnaround_time)
                        AS turnaround_orders,

                    (SELECT SUM(total_observations)
                     FROM vw_test_utilization)
                        AS utilization_results,

                    (SELECT COUNT(*)
                     FROM vw_hl7_performance)
                        AS hl7_messages,

                    (SELECT SUM(obx_count)
                     FROM vw_hl7_performance)
                        AS hl7_observations,

                    (SELECT SUM(total_orders)
                     FROM vw_patient_utilization)
                        AS patient_orders,

                    (SELECT SUM(total_results)
                     FROM vw_patient_utilization)
                        AS patient_results;
            """)

            crosscheck_record = cursor.fetchone()

            crosscheck_columns = [
                description.name
                for description in cursor.description
            ]

            df_final_crosscheck = pd.DataFrame(
                [crosscheck_record],
                columns=crosscheck_columns
            )

            # Expected relational table counts
            expected_tables = {
                "patients": 1000,
                "laboratory_tests": 23,
                "hl7_messages": 2000,
                "laboratory_orders": 2000,
                "laboratory_results": 20612
            }

            # Expected analytical totals
            expected_crosschecks = {
                "panel_orders": 2000,
                "panel_results": 20612,
                "turnaround_orders": 2000,
                "utilization_results": 20612,
                "hl7_messages": 2000,
                "hl7_observations": 20612,
                "patient_orders": 2000,
                "patient_results": 20612
            }

            # Verify all analytical views
            views_passed = (
                df_final_views["Status"].eq("PASSED").all()
            )

            # Verify relational table counts
            tables_passed = all(
                int(df_final_tables.loc[0, column]) == expected
                for column, expected in expected_tables.items()
            )

            # Verify analytical cross-check totals
            crosschecks_passed = all(
                int(df_final_crosscheck.loc[0, column]) == expected
                for column, expected in expected_crosschecks.items()
            )

            overall_passed = (
                views_passed
                and tables_passed
                and crosschecks_passed
            )

    print("\nPostgreSQL Analytical View Verification:")
    display(df_final_views)

    print("\nUnderlying Relational Table Counts:")
    display(df_final_tables)

    print("\nAnalytical View Cross-Check Totals:")
    display(df_final_crosscheck)

    print("\nFinal Verification Summary:")
    print("Analytical Views Found:", len(expected_views))
    print("Analytical View Counts Match:", views_passed)
    print("Relational Table Counts Match:", tables_passed)
    print("Analytical Totals Match:", crosschecks_passed)

    print(
        "\nOverall Verification Status:",
        "PASSED" if overall_passed else "REVIEW REQUIRED"
    )

except (psycopg.Error, ValueError, TypeError, KeyError) as error:
    print("\nOverall Verification Status: REVIEW REQUIRED")
    print("Error Type:", type(error).__name__)
    print("Error Details:", str(error).splitlines()[0])

finally:
    del db_password


Step 9F-6: Final PostgreSQL Analytical View Verification
-------------------------------------------------------


Enter PostgreSQL password:  ········



PostgreSQL Analytical View Verification:


,View,Expected_Records,Actual_Records,Status
0,vw_panel_volume,3,3,PASSED
1,vw_turnaround_time,2000,2000,PASSED
2,vw_test_utilization,23,23,PASSED
3,vw_hl7_performance,2000,2000,PASSED
4,vw_patient_utilization,1000,1000,PASSED



Underlying Relational Table Counts:


,patients,laboratory_tests,hl7_messages,laboratory_orders,laboratory_results
0,1000,23,2000,2000,20612



Analytical View Cross-Check Totals:


,panel_orders,panel_results,turnaround_orders,utilization_results,hl7_messages,hl7_observations,patient_orders,patient_results
0,2000,20612,2000,20612,2000,20612,2000,20612



Final Verification Summary:
Analytical Views Found: 5
Analytical View Counts Match: True
Relational Table Counts Match: True
Analytical Totals Match: True

Overall Verification Status: PASSED



#### Step 9F-6: Results and Interpretation

The final PostgreSQL verification confirmed that all five analytical views were successfully created and returned the expected number of records. The views included three panel-level utilization records, 2,000 turnaround time records, 23 individual laboratory test definitions, 2,000 HL7 message records, and 1,000 patient-level utilization records. All five views passed their record count verification checks.

The underlying relational database also contained the expected 25,635 records distributed across five tables: 1,000 patients, 23 laboratory test definitions, 2,000 HL7 messages, 2,000 laboratory orders, and 20,612 individual laboratory results.

Additional cross-checks confirmed that the analytical views consistently represented 2,000 laboratory orders and 20,612 laboratory observations. The HL7 message view also accounted for all 2,000 messages and 20,612 OBX segments, confirming consistency between message processing information and laboratory testing activity.

No discrepancies were identified during the final verification. These results support the earlier data preparation, HL7 parsing, reconciliation, and PostgreSQL integrity checks completed throughout the project.

The five analytical views provide reusable reporting datasets for laboratory panel volume, turnaround time, individual test utilization, HL7 message processing, and patient-level utilization. By organizing these measures within PostgreSQL, the project establishes a consistent foundation for developing Power BI dashboards without requiring separate CSV exports for each analysis.

This verification completes the PostgreSQL analytical reporting preparation. The next phase will connect Power BI to the database and develop interactive visualizations demonstrating laboratory operations and HL7 interoperability performance using synthetic healthcare data.
